<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NBA_Prop_Hit_Sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NBA 3:1 props.csv to NBA 3:1 props.csv


In [ ]:
import pandas as pd

# Get uploaded filename dynamically
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Raw Columns:")
print(df.columns.tolist())

df.head()

Raw Columns:
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,NaN,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW
1,NaN,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,BET\n-110,-1102,52.4%,100%,100%,100%,100%,100%,100%,100%,100%,100%
2,NaN,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,BET\n+105,1052,48.8%,100%,100%,100%,100%,100%,100%,100%,100%,100%
3,NaN,Jarrett Allen,CLE at BKN,Rebounds,o0.5,BET\n-145,-1452,59.2%,100%,100%,100%,100%,100%,100%,100%,98.3%,96.6%
4,NaN,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,BET\n-129,-1304,56.3%,100%,90%,97.7%,100%,100%,20%,30%,15.5%,13.3%


In [ ]:
import numpy as np
import re

# Promote first row to header
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)

df.columns = df.columns.astype(str).str.strip()

# Drop empty columns
df = df.loc[:, df.columns.notna()]
df = df.dropna(axis=1, how='all')

# Extract American odds from "BET\n-169"
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'([+-]\d+)', str(x))
    return float(match.group(1)) if match else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Normalize percentage columns
pct_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H",
            "DVP L5","DVP L10","DVP 2025","DVP HM/AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("Cleaned Columns:")
print(df.columns.tolist())

df.head()

In [ ]:
 import numpy as np
import pandas as pd

# ---------- Helpers ----------
def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

def american_to_decimal(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return 1.0 + (100.0/(-o))
    return 1.0 + (o/100.0)

def kelly_fraction(p, dec_odds):
    # For decimal odds d: b = d - 1
    if pd.isna(p) or pd.isna(dec_odds):
        return np.nan
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0:
        return np.nan
    f = (b*p - q) / b
    return max(0.0, f)

def safe_mean(row, cols):
    vals = [row[c] for c in cols if c in row.index and pd.notna(row[c])]
    return np.mean(vals) if len(vals) else np.nan

# ---------- 1) Book probabilities ----------
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# ---------- 2) Weighted True Probability (NBA tuned) ----------
# NBA: more stable; avoid overfitting L5. Use L10/Season heavier.
W = {
    "L5": 0.20,
    "L10": 0.30,
    "2025": 0.25,
    "HM/AW": 0.10,
    "H2H": 0.05,
    "IM PROB %": 0.10,   # optional “market-like” signal; keeps us anchored
}

def weighted_prob(row):
    num, den = 0.0, 0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w * float(v)
            den += w
    return num/den if den > 0 else np.nan

df["P_TREND"] = df.apply(weighted_prob, axis=1)

# ---------- 3) DVP Matchup Layer (capped) ----------
# Use available dvp cols; compute mean and cap its influence so it can't dominate.
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df.apply(lambda r: safe_mean(r, dvp_cols), axis=1)

def dvp_adjust(p_dvp):
    if pd.isna(p_dvp):
        return 0.0
    # centered around 0.50; cap impact to +/- 0.06 for NBA
    adj = float(p_dvp) - 0.50
    return float(np.clip(adj, -0.06, 0.06))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust)

# ---------- 4) Final Model Probability ----------
df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

# ---------- 5) Edge + EV ----------
df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"] * (df["DEC_ODDS"] - 1.0) - (1.0 - df["MODEL_PROB"])

# ---------- 6) Kelly sizing ----------
df["KELLY_FULL"] = df.apply(lambda r: kelly_fraction(r["MODEL_PROB"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF"] = 0.5 * df["KELLY_FULL"]

# Convert to "units" with a simple bankroll convention:
# If 1u = 1% bankroll, units = kelly_half * 100
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"] * 100).round(2)
df["KELLY_FULL_UNITS"] = (df["KELLY_FULL"] * 100).round(2)

# ---------- 7) Outputs ----------
# Top 10 Most Likely (by MODEL_PROB)
top10_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)

# Top 10 Highest EV (by EV_1U)
top10_ev = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top10_likely[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"
]])

print("\nTOP 10 HIGHEST EV")
display(top10_ev[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"
]])

# Save files
df.to_csv("nba_prop_engine_v1_full_output.csv", index=False)
top10_likely.to_csv("nba_prop_engine_v1_top10_most_likely.csv", index=False)
top10_ev.to_csv("nba_prop_engine_v1_top10_highest_ev.csv", index=False)

print("\nSaved: nba_prop_engine_v1_full_output.csv")
print("Saved: nba_prop_engine_v1_top10_most_likely.csv")
print("Saved: nba_prop_engine_v1_top10_highest_ev.csv")

In [ ]:
# =============================
# NBA PROP ENGINE — MASTER LOAD
# =============================

from google.colab import files
import pandas as pd
import numpy as np
import re

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

# Promote first row to header
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)
df.columns = df.columns.astype(str).str.strip()

# Extract American odds
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'([+-]\d+)', str(x))
    return float(match.group(1)) if match else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Normalize % fields
pct_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H",
            "DVP L5","DVP L10","DVP 2025","DVP HM/AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("NBA data loaded and cleaned.")
print("Rows:", len(df))
df.head()

In [ ]:
# Drop junk columns created by blank header cells
df = df.loc[:, ~df.columns.isna()]  # drop NaN column names
df = df.loc[:, ~df.columns.astype(str).str.lower().isin(["nan"])]  # drop literal "nan"
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # drop Unnamed: 0, etc.

# Strip column names again (safe)
df.columns = df.columns.astype(str).str.strip()

print("Columns now:", df.columns.tolist())
df.head()

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime, timezone

ODDS_API_KEY = "PASTE_YOUR_KEY_HERE"

def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

def fetch_odds_api_nba_markets(regions="us", markets="player_points,player_rebounds,player_assists,player_threes,player_points_rebounds_assists"):
    url = "https://api.the-odds-api.com/v4/sports/basketball_nba/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": "american",
        "dateFormat": "iso"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

odds_json = fetch_odds_api_nba_markets()
print("Games returned:", len(odds_json))

In [ ]:
import numpy as np
import pandas as pd

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 1.0 + (100.0/(-o)) if o < 0 else 1.0 + (o/100.0)

def kelly_fraction(p, dec_odds):
    if pd.isna(p) or pd.isna(dec_odds): return np.nan
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0: return np.nan
    f = (b*p - q)/b
    return max(0.0, f)

# Book probs from your CSV odds
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# NBA weights (avoid overfitting L5)
W = {"L5":0.20, "L10":0.30, "2025":0.25, "HM/AW":0.10, "H2H":0.05, "IM PROB %":0.10}

def weighted_prob(row):
    num=den=0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w*float(v); den += w
    return num/den if den>0 else np.nan

df["P_TREND"] = df.apply(weighted_prob, axis=1)

# DVP capped (NBA)
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True)

def dvp_adjust(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.06, 0.06))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust)

df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1.0) - (1.0-df["MODEL_PROB"])

df["KELLY_FULL"] = df.apply(lambda r: kelly_fraction(r["MODEL_PROB"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF"] = 0.5*df["KELLY_FULL"]
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"]*100).round(2)

top10_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)
top10_ev     = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top10_likely[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","KELLY_HALF_UNITS"]])

print("\nTOP 10 HIGHEST EV")
display(top10_ev[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"]])

df.to_csv("nba_prop_engine_v1_full_output.csv", index=False)
top10_likely.to_csv("nba_prop_engine_v1_top10_most_likely.csv", index=False)
top10_ev.to_csv("nba_prop_engine_v1_top10_highest_ev.csv", index=False)

print("\nSaved: nba_prop_engine_v1_full_output.csv")
print("Saved: nba_prop_engine_v1_top10_most_likely.csv")
print("Saved: nba_prop_engine_v1_top10_highest_ev.csv")

In [ ]:
import requests, re, numpy as np, pandas as pd
from functools import lru_cache

ESPN_NBA_SUMMARY  = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary"
ESPN_NBA_TEAMS    = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_NBA_SCHEDULE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/schedule"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=4096)
def get_team_list():
    return get_json(ESPN_NBA_TEAMS)

@lru_cache(maxsize=4096)
def get_schedule(team_id: str):
    return get_json(ESPN_NBA_SCHEDULE.format(team_id=str(team_id)))

@lru_cache(maxsize=8192)
def get_summary(event_id: str):
    return get_json(ESPN_NBA_SUMMARY, params={"event": str(event_id)})

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_minutes(mmss):
    if mmss is None or (isinstance(mmss,float) and np.isnan(mmss)):
        return np.nan
    s = str(mmss).strip()
    if ":" in s:
        m, sec = s.split(":", 1)
        try: return float(m) + float(sec)/60.0
        except: return np.nan
    try: return float(s)
    except: return np.nan

In [ ]:
# Build abbrev -> team_id map from ESPN
teams_json = get_team_list()

abbrev_to_id = {}
for sp in teams_json.get("sports", []):
    for lg in sp.get("leagues", []):
        for tm in lg.get("teams", []):
            t = tm.get("team", {})
            ab = (t.get("abbreviation") or "").upper()
            tid = t.get("id")
            if ab and tid:
                abbrev_to_id[ab] = str(tid)

print("ESPN teams mapped:", len(abbrev_to_id))
print("Example:", list(abbrev_to_id.items())[:10])

# Parse GAME into away/home abbrevs
def parse_game_abbrevs(game_str):
    s = str(game_str).upper().strip()
    if " AT " in s:
        away, home = s.split(" AT ", 1)
    elif " VS " in s:
        away, home = s.split(" VS ", 1)
    else:
        return None, None
    return away.strip(), home.strip()

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].apply(parse_game_abbrevs))

df["AWAY_TEAM_ID"] = df["AWAY_ABBR"].map(abbrev_to_id)
df["HOME_TEAM_ID"] = df["HOME_ABBR"].map(abbrev_to_id)

missing_team = df[df["AWAY_TEAM_ID"].isna() | df["HOME_TEAM_ID"].isna()][["GAME","AWAY_ABBR","HOME_ABBR","AWAY_TEAM_ID","HOME_TEAM_ID"]].drop_duplicates()
print("Games with missing ESPN team ids:", len(missing_team))
display(missing_team.head(20))

In [ ]:
def completed_event_ids_from_schedule(team_id, max_games=15):
    j = get_schedule(team_id)
    events = j.get("events", []) or []
    evs = []

    for e in events:
        # event id
        eid = e.get("id") or None
        if eid is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            eid = e["competitions"][0].get("id")

        # status
        status = None
        if isinstance(e.get("status"), dict):
            status = (e["status"].get("type") or {}).get("name") or (e["status"].get("type") or {}).get("state")
        if status is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            st = (e["competitions"][0].get("status") or {}).get("type") or {}
            status = st.get("name") or st.get("state")

        st = str(status).upper() if status else ""
        if ("FINAL" not in st) and ("POST" not in st):
            continue

        dt = e.get("date")
        if dt is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            dt = e["competitions"][0].get("date")

        evs.append({"eventId": str(eid), "date": dt})

    # sort most recent first
    evs = sorted(evs, key=lambda x: pd.to_datetime(x["date"], utc=True, errors="coerce"), reverse=True)
    evs = [x for x in evs if x["eventId"] not in ["None","nan",""]]
    return [x["eventId"] for x in evs[:max_games]]

team_ids = pd.unique(pd.concat([df["AWAY_TEAM_ID"], df["HOME_TEAM_ID"]]).dropna()).tolist()
team_recent_events = {}

for tid in team_ids:
    try:
        team_recent_events[str(tid)] = completed_event_ids_from_schedule(str(tid), max_games=15)
    except Exception as e:
        team_recent_events[str(tid)] = []
        print("Schedule fail team", tid, e)

print("Teams pulled:", len(team_recent_events))
print("Example counts:", {k: len(v) for k,v in list(team_recent_events.items())[:5]})

In [ ]:
def extract_minutes_for_player_in_event(event_id: str, player_norm: str):
    j = get_summary(event_id)
    box = j.get("boxscore", {})
    players = box.get("players", [])
    if not players:
        return np.nan

    for team_block in players:
        for stat_group in team_block.get("statistics", []):
            labels = stat_group.get("labels", [])
            if "MIN" not in labels:
                continue
            min_idx = labels.index("MIN")

            for a in stat_group.get("athletes", []):
                ath = a.get("athlete", {}) or {}
                nm = norm_name(ath.get("displayName",""))
                if nm != player_norm:
                    continue
                stats = a.get("stats", [])
                mm = stats[min_idx] if min_idx < len(stats) else None
                return parse_minutes(mm)

    return np.nan

# Build unique player keys from your sheet
df["PLAYER_NORM"] = df["PLAYER"].apply(norm_name)
unique_players = df[["PLAYER","PLAYER_NORM","AWAY_TEAM_ID","HOME_TEAM_ID"]].drop_duplicates()

N_BACK = 10
hist_rows = []

for _, r in unique_players.iterrows():
    pnorm = r["PLAYER_NORM"]
    away_tid = str(r["AWAY_TEAM_ID"]) if pd.notna(r["AWAY_TEAM_ID"]) else None
    home_tid = str(r["HOME_TEAM_ID"]) if pd.notna(r["HOME_TEAM_ID"]) else None

    candidate_events = []
    for tid in [away_tid, home_tid]:
        if tid and tid in team_recent_events:
            candidate_events += team_recent_events[tid][:N_BACK]

    # dedupe preserve order
    seen=set()
    candidate_events = [x for x in candidate_events if not (x in seen or seen.add(x))]
    candidate_events = candidate_events[:N_BACK]

    mins_list = []
    for eid in candidate_events:
        try:
            m = extract_minutes_for_player_in_event(eid, pnorm)
            if pd.notna(m):
                mins_list.append(m)
        except:
            continue

    if len(mins_list) == 0:
        hist_rows.append({"PLAYER_NORM": pnorm, "MIN_L5": np.nan, "MIN_L10": np.nan, "MIN_SD_L10": np.nan, "GAMES_FOUND": 0})
    else:
        arr = np.array(mins_list, dtype=float)
        hist_rows.append({
            "PLAYER_NORM": pnorm,
            "MIN_L5": float(np.nanmean(arr[:5])),
            "MIN_L10": float(np.nanmean(arr[:10])),
            "MIN_SD_L10": float(np.nanstd(arr[:10])) if len(arr[:10]) >= 2 else np.nan,
            "GAMES_FOUND": int(len(arr))
        })

mins_hist_df = pd.DataFrame(hist_rows)
print("Minutes rows:", len(mins_hist_df))
display(mins_hist_df.head(15))

# merge into df
df = df.merge(mins_hist_df, on="PLAYER_NORM", how="left")
print("Rows with GAMES_FOUND>0:", int((df["GAMES_FOUND"].fillna(0)>0).sum()), "of", len(df))

In [ ]:
def role_adj(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = 0.0
    elif min_l10 >= 34:
        base = +0.030
    elif min_l10 >= 30:
        base = +0.015
    elif min_l10 >= 24:
        base = +0.005
    elif min_l10 >= 18:
        base = -0.020
    else:
        base = -0.045

    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.020
    elif sd_l10 >= 5:
        stab = -0.010
    else:
        stab = 0.0

    return base + stab

df["ADJ_ROLE"] = df.apply(lambda r: role_adj(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

df["DNP_RATE_L10"] = (10 - df["GAMES_FOUND"].fillna(0)).clip(0,10) / 10.0
df["MIN_TREND_L5_L10"] = df["MIN_L5"] - df["MIN_L10"]

def inj_proxy(dnp, trend):
    pen = 0.0
    if pd.notna(dnp) and dnp >= 0.3: pen -= 0.02
    if pd.notna(dnp) and dnp >= 0.5: pen -= 0.04
    if pd.notna(trend) and trend <= -4: pen -= 0.02
    if pd.notna(trend) and trend <= -7: pen -= 0.04
    return pen

df["ADJ_INJ_PROXY"] = df.apply(lambda r: inj_proxy(r["DNP_RATE_L10"], r["MIN_TREND_L5_L10"]), axis=1)

# Final ESPN-adjusted model probability
df["MODEL_PROB_ESPN_V1"] = (df["MODEL_PROB"] + df["ADJ_ROLE"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)

# Top 10 Most Likely (role-verified; dedupe by player)
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V1", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10 = df_rank.head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

print(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB","ADJ_ROLE","ADJ_INJ_PROXY","MODEL_PROB_ESPN_V1",
    "MIN_L10","MIN_SD_L10","GAMES_FOUND","MIN_TREND_L5_L10","DNP_RATE_L10"
]].to_string(index=False))

top10.to_csv("nba_prop_engine_v1_top10_most_likely_ESPN_ROLE.csv", index=False)
df.to_csv("nba_prop_engine_v1_full_ESPN_role.csv", index=False)
print("\nSaved: nba_prop_engine_v1_top10_most_likely_ESPN_ROLE.csv")
print("Saved: nba_prop_engine_v1_full_ESPN_role.csv")

In [ ]:
def discord_top10(top10_df):
    lines = ["NBA PROP ENGINE v1 — TOP 10 MOST LIKELY (ESPN ROLE)", "—"]
    for r in top10_df.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME}\n"
            f"Model: {r.MODEL_PROB_ESPN_V1*100:.1f}% | MIN_L10: {r.MIN_L10:.1f} | Trend: {r.MIN_TREND_L5_L10:+.1f}"
        )
    return "\n".join(lines)

print(discord_top10(top10))

In [ ]:
# --- Reduce DVP influence (NBA more efficient than college) ---
def dvp_adjust_nba(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.04, 0.04))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust_nba)

df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.95)


# --- Stronger Role Penalty ---
def role_adj_v2(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = -0.02
    elif min_l10 >= 35:
        base = +0.035
    elif min_l10 >= 32:
        base = +0.020
    elif min_l10 >= 28:
        base = +0.010
    elif min_l10 >= 24:
        base = -0.010
    elif min_l10 >= 20:
        base = -0.035
    else:
        base = -0.060

    # volatility penalty
    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.025
    elif sd_l10 >= 6:
        stab = -0.015
    else:
        stab = 0.0

    return base + stab

df["ADJ_ROLE"] = df.apply(lambda r: role_adj_v2(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

df["MODEL_PROB_ESPN_V2"] = (
    df["MODEL_PROB"] + df["ADJ_ROLE"] + df["ADJ_INJ_PROXY"]
).clip(0.02, 0.90)   # hard NBA realism cap


# Re-rank
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V2", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10_v2 = df_rank.head(10).copy()
top10_v2.insert(0, "RANK", range(1, len(top10_v2)+1))

print(top10_v2[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN_V2",
    "MIN_L10","MIN_SD_L10","MIN_TREND_L5_L10"
]].to_string(index=False))

In [ ]:
# --- Hard minutes ceiling governor ---
def minutes_ceiling(prob, min_l10):
    if pd.isna(min_l10):
        return min(prob, 0.75)
    if min_l10 < 22:
        return min(prob, 0.70)
    if min_l10 < 24:
        return min(prob, 0.75)
    return prob

df["MODEL_PROB_ESPN_V3"] = df.apply(
    lambda r: minutes_ceiling(r["MODEL_PROB_ESPN_V2"], r["MIN_L10"]),
    axis=1
)

# Re-rank
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V3", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10_v3 = df_rank.head(10).copy()
top10_v3.insert(0, "RANK", range(1, len(top10_v3)+1))

print(top10_v3[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN_V3",
    "MIN_L10","MIN_SD_L10"
]].to_string(index=False))

In [ ]:
# Recalculate book probabilities
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 1.0 + (100.0/(-o)) if o < 0 else 1.0 + (o/100.0)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

df["EDGE_VS_BOOK"] = df["MODEL_PROB_ESPN_V3"] - df["BOOK_PROB"]

df["EV_1U"] = df["MODEL_PROB_ESPN_V3"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB_ESPN_V3"])

def kelly(p, d):
    if pd.isna(p) or pd.isna(d): return np.nan
    b = d-1
    q = 1-p
    if b <= 0: return np.nan
    return max(0, (b*p - q)/b)

df["KELLY_HALF"] = 0.5 * df.apply(lambda r: kelly(r["MODEL_PROB_ESPN_V3"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"]*100).round(2)

# Final ranking
final_rank = df[df["GAMES_FOUND"].fillna(0)>=3].copy()
final_rank = final_rank.sort_values("MODEL_PROB_ESPN_V3", ascending=False)
final_rank = final_rank.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10 = final_rank.head(10).copy()
final_top10.insert(0,"RANK",range(1,len(final_top10)+1))

print(final_top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS",
    "MODEL_PROB_ESPN_V3",
    "BOOK_PROB",
    "EDGE_VS_BOOK",
    "KELLY_HALF_UNITS"
]].to_string(index=False))

In [ ]:
def discord_ready(df_block):
    lines = ["NBA PROP ENGINE v1.3 — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.MODEL_PROB_ESPN_V3*100:.1f}% | Edge: {r.EDGE_VS_BOOK*100:+.1f} pts | Half-Kelly: {r.KELLY_HALF_UNITS}u"
        )
    return "\n".join(lines)

print(discord_ready(final_top10))

In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NBA PROP ENGINE — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.MODEL_PROB_CAL*100:.1f}% | Edge: {r.EDGE_CAL*100:+.1f} pts | MIN_L10: {r.MIN_L10:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(final_top10))

In [ ]:
# Ensure book prob exists
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)

# Final probability (no compression)
df["FINAL_PROB"] = df["MODEL_PROB_ESPN_V3"]

# Edge
df["FINAL_EDGE"] = df["FINAL_PROB"] - df["BOOK_PROB"]

# Re-rank
final_rank = df[df["GAMES_FOUND"].fillna(0)>=3].copy()
final_rank = final_rank.sort_values("FINAL_PROB", ascending=False)
final_rank = final_rank.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10 = final_rank.head(10).copy()
final_top10.insert(0, "RANK", range(1, len(final_top10)+1))

print(final_top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS",
    "FINAL_PROB",
    "FINAL_EDGE",
    "MIN_L10"
]].to_string(index=False))

In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NBA PROP ENGINE — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.FINAL_PROB*100:.1f}% | Edge: {r.FINAL_EDGE*100:+.1f} pts | MIN_L10: {r.MIN_L10:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(final_top10))

In [ ]:
variance_mask = df["PROP"].str.contains("BLOCK|STEAL", case=False, na=False)
final_rank_novariance = df[(df["GAMES_FOUND"].fillna(0)>=3) & (~variance_mask)].copy()
final_rank_novariance = final_rank_novariance.sort_values("FINAL_PROB", ascending=False)
final_rank_novariance = final_rank_novariance.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10_novariance = final_rank_novariance.head(10).copy()
final_top10_novariance.insert(0, "RANK", range(1, len(final_top10_novariance)+1))

print(discord_hit_sheet(final_top10_novariance))

In [ ]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "NBA 28 NIGHT.csv"  # uploaded file name

df = pd.read_csv(CSV_PATH)

# drop empty index column if present (common in exports)
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False)]
df.columns = [c.strip() for c in df.columns]

# normalize column names to match the notebook
rename_map = {
    "Player": "PLAYER",
    "IM PROB %": "IM_PROB_PCT",
    "HM/AW": "HM_AW",
    "DVP L5": "DVP_L5",
    "DVP L10": "DVP_L10",
    "DVP 2025": "DVP_2025",
    "DVP HM/AW": "DVP_HM_AW",
    "ODDS": "ODDS_RAW",
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

# parse AMERICAN_ODDS if embedded like "BET\n-169"
if "AMERICAN_ODDS" not in df.columns:
    if "ODDS_RAW" in df.columns:
        def extract_american(x):
            if pd.isna(x): return np.nan
            m = re.search(r"([+-]\d{2,5})", str(x))
            return float(m.group(1)) if m else np.nan
        df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american)
    else:
        df["AMERICAN_ODDS"] = np.nan

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(5))

In [ ]:
import os

fname = "NBA 28 NIGHT.csv"
print("cwd:", os.getcwd())
print("exists:", os.path.exists(fname))
print("size bytes:", os.path.getsize(fname) if os.path.exists(fname) else None)
print("dir sample:", os.listdir(".")[:25])

In [ ]:
# ==============================
# NBA ENRICHMENTS (Pace + Defensive Rating) via NBA Stats API
# Adds: OPP_DEF_RTG, GAME_PACE
# ==============================

import pandas as pd
import numpy as np
import re
import requests

# Use FINAL_PROB if available, otherwise fallback to MODEL_PROB_ESPN_V3
PROB_COL = "FINAL_PROB" if "FINAL_PROB" in df.columns else "MODEL_PROB_ESPN_V3"
if PROB_COL not in df.columns:
    raise ValueError("Need FINAL_PROB or MODEL_PROB_ESPN_V3 in df. Run your base NBA model cells first.")

# Parse "AAA at BBB"
def parse_game_tokens(g):
    s = str(g).strip()
    m = re.match(r"^\s*([A-Z]{2,4})\s+at\s+([A-Z]{2,4})\s*$", s)
    if not m:
        return (None, None)
    return (m.group(1), m.group(2))

away_tokens, home_tokens = zip(*df["GAME"].map(parse_game_tokens))
df["AWAY_ABBR"] = away_tokens
df["HOME_ABBR"] = home_tokens

# NBA Stats headers
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Accept-Language": "en-US,en;q=0.9",
}

def nba_stats_get(url, params):
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# 1) Team list (abbr -> team_id)
teams_url = "https://stats.nba.com/stats/commonteamyears"
teams_js = nba_stats_get(teams_url, {"LeagueID": "00"})
teams_rs = teams_js["resultSets"][0]
teams_df = pd.DataFrame(teams_rs["rowSet"], columns=teams_rs["headers"])
abbr_map = teams_df.set_index("ABBREVIATION")["TEAM_ID"].to_dict()

# 2) Team advanced stats (PACE, DEF_RATING)
adv_url = "https://stats.nba.com/stats/leaguedashteamstats"
adv_js = nba_stats_get(adv_url, {
    "Conference": "",
    "DateFrom": "",
    "DateTo": "",
    "Division": "",
    "GameScope": "",
    "GameSegment": "",
    "LastNGames": "0",
    "LeagueID": "00",
    "Location": "",
    "MeasureType": "Advanced",
    "Month": "0",
    "OpponentTeamID": "0",
    "Outcome": "",
    "PORound": "0",
    "PaceAdjust": "N",
    "PerMode": "PerGame",
    "Period": "0",
    "PlayerExperience": "",
    "PlayerPosition": "",
    "PlusMinus": "N",
    "Rank": "N",
    "Season": "2025-26",     # adjust if needed
    "SeasonSegment": "",
    "SeasonType": "Regular Season",
    "ShotClockRange": "",
    "StarterBench": "",
    "TeamID": "0",
    "TwoWay": "0",
    "VsConference": "",
    "VsDivision": ""
})
adv_rs = adv_js["resultSets"][0]
adv_df = pd.DataFrame(adv_rs["rowSet"], columns=adv_rs["headers"])

team_adv = adv_df[["TEAM_ID","PACE","DEF_RATING"]].copy()
team_adv.rename(columns={"PACE":"TEAM_PACE","DEF_RATING":"TEAM_DEF_RTG"}, inplace=True)

# Map ids
df["HOME_ID"] = df["HOME_ABBR"].map(abbr_map)
df["AWAY_ID"] = df["AWAY_ABBR"].map(abbr_map)

# Merge team pace/def
df = df.merge(team_adv.add_prefix("HOME_"), left_on="HOME_ID", right_on="HOME_TEAM_ID", how="left")
df = df.merge(team_adv.add_prefix("AWAY_"), left_on="AWAY_ID", right_on="AWAY_TEAM_ID", how="left")

# Game pace proxy = average of both teams' pace
df["GAME_PACE"] = (df["HOME_TEAM_PACE"] + df["AWAY_TEAM_PACE"]) / 2.0

# Opponent for a prop row: if we can’t infer player team, use average opponent def (stable)
df["OPP_DEF_RTG"] = (df["HOME_TEAM_DEF_RTG"] + df["AWAY_TEAM_DEF_RTG"]) / 2.0

print("Merged NBA Pace/Def.")
print("Null GAME_PACE:", df["GAME_PACE"].isna().sum(), " / ", len(df))
print("Null OPP_DEF_RTG:", df["OPP_DEF_RTG"].isna().sum(), " / ", len(df))

ValueError: Need FINAL_PROB or MODEL_PROB_ESPN_V3 in df. Run your base NBA model cells first.

In [ ]:
# ==============================
# AUTO-DETECT HIT PROB COLUMN (NBA) + STANDARDIZE TO HIT_PROB
# Run this BEFORE the NBA enrichment cell
# ==============================

import numpy as np
import pandas as pd
import re

print("df shape:", df.shape)
print("df columns:", list(df.columns))

# Priority order (most likely -> least likely)
CANDIDATES = [
    "FINAL_PROB",
    "MODEL_PROB_ESPN_V3",
    "MODEL_PROB_CTX_V2",
    "MODEL_PROB_CTX",
    "MODEL_PROB",
    "PROB",
    "HIT_PROB",
    "P_HIT",
    "WIN_PROB",
]

prob_col = None
for c in CANDIDATES:
    if c in df.columns:
        prob_col = c
        break

# If none matched, try fuzzy find: any column that contains "prob" and looks numeric
if prob_col is None:
    prob_like = [c for c in df.columns if re.search(r"prob", str(c), re.I)]
    for c in prob_like:
        if pd.api.types.is_numeric_dtype(df[c]):
            prob_col = c
            break

# If still none, try hit-rate style columns (convert % -> prob if needed)
if prob_col is None:
    hit_like = [c for c in df.columns if re.search(r"hit|rate|pct|percent", str(c), re.I)]
    for c in hit_like:
        if pd.api.types.is_numeric_dtype(df[c]):
            prob_col = c
            break

if prob_col is None:
    raise ValueError(
        "Could not find a probability/hit-rate column in df.\n"
        "Run your base hit-rater/model cells first OR tell me which column is your probability."
    )

print("Using probability column:", prob_col)

# Standardize to HIT_PROB
df["HIT_PROB"] = df[prob_col].astype(float)

# If it looks like percentages (0–100), convert to 0–1
if df["HIT_PROB"].max() > 1.5:
    df["HIT_PROB"] = df["HIT_PROB"] / 100.0

df["HIT_PROB"] = df["HIT_PROB"].clip(0.01, 0.99)

print("HIT_PROB summary:")
display(df["HIT_PROB"].describe())

df shape: (101, 17)
df columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']


ValueError: Could not find a probability/hit-rate column in df.
Run your base hit-rater/model cells first OR tell me which column is your probability.

In [ ]:
# ==============================
# FIX NBA CSV LOAD (auto-detect header row) + rebuild df
# ==============================

import pandas as pd
import numpy as np
import os, re

CSV_PATH = "NBA 3:1 props.csv"   # if in Colab root; otherwise update to your path

# If you're unsure path, show files
print("Files in cwd:", [f for f in os.listdir(".") if f.lower().endswith(".csv")])

def find_header_row(path, max_rows=30):
    """
    Find the first row that looks like a header.
    We look for any of these tokens: player, game, prop, outcome, odds
    """
    preview = pd.read_csv(path, header=None, nrows=max_rows, dtype=str, engine="python")
    key_re = re.compile(r"(player|name|game|match|prop|outcome|odds|american)", re.I)
    best_i, best_score = None, -1
    for i in range(len(preview)):
        row = preview.iloc[i].fillna("").astype(str).tolist()
        score = sum(1 for x in row if key_re.search(x.strip()))
        if score > best_score:
            best_score = score
            best_i = i
    return best_i, best_score, preview.iloc[best_i].tolist()

hdr_i, score, hdr_row = find_header_row(CSV_PATH)
print("Detected header row index:", hdr_i, "score:", score)
print("Header row sample:", hdr_row[:12])

# Load using detected header row
df = pd.read_csv(CSV_PATH, header=hdr_i)

# Normalize column names
def norm_col(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    c = c.replace("__", "_")
    return c

df.columns = [norm_col(c) for c in df.columns]

# Drop totally empty columns
df = df.loc[:, ~df.columns.str.match(r"^Unnamed", case=False)]
df = df.dropna(axis=1, how="all")

print("Loaded fixed df shape:", df.shape)
print("Columns:", list(df.columns))

# Quick checks for required fields (we'll also try to rename common variants)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if lc in ["player", "player_name", "name"]: rename_map[c] = "PLAYER"
    if lc in ["game", "matchup", "match"]: rename_map[c] = "GAME"
    if "prop" in lc: rename_map[c] = "PROP"
    if lc in ["outcome", "side", "pick"]: rename_map[c] = "OUTCOME"
    if "american" in lc or lc in ["odds", "american_odds"]: rename_map[c] = "AMERICAN_ODDS"

df = df.rename(columns=rename_map)

print("After rename columns:", list(df.columns))

need = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]
missing = [c for c in need if c not in df.columns]
print("Missing required:", missing)

# Show top rows so you can confirm visually
display(df.head(5))

Files in cwd: ['NBA 3:1 props.csv']
Detected header row index: 1 score: 5
Header row sample: [nan, 'PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW']
Loaded fixed df shape: (100, 16)
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM/AW']
After rename columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'AMERICAN_ODDS', 'AVG', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM/AW']
Missing required: []


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,AVG,IM_PROB_%,L5,L10,2025,HM/AW,H2H,DVP_L5,DVP_L10,DVP_2025,DVP_HM/AW
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,BET\n-110,-1102,52.4%,100%,100%,100%,100%,100%,100%,100%,100%,100%
1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,BET\n+105,1052,48.8%,100%,100%,100%,100%,100%,100%,100%,100%,100%
2,Jarrett Allen,CLE at BKN,Rebounds,o0.5,BET\n-145,-1452,59.2%,100%,100%,100%,100%,100%,100%,100%,98.3%,96.6%
3,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,BET\n-129,-1304,56.3%,100%,90%,97.7%,100%,100%,20%,30%,15.5%,13.3%
4,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,BET\n-126,-1282,55.8%,100%,90%,95.3%,100%,100%,20%,30%,17.2%,16.7%


In [ ]:
# ==============================
# CLEAN NBA CSV (odds + % cols) + BUILD HIT_PROB (hit-rater model)
# ==============================

import numpy as np
import pandas as pd
import re

# --- 1) Fix AMERICAN_ODDS ---
def extract_american_odds(x):
    s = str(x)
    m = re.search(r"([+-]\d{2,4})", s.replace("−","-"))
    if m:
        return int(m.group(1))
    # sometimes odds got stuck like -1102 or 1052: take first 3-4 digits with sign
    m2 = re.search(r"([+-]\d{2,4})\d+", s.replace("−","-"))
    if m2:
        return int(m2.group(1))
    return np.nan

df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)

# Drop rows with no odds
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

# --- 2) Convert percent-like columns to floats in [0,1] ---
pct_cols = [c for c in df.columns if c in ["IM_PROB_%","L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]]

def pct_to_prob(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    s = s.replace("%","")
    s = s.replace("—","").replace("–","").replace("nan","")
    if s == "":
        return np.nan
    try:
        return float(s)/100.0
    except:
        return np.nan

for c in pct_cols:
    df[c] = df[c].apply(pct_to_prob)

# --- 3) Create BOOK_PROB from odds (market implied) ---
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# --- 4) Build HIT_PROB from hit-rate features (symmetrical to NCAA) ---
# We’ll use a stable weighted blend with light DVP influence.
BASE_WEIGHTS = {
    "L5": 0.18,
    "L10": 0.22,
    "2025": 0.18,
    "HM/AW": 0.10,
    "H2H": 0.08,
    "DVP_L5": 0.08,
    "DVP_L10": 0.08,
    "DVP_2025": 0.06,
    "DVP_HM/AW": 0.02,
}

# Fill missing rates with column means (stable)
for k in BASE_WEIGHTS.keys():
    if k in df.columns:
        df[k] = df[k].astype(float)
        df[k] = df[k].fillna(df[k].mean())

# Weighted average
num = 0.0
den = 0.0
for k,w in BASE_WEIGHTS.items():
    if k in df.columns:
        num += w * df[k]
        den += w
df["HIT_PROB"] = (num/den).clip(0.01, 0.99)

# --- 5) Quick sanity prints ---
print("Rows:", len(df))
print("Odds sample:", df["AMERICAN_ODDS"].head(10).tolist())
print("HIT_PROB range:", (df["HIT_PROB"].min(), df["HIT_PROB"].max()))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","HIT_PROB"]].head(10))

Rows: 100
Odds sample: [-110, 105, -145, -129, -126, -125, -130, -142, -101, -115]
HIT_PROB range: (0.54434, 0.99)


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,HIT_PROB
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,0.523810,0.99000
1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,0.487805,0.99000
2,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,0.591837,0.99000
3,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,-129,0.563319,0.78582
4,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.557522,0.78320
5,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,-125,0.555556,0.76760
6,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,0.565217,0.76184
7,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.586777,0.76676
8,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.502488,0.78552
9,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.534884,0.75420


In [ ]:
# ==============================
# NBA MATCHUP ENRICHMENT (Pace + Defensive Rating) via NBA Stats API
# ==============================

import pandas as pd
import numpy as np
import re
import requests

# Parse game tokens: supports "MIN at DEN" and "MIN @ DEN"
def parse_game_tokens(g):
    s = str(g).strip().upper()
    m = re.match(r"^\s*([A-Z]{2,4})\s+(?:AT|@)\s+([A-Z]{2,4})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None, None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Accept-Language": "en-US,en;q=0.9",
}

def nba_stats_get(url, params):
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# Team list (abbr -> team_id)
teams_url = "https://stats.nba.com/stats/commonteamyears"
teams_js = nba_stats_get(teams_url, {"LeagueID": "00"})
teams_rs = teams_js["resultSets"][0]
teams_df = pd.DataFrame(teams_rs["rowSet"], columns=teams_rs["headers"])
abbr_map = teams_df.set_index("ABBREVIATION")["TEAM_ID"].to_dict()

# Team advanced stats (PACE, DEF_RATING)
adv_url = "https://stats.nba.com/stats/leaguedashteamstats"
adv_js = nba_stats_get(adv_url, {
    "LeagueID": "00",
    "Season": "2025-26",
    "SeasonType": "Regular Season",
    "PerMode": "PerGame",
    "MeasureType": "Advanced",
    "LastNGames": "0",
    "Month": "0",
    "OpponentTeamID": "0",
    "TeamID": "0",
    "PaceAdjust": "N",
    "PlusMinus": "N",
    "Rank": "N",
    "Conference": "",
    "Division": "",
    "Location": "",
    "Outcome": "",
    "PORound": "0",
    "Period": "0",
    "GameSegment": "",
    "DateFrom": "",
    "DateTo": "",
    "ShotClockRange": "",
    "PlayerExperience": "",
    "PlayerPosition": "",
    "StarterBench": "",
    "TwoWay": "0",
    "VsConference": "",
    "VsDivision": ""
})
adv_rs = adv_js["resultSets"][0]
adv_df = pd.DataFrame(adv_rs["rowSet"], columns=adv_rs["headers"])
team_adv = adv_df[["TEAM_ID","PACE","DEF_RATING"]].copy()
team_adv.rename(columns={"PACE":"TEAM_PACE","DEF_RATING":"TEAM_DEF_RTG"}, inplace=True)

# Map ids and merge
df["HOME_ID"] = df["HOME_ABBR"].map(abbr_map)
df["AWAY_ID"] = df["AWAY_ABBR"].map(abbr_map)

df = df.merge(team_adv.add_prefix("HOME_"), left_on="HOME_ID", right_on="HOME_TEAM_ID", how="left")
df = df.merge(team_adv.add_prefix("AWAY_"), left_on="AWAY_ID", right_on="AWAY_TEAM_ID", how="left")

df["GAME_PACE"] = (df["HOME_TEAM_PACE"] + df["AWAY_TEAM_PACE"]) / 2.0

# opponent DEF proxy (stable if we can't infer player team)
df["OPP_DEF_RTG"] = (df["HOME_TEAM_DEF_RTG"] + df["AWAY_TEAM_DEF_RTG"]) / 2.0

print("Merged NBA matchup context.")
print("Null GAME_PACE:", df["GAME_PACE"].isna().sum(), "/", len(df))
print("Null OPP_DEF_RTG:", df["OPP_DEF_RTG"].isna().sum(), "/", len(df))
display(df[["GAME","AWAY_ABBR","HOME_ABBR","GAME_PACE","OPP_DEF_RTG"]].drop_duplicates().head(10))

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
# ==============================
# MATCHUP PROXIES (NO NBA STATS CALLS)
# Uses DVP_* fields + HM/AW + season to create:
#   OPP_DEF_PROXY  (higher = tougher)
#   PACE_PROXY     (higher = faster)
# ==============================

import numpy as np
import pandas as pd

# Ensure DVP columns exist (they do in your CSV)
need = ["DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW","HM/AW","2025"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Missing columns for proxies: {miss}")

# DVP proxies: treat as "opponent difficulty" signals; blend them
# (We only need a stable relative number to shift μ.)
df["OPP_DEF_PROXY_RAW"] = (
    0.30*df["DVP_L10"] +
    0.25*df["DVP_2025"] +
    0.20*df["DVP_L5"] +
    0.15*df["DVP_HM/AW"] +
    0.10*df["HM/AW"]
)

# Convert to a 0–100-ish scale where higher = tougher
# Use rank-based percentile to avoid weird units
df["OPP_DEF_PROXY"] = df["OPP_DEF_PROXY_RAW"].rank(pct=True) * 100.0

# Pace proxy: if you don't have a pace column, use a stable constant + minor spread from matchup percentile
# (Once NBA Stats works, this gets replaced by real pace.)
df["PACE_PROXY"] = 100.0 + (df["OPP_DEF_PROXY"].rank(pct=True) - 0.5) * 10.0  # 95–105 range

print("Proxy context built (no NBA Stats).")
display(df[["GAME","PLAYER","PROP","OPP_DEF_PROXY","PACE_PROXY"]].head(8))

Proxy context built (no NBA Stats).


,GAME,PLAYER,PROP,OPP_DEF_PROXY,PACE_PROXY
0,MIN at DEN,Anthony Edwards,Points + Rebounds + Assists,99.5,104.95
1,CLE at BKN,Michael Porter Jr.,Rebounds + Assists,99.5,104.95
2,CLE at BKN,Jarrett Allen,Rebounds,98.0,104.80
3,MIL at CHI,Rob Dillingham,Points + Rebounds + Assists,20.0,97.00
4,MIL at CHI,Rob Dillingham,Points + Assists,21.0,97.10
5,MEM at IND,Walter Clayton Jr.,Points + Rebounds + Assists,10.0,96.00
6,MEM at IND,Walter Clayton Jr.,Points,15.0,96.50
7,MEM at IND,Scotty Pippen Jr.,Rebounds,32.0,98.20


In [ ]:
# ==============================
# NBA FULL PIPELINE (NO NBA STATS)
# Uses:
#  HIT_PROB (from hit-rater blend cell)
#  OPP_DEF_PROXY + PACE_PROXY (from proxy cell)
# Outputs:
#  Elite 3–5 + Kelly units + clean Discord
# ==============================

import numpy as np
import pandas as pd
import re
from scipy.stats import norm, poisson, nbinom

# ---- helpers ----
def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

def dist_for_market(mkey):
    if mkey in ["three","steals","blocks"]:
        return "poisson"
    if mkey in ["assists","rebounds"]:
        return "nbinom"
    return "normal"

def nb_params_from_mu_k(mu, k):
    mu = max(0.05, float(mu))
    k = max(0.5, float(k))
    p = k/(k+mu)
    n = k
    return n, p

def poisson_prob_over(line, lam):
    k = int(np.floor(line))
    return float(1 - poisson.cdf(k, mu=lam))

def poisson_lam_from_pover(line, p_over):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(12.0, line + 12.0)
    for _ in range(60):
        mid = (lo+hi)/2
        p = poisson_prob_over(line, mid)
        if p > p_over: hi = mid
        else: lo = mid
    return float((lo+hi)/2)

def nb_k_for_market(mkey):
    return 4.0 if mkey=="assists" else 5.0

def nb_prob_over(line, mu, k):
    n,p = nb_params_from_mu_k(mu, k)
    cut = int(np.floor(line))
    return float(1 - nbinom.cdf(cut, n=n, p=p))

def nb_mu_from_pover(line, p_over, k):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(18.0, line + 18.0)
    for _ in range(60):
        mid = (lo+hi)/2
        p = nb_prob_over(line, mid, k)
        if p > p_over: hi = mid
        else: lo = mid
    return float((lo+hi)/2)

def normal_sigma(mkey, line):
    line = float(line)
    if mkey in ["pra","pr","pa","ra"]: return max(4.0, 0.17*line)
    if mkey == "points": return max(3.0, 0.18*line)
    return max(2.5, 0.18*max(line,1.0))

def normal_prob_over(line, mu, sigma):
    x = ((line + 0.5) - mu)/sigma
    return float(1 - norm.cdf(x))

def normal_mu_from_pover(line, p_over, sigma):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    z = norm.ppf(1 - p_over)
    return float((line + 0.5) - sigma*z)

def prob_from_mu(line, mu, dist_used, side, aux=np.nan, mkey="other"):
    if dist_used == "poisson":
        p_over = poisson_prob_over(line, mu)
    elif dist_used == "nbinom":
        k = float(aux) if pd.notna(aux) else nb_k_for_market(mkey)
        p_over = nb_prob_over(line, mu, k)
    else:
        sigma = float(aux) if pd.notna(aux) else normal_sigma(mkey, line)
        p_over = normal_prob_over(line, mu, sigma)
    return p_over if side=="over" else (1-p_over)

# ---- required columns ----
for c in ["HIT_PROB","BOOK_PROB","DEC_ODDS","OPP_DEF_PROXY","PACE_PROXY"]:
    if c not in df.columns:
        raise ValueError(f"Missing {c}. Run the prior cells first.")

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))
df["MKT"] = df["PROP"].map(market_key)

# ---- invert HIT_PROB -> μ_base ----
mu_base=[]; dist_used=[]; aux=[]
for r in df.itertuples(index=False):
    mkey = r.MKT
    dist = dist_for_market(mkey)
    dist_used.append(dist)

    p_over = r.HIT_PROB if r.SIDE=="over" else (1-r.HIT_PROB)
    line = float(r.LINE)

    if dist == "poisson":
        mu = poisson_lam_from_pover(line, p_over)
        mu_base.append(mu); aux.append(np.nan)
    elif dist == "nbinom":
        k = nb_k_for_market(mkey)
        mu = nb_mu_from_pover(line, p_over, k)
        mu_base.append(mu); aux.append(k)
    else:
        sigma = normal_sigma(mkey, line)
        mu = normal_mu_from_pover(line, p_over, sigma)
        mu_base.append(mu); aux.append(sigma)

df["DIST_USED"]=dist_used
df["AUX_PARAM"]=aux
df["PROJ_MU_BASE"]=mu_base

# ---- CTX-to-μ using proxies ----
BASE_DEF = float(df["OPP_DEF_PROXY"].median())   # ~50
BASE_PACE = float(df["PACE_PROXY"].median())     # ~100

# proxy coefficients
BETA = {
    "points": (0.55, 0.35),
    "rebounds": (0.20, 0.15),
    "assists": (0.18, 0.14),
    "three": (0.10, 0.08),
    "steals": (0.06, 0.05),
    "blocks": (0.06, 0.05),
    "pra": (0.85, 0.55),
    "pr": (0.70, 0.45),
    "pa": (0.70, 0.45),
    "ra": (0.55, 0.35),
    "other": (0.35, 0.25),
}
MU_CAP = {
    "points": 2.0, "rebounds": 1.2, "assists": 1.2, "three": 0.8,
    "steals": 0.5, "blocks": 0.5, "pra": 3.0, "pr": 2.5, "pa": 2.5, "ra": 2.0, "other": 1.5
}
def clip(v, lo, hi): return float(max(lo, min(hi, v)))

mu_ctx=[]; mu_shift=[]
for r in df.itertuples(index=False):
    mkey = r.MKT
    b_def, b_pace = BETA.get(mkey, BETA["other"])
    cap = MU_CAP.get(mkey, MU_CAP["other"])

    d_def = float(r.OPP_DEF_PROXY) - BASE_DEF      # +/- ~50
    d_pace = float(r.PACE_PROXY) - BASE_PACE       # +/- ~5

    # scale def by 25, pace by 5 (proxy units)
    shift = (b_def*(d_def/25.0)) + (b_pace*(d_pace/5.0))
    shift = clip(shift, -cap, cap)

    mu_shift.append(shift)
    mu_ctx.append(float(r.PROJ_MU_BASE) + shift)

df["PROJ_MU_SHIFT"]=mu_shift
df["PROJ_MU_CTX"]=mu_ctx

# ---- probability from μ_ctx ----
df["PROJ_PROB_CTX"] = [
    np.clip(prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT), 0.01, 0.99)
    for r in df.itertuples(index=False)
]

# ---- blend + Mode A shrink ----
W_HIT, W_PROJ = 0.70, 0.30
df["BLEND_PROB"] = np.clip(W_HIT*df["HIT_PROB"] + W_PROJ*df["PROJ_PROB_CTX"], 0.01, 0.99)

SHRINK_TO_BOOK = 0.25
df["SAFE_PROB"] = np.clip((1-SHRINK_TO_BOOK)*df["BLEND_PROB"] + SHRINK_TO_BOOK*df["BOOK_PROB"], 0.01, 0.99)
df["EV_SAFE_1U"] = df["SAFE_PROB"]*(df["DEC_ODDS"]-1) - (1-df["SAFE_PROB"])

# ---- injury sensitivity scenarios ----
UP_MULT  = {"points":1.06, "assists":1.07, "rebounds":1.04, "three":1.05, "pra":1.06, "pr":1.05, "pa":1.05, "ra":1.05, "other":1.05}
DOWN_MULT= {"points":0.92, "assists":0.90, "rebounds":0.93, "three":0.92, "pra":0.91, "pr":0.92, "pa":0.92, "ra":0.92, "other":0.92}

def apply_mu_scenario(mu, mkt, mode):
    if mode=="up": return float(mu)*UP_MULT.get(mkt,1.05)
    if mode=="down": return float(mu)*DOWN_MULT.get(mkt,0.92)
    return float(mu)

p_up=[]; p_dn=[]
for r in df.itertuples(index=False):
    mu0 = r.PROJ_MU_CTX
    pu = prob_from_mu(r.LINE, apply_mu_scenario(mu0, r.MKT, "up"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT)
    pdn= prob_from_mu(r.LINE, apply_mu_scenario(mu0, r.MKT, "down"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT)
    p_up.append(pu); p_dn.append(pdn)

df["INJ_SENS_PTS"] = (np.array(p_up) - np.array(p_dn)) * 100
df["INJ_SENS_ABS"] = df["INJ_SENS_PTS"].abs()
df["MU_SHIFT_ABS"] = df["PROJ_MU_SHIFT"].abs()

df["SAFE_SCORE"] = (df["EV_SAFE_1U"]*100) - (df["INJ_SENS_ABS"]*1.25) - (df["MU_SHIFT_ABS"]*6.0)

# ---- Elite 3–5 filter + Kelly units ----
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_INJ  = 10.0

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV) &
    (df["INJ_SENS_ABS"] <= MAX_INJ)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(5).copy()
elite["RANK"] = range(1, len(elite)+1)

HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# ---- Clean Discord output ----
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
1,1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,0.37
0,2,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,0.37
2,3,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,0.36
38,4,Kobe Brown,MEM at IND,Assists,u1.5,130,0.25
8,5,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25


NBA ELITE PROP CARD
—
1) Michael Porter Jr. — CLE at BKN
Rebounds + Assists o0.5 (105) — 0.37u
2) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o0.5 (-110) — 0.37u
3) Jarrett Allen — CLE at BKN
Rebounds o0.5 (-145) — 0.36u
4) Kobe Brown — MEM at IND
Assists u1.5 (130) — 0.25u
5) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u


In [ ]:
# ==============================
# FIX BAD o0.5 LINES (NBA)
# - For PRA/RA/PR/PA/Reb/Ast/Pts: o0.5 is invalid -> rebuild line from AVG
# - Keep o0.5 ONLY for low-count markets if it makes sense (e.g., blocks/steals/threes)
# Then rerun elite extraction + Discord output
# ==============================

import numpy as np
import pandas as pd
import re

# 1) ensure AVG numeric
def to_float(x):
    if pd.isna(x): return np.nan
    s=str(x).strip().replace("%","")
    s=re.sub(r"[^0-9\.\-]", "", s)
    try: return float(s)
    except: return np.nan

df["AVG_NUM"] = df["AVG"].apply(to_float)

# 2) market key helper (must match pipeline)
def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

# 3) parse outcome safely
def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# 4) identify bogus o0.5 for markets where it makes no sense
ALLOW_HALF_OVER = set(["three","steals","blocks"])   # can be o0.5 in these markets
bad = (
    (df["SIDE"]=="over") &
    (df["LINE"]==0.5) &
    (~df["MKT"].isin(ALLOW_HALF_OVER))
)

print("Bad o0.5 rows detected:", int(bad.sum()))
display(df.loc[bad, ["PLAYER","GAME","PROP","OUTCOME","AVG","AVG_NUM"]].head(10))

# 5) rebuild line from AVG for bad rows
# Use: line = round_to_half(AVG) with minimum reasonable floors by market
FLOOR = {
    "points": 6.5,
    "rebounds": 2.5,
    "assists": 1.5,
    "pra": 14.5,
    "pr": 10.5,
    "pa": 10.5,
    "ra": 6.5,
    "other": 4.5
}

def round_to_half(x):
    return np.round(x*2)/2

for idx in df.index[bad]:
    mkt = df.at[idx, "MKT"]
    avg = df.at[idx, "AVG_NUM"]
    if pd.isna(avg):
        # if no AVG, just drop later
        df.at[idx, "LINE"] = np.nan
        continue
    line = float(round_to_half(avg))
    line = max(line, FLOOR.get(mkt, 4.5))
    df.at[idx, "LINE"] = line
    df.at[idx, "OUTCOME"] = f"o{line:g}"  # keep over but correct line

# 6) drop any remaining nonsense (missing line)
pre = len(df)
df = df[df["LINE"].notna()].copy()
print("Dropped rows missing LINE:", pre - len(df))

# 7) rerun: recompute HIT_PROB / BOOK_PROB etc already exist; just rerun the elite pipeline parts
# (Assumes you've already run the pipeline cell that creates PROJ_MU_BASE etc; if not, rerun after this fix.)

print("Outcome/Line fixed. Sample:")
display(df[["PLAYER","PROP","OUTCOME","LINE","MKT","AVG_NUM"]].head(12))

Bad o0.5 rows detected: 4


,PLAYER,GAME,PROP,OUTCOME,AVG,AVG_NUM
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-1102,-1102.0
1,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,1052,1052.0
2,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-1452,-1452.0
14,Obi Toppin,MEM at IND,Assists,o0.5,-1963,-1963.0


Dropped rows missing LINE: 0
Outcome/Line fixed. Sample:


,PLAYER,PROP,OUTCOME,LINE,MKT,AVG_NUM
0,Anthony Edwards,Points + Rebounds + Assists,o14.5,14.5,pra,-1102.0
1,Michael Porter Jr.,Rebounds + Assists,o1052,1052.0,ra,1052.0
2,Jarrett Allen,Rebounds,o2.5,2.5,rebounds,-1452.0
3,Rob Dillingham,Points + Rebounds + Assists,u16.5,16.5,pra,-1304.0
4,Rob Dillingham,Points + Assists,u13.5,13.5,pa,-1282.0
5,Walter Clayton Jr.,Points + Rebounds + Assists,u19.5,19.5,pra,-1254.0
6,Walter Clayton Jr.,Points,u11.5,11.5,points,-1448.0
7,Scotty Pippen Jr.,Rebounds,u3.5,3.5,rebounds,-1839.0
8,Rob Dillingham,Points,u8.5,8.5,points,-11012.0
9,Walter Clayton Jr.,Points + Rebounds,u14.5,14.5,pr,-1225.0


In [ ]:
# ==============================
# FINAL FIX: DROP bogus o0.5 rows (non-count markets) + resanitize odds
# Then rerun your pipeline cell
# ==============================

import numpy as np
import pandas as pd
import re

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def extract_american_odds(x):
    s = str(x)
    # prefer the first true odds token like -110 or +105
    m = re.search(r"([+-]\d{2,4})", s.replace("−","-"))
    if m:
        return int(m.group(1))
    # fallback: if it's numeric but too long, clip to 3-4 digits keeping sign
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2:
        return int(m2.group(1) + m2.group(2))
    return np.nan

# Recompute market + line
df["MKT"] = df["PROP"].map(market_key)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# Identify bogus 0.5 overs for non-count markets
ALLOW_HALF_OVER = set(["three","steals","blocks"])
bad = (df["SIDE"]=="over") & (df["LINE"]==0.5) & (~df["MKT"].isin(ALLOW_HALF_OVER))

print("Dropping bogus 0.5 rows:", int(bad.sum()))
display(df.loc[bad, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]])

df = df.loc[~bad].copy()

# Re-sanitize odds (handles any -11012 type junk)
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

# Rebuild book probs / decimal odds
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Post-fix rows:", len(df))
print("Odds sample:", df["AMERICAN_ODDS"].head(12).tolist())
print("Outcome sample:", df["OUTCOME"].head(12).tolist())

# IMPORTANT:
# Now rerun your long "NBA FULL PIPELINE" cell (μ + ctx + Mode A + Kelly + Discord)

Dropping bogus 0.5 rows: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


Post-fix rows: 85
Odds sample: [-110, -145, -129, -126, -125, -130, -142, -101, -115, -105, -110, -145]
Outcome sample: ['o14.5', 'o2.5', 'u16.5', 'u13.5', 'u19.5', 'u11.5', 'u3.5', 'u8.5', 'u14.5', 'u15.5', 'u18.5', 'u7.5']


In [ ]:
df.loc[df["LINE"] < 1, ["PLAYER","PROP","OUTCOME","AMERICAN_ODDS"]].head(20)

,PLAYER,PROP,OUTCOME,AMERICAN_ODDS
34,DeAndre Jordan,Assists,u0.5,-115
75,Tristan da Silva,Blocks,u0.5,-200
81,Dyson Daniels,Three Pointers Made,u0.5,-200


In [ ]:
# ==============================
# 0.5 LINE RULES (NBA)
# - Always allow 0.5 for three/steals/blocks (over or under)
# - For points/rebounds/assists/combos:
#     allow ONLY under 0.5
#     drop over 0.5
# ==============================

import numpy as np
import pandas as pd
import re

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["MKT"] = df["PROP"].map(market_key)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

half_ok_any = set(["three","steals","blocks"])
half_noncount = set(["points","rebounds","assists","pra","pr","pa","ra","other"])

drop_over_half = (
    (df["LINE"] == 0.5) &
    (df["SIDE"] == "over") &
    (df["MKT"].isin(half_noncount))
)

print("Dropping OVER 0.5 rows (non-count markets):", int(drop_over_half.sum()))
display(df.loc[drop_over_half, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]].head(20))

df = df.loc[~drop_over_half].copy()

# Optional: if you want to exclude assists u0.5 entirely, uncomment:
# df = df[~((df["MKT"]=="assists") & (df["LINE"]==0.5))].copy()

print("Remaining 0.5 lines sample:")
display(df.loc[df["LINE"]==0.5, ["PLAYER","PROP","OUTCOME","AMERICAN_ODDS","MKT"]].head(20))

Dropping OVER 0.5 rows (non-count markets): 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


Remaining 0.5 lines sample:


,PLAYER,PROP,OUTCOME,AMERICAN_ODDS,MKT
34,DeAndre Jordan,Assists,u0.5,-115,assists
75,Tristan da Silva,Blocks,u0.5,-200,blocks
81,Dyson Daniels,Three Pointers Made,u0.5,-200,three


In [ ]:
# ==============================
# MODE B: CLEAN 0.5 LINES
# - Keep 0.5 only for: three/blocks/steals
# - Drop 0.5 for assists/rebounds/points/combos
# ==============================

drop_half_noncount = (
    (df["LINE"] == 0.5) &
    (~df["MKT"].isin(["three","blocks","steals"]))
)

print("Dropping 0.5 lines outside count markets:", int(drop_half_noncount.sum()))
display(df.loc[drop_half_noncount, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MKT"]].head(20))

df = df.loc[~drop_half_noncount].copy()

print("Remaining 0.5 lines:")
display(df.loc[df["LINE"]==0.5, ["PLAYER","PROP","OUTCOME","AMERICAN_ODDS","MKT"]].head(20))

Dropping 0.5 lines outside count markets: 1


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MKT
34,DeAndre Jordan,NOP at LAC,Assists,u0.5,-115,assists


Remaining 0.5 lines:


,PLAYER,PROP,OUTCOME,AMERICAN_ODDS,MKT
75,Tristan da Silva,Blocks,u0.5,-200,blocks
81,Dyson Daniels,Three Pointers Made,u0.5,-200,three


In [ ]:
# ==============================
# MODE B: ELITE CARD + KELLY + CLEAN DISCORD (no re-run needed)
# Requires df already has SAFE_PROB, EV_SAFE_1U, SAFE_SCORE, DEC_ODDS, AMERICAN_ODDS
# ==============================

import numpy as np

# If these aren't present, rerun the big pipeline cell once.
req = ["SAFE_PROB","EV_SAFE_1U","SAFE_SCORE","DEC_ODDS","AMERICAN_ODDS","PLAYER","GAME","PROP","OUTCOME"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns {missing}. Rerun the big pipeline cell once after Mode B filter.")

MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_PLAYS = 5

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"] = range(1, len(elite)+1)

# Half Kelly units (keep unsnapped units like NCAA)
HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# Clean Discord output
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.37
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.36
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25


NBA ELITE PROP CARD
—
1) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o14.5 (-110) — 0.37u
2) Jarrett Allen — CLE at BKN
Rebounds o2.5 (-145) — 0.36u
3) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
5) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u


In [ ]:
# ==============================
# TEMP EXCLUDE: Anthony Edwards PRA o14.5
# Then rebuild elite card
# ==============================

import numpy as np
import pandas as pd

# Remove that specific corrupted row
mask_exclude = (
    (df["PLAYER"].str.contains("Anthony Edwards", case=False, na=False)) &
    (df["PROP"].str.contains("Points + Rebounds + Assists", case=False, na=False)) &
    (df["LINE"] == 14.5)
)

print("Excluding rows:", int(mask_exclude.sum()))
display(df.loc[mask_exclude, ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]])

df = df.loc[~mask_exclude].copy()

# ---- rebuild elite card ----
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_PLAYS = 5

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"] = range(1, len(elite)+1)

# Half Kelly
HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# Clean Discord output
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )

print("\n".join(lines))

Excluding rows: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.37
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.36
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25


NBA ELITE PROP CARD
—
1) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o14.5 (-110) — 0.37u
2) Jarrett Allen — CLE at BKN
Rebounds o2.5 (-145) — 0.36u
3) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
5) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u


In [ ]:
# ==============================
# GUARANTEED EXCLUDE: Edwards PRA o14.5 (use OUTCOME text)
# ==============================

import pandas as pd

# 1) confirm the row exists in df
target = df[
    df["PLAYER"].str.contains("Anthony Edwards", case=False, na=False) &
    df["PROP"].str.contains("Points + Rebounds + Assists", case=False, na=False) &
    df["OUTCOME"].astype(str).str.strip().str.lower().eq("o14.5")
].copy()

print("Matches in df:", len(target))
display(target[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS"]].head(10))

# 2) drop it
df = df.drop(target.index).copy()

# 3) rebuild elite from scratch (no stale elite variable)
MIN_PROB = 0.60
MIN_EV   = 0.08
MAX_PLAYS = 5

elite = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"] = range(1, len(elite)+1)

# Half Kelly
HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f)*HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

elite["UNITS"] = [half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# Discord output
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in elite.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

Matches in df: 0


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.37
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.36
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25


NBA ELITE PROP CARD
—
1) Anthony Edwards — MIN at DEN
Points + Rebounds + Assists o14.5 (-110) — 0.37u
2) Jarrett Allen — CLE at BKN
Rebounds o2.5 (-145) — 0.36u
3) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
5) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u


In [ ]:
# ==============================
# Remove Edwards + Allen, then pull next two plays (old ranks 6 & 7)
# ==============================

import numpy as np

# 1) Build the full ranked board exactly like the last card selection
MIN_PROB = 0.60
MIN_EV   = 0.08

board = df[
    (df["SAFE_PROB"] >= MIN_PROB) &
    (df["EV_SAFE_1U"] >= MIN_EV)
].copy()

board = board.sort_values("SAFE_SCORE", ascending=False).copy()
board["RANK_ALL"] = range(1, len(board)+1)

# 2) Snapshot the top 7 BEFORE removing anyone (so we can pull old ranks 6/7)
top7_before = board.head(7).copy()
print("Top 7 BEFORE removals:")
display(top7_before[["RANK_ALL","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SAFE_PROB","EV_SAFE_1U","SAFE_SCORE"]])

# 3) Remove Edwards + Allen (robust contains match)
remove_mask = (
    top7_before["PLAYER"].str.contains("Anthony Edwards", case=False, na=False) |
    top7_before["PLAYER"].str.contains("Jarrett Allen", case=False, na=False)
)

# Old ranks 6 and 7 AFTER removing the two names:
# (i.e., take the top7 list and drop those names, then take the next two at the bottom)
remaining = top7_before.loc[~remove_mask].copy()

# If Edwards/Allen weren’t in the top7_before for some reason, fall back to removing from full board
if len(remaining) == len(top7_before):
    board_removed = board[
        ~(
            board["PLAYER"].str.contains("Anthony Edwards", case=False, na=False) |
            board["PLAYER"].str.contains("Jarrett Allen", case=False, na=False)
        )
    ].copy()
    # take top 5 from this cleaned board
    final = board_removed.head(5).copy()
else:
    # remaining currently contains (original top7 minus removed players) => should be 5 rows
    final = remaining.copy()
    final = final.head(5).copy()

# 4) Re-rank final list 1–5 and compute units (same half-Kelly)
final = final.copy()
final["RANK"] = range(1, len(final)+1)

HALF_KELLY = 0.50
MIN_U, MAX_U = 0.25, 1.00

def half_kelly_units(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f) * HALF_KELLY
    u = min(max(f, 0.0), MAX_U)
    if 0 < u < MIN_U:
        u = MIN_U
    return float(round(u, 2))

final["UNITS"] = [half_kelly_units(p,d) for p,d in zip(final["SAFE_PROB"], final["DEC_ODDS"])]

print("\nFINAL CARD (Edwards + Allen removed):")
display(final[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

# 5) Discord printout
lines=[]
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for r in final.itertuples(index=False):
    odds = f"({int(r.AMERICAN_ODDS)})"
    lines.append(
        f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n"
        f"{r.PROP} {r.OUTCOME} {odds} — {r.UNITS:.2f}u"
    )
print("\n".join(lines))

# 6) Also explicitly show the two replacements (old ranks 6 & 7)
# These are the bottom 2 rows of the remaining top7_before after removals
replacements = remaining.tail(2).copy() if len(remaining) >= 5 else final.tail(2).copy()
print("\nREPLACEMENTS (the two you’re adding):")
display(replacements[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SAFE_PROB","EV_SAFE_1U","SAFE_SCORE"]])

Top 7 BEFORE removals:


,RANK_ALL,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,SAFE_PROB,EV_SAFE_1U,SAFE_SCORE
0,1,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,-110,0.873452,0.667500,52.684133
2,2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,0.889146,0.502350,46.449934
8,3,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.724559,0.441945,32.394688
21,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.716886,0.334891,23.563812
4,5,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.742823,0.332364,17.957520
7,6,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.730011,0.244103,17.933711
9,7,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.721795,0.349443,16.551355



FINAL CARD (Edwards + Allen removed):


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
8,1,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.25
21,2,Day'Ron Sharpe,CLE at BKN,Rebounds,u9.5,-116,0.25
4,3,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.25
7,4,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.25
9,5,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.25


NBA ELITE PROP CARD
—
1) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.25u
2) Day'Ron Sharpe — CLE at BKN
Rebounds u9.5 (-116) — 0.25u
3) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.25u
4) Scotty Pippen Jr. — MEM at IND
Rebounds u3.5 (-142) — 0.25u
5) Walter Clayton Jr. — MEM at IND
Points + Rebounds u14.5 (-115) — 0.25u

REPLACEMENTS (the two you’re adding):


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,SAFE_PROB,EV_SAFE_1U,SAFE_SCORE
7,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.730011,0.244103,17.933711
9,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.721795,0.349443,16.551355


In [ ]:
# ==============================
# STRICT NBA INPUT GATES (NCAA-symmetrical)
# - Fix odds parsing
# - Parse OUTCOME safely
# - Drop corrupted 0.5 overs in non-count markets
# - Drop insane line ranges by market
# ==============================

import numpy as np, pandas as pd, re

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    # numeric junk like -11012 -> take sign + first 3 digits
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

# sanitize odds
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

# parse outcome
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))
df["MKT"] = df["PROP"].map(market_key)

# Gate 1: drop OVER 0.5 for non-count markets
ALLOW_HALF_ANY = set(["three","steals","blocks"])
drop_over_half = (df["LINE"]==0.5) & (df["SIDE"]=="over") & (~df["MKT"].isin(ALLOW_HALF_ANY))
print("Dropping OVER 0.5 non-count:", int(drop_over_half.sum()))
df = df.loc[~drop_over_half].copy()

# Gate 2: line sanity by market (prevents Edwards PRA o14.5, Allen o2.5 junk, etc.)
RANGES = {
    "points":   (5.0, 45.0),
    "rebounds": (2.0, 20.0),
    "assists":  (1.0, 15.0),
    "three":    (0.5, 6.5),
    "blocks":   (0.5, 4.5),
    "steals":   (0.5, 4.5),
    "pra":      (20.0, 70.0),
    "pr":       (10.0, 55.0),
    "pa":       (10.0, 55.0),
    "ra":       (8.0, 35.0),
    "other":    (1.0, 80.0),
}
lo = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[0])
hi = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[1])
bad_line = (df["LINE"].isna()) | (df["LINE"] < lo) | (df["LINE"] > hi)

print("Dropping insane/missing lines:", int(bad_line.sum()))
display(df.loc[bad_line, ["PLAYER","GAME","PROP","OUTCOME","LINE","MKT","AMERICAN_ODDS"]].head(25))

df = df.loc[~bad_line].copy()

# rebuild book probs
def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("STRICT gates passed. Rows:", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MKT","LINE"]].head(10))

Dropping OVER 0.5 non-count: 0
Dropping insane/missing lines: 13


,PLAYER,GAME,PROP,OUTCOME,LINE,MKT,AMERICAN_ODDS
0,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o14.5,14.5,pra,-110
3,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,16.5,pra,-129
5,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,19.5,pra,-125
31,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u18.5,18.5,pra,-104
37,Rob Dillingham,MIL at CHI,Rebounds + Assists,u7.5,7.5,ra,-150
44,Kobe Brown,MEM at IND,Points + Rebounds + Assists,u19.5,19.5,pra,-105
47,Guerschon Yabusele,MIL at CHI,Rebounds + Assists,u7.5,7.5,ra,-109
48,Nolan Traore,CLE at BKN,Points + Rebounds + Assists,u19.5,19.5,pra,-125
52,DeMar DeRozan,SAC at LAL,Rebounds + Assists,o5.5,5.5,ra,-125
57,Nolan Traore,CLE at BKN,Points + Rebounds + Assists,u18.5,18.5,pra,-117


STRICT gates passed. Rows: 71


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MKT,LINE
2,Jarrett Allen,CLE at BKN,Rebounds,o2.5,-145,rebounds,2.5
4,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,pa,13.5
6,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,points,11.5
7,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,rebounds,3.5
8,Rob Dillingham,MIL at CHI,Points,u8.5,-101,points,8.5
9,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,pr,14.5
10,Walter Clayton Jr.,MEM at IND,Points + Assists,u15.5,-105,pa,15.5
11,Jarace Walker,MEM at IND,Points,u18.5,-110,points,18.5
12,Kobe Brown,MEM at IND,Rebounds,u7.5,-145,rebounds,7.5
13,Walter Clayton Jr.,MEM at IND,Three Pointers Made,u1.5,-142,three,1.5


In [ ]:
# ==============================
# ESPN ENRICHMENT (NBA) — DEF PPG ALLOWED + PACE PROXY
# - Pull team list + stats from ESPN
# - Cache results so it doesn't hammer endpoints
# ==============================

import requests, json, time
import pandas as pd
import numpy as np
import re

ESPN_TEAM_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_STATS_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/statistics"

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})

def get_json(url, tries=4, timeout=25):
    for i in range(tries):
        try:
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if i == tries-1:
                raise
            time.sleep(1.5*(i+1))
    return None

# Parse "AAA at BBB"
def parse_game_tokens(g):
    s = str(g).strip().upper()
    m = re.match(r"^\s*([A-Z]{2,4})\s+(?:AT|@)\s+([A-Z]{2,4})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None, None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

# Team directory (abbr -> ESPN team id)
teams_js = get_json(ESPN_TEAM_URL)
teams = []
for item in teams_js.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
    t = item.get("team",{})
    teams.append({
        "abbr": t.get("abbreviation"),
        "team_id": t.get("id"),
        "name": t.get("displayName")
    })
team_dir = pd.DataFrame(teams).dropna()
abbr_to_id = dict(zip(team_dir["abbr"], team_dir["team_id"]))

df["HOME_ESPN_ID"] = df["HOME_ABBR"].map(abbr_to_id)
df["AWAY_ESPN_ID"] = df["AWAY_ABBR"].map(abbr_to_id)

need_ids = pd.unique(pd.concat([df["HOME_ESPN_ID"], df["AWAY_ESPN_ID"]]).dropna()).tolist()

# Cache stats
stats_cache = {}
def fetch_team_stats(team_id):
    if team_id in stats_cache:
        return stats_cache[team_id]
    js = get_json(ESPN_STATS_URL.format(team_id=team_id))
    # Find defensive PPG allowed and a pace-like proxy if available
    def_ppg = np.nan
    pace = np.nan

    # ESPN returns categories with stats; we search textually
    cats = js.get("categories", [])
    for c in cats:
        for st in c.get("stats", []):
            name = (st.get("name","") or "").lower()
            disp = (st.get("displayName","") or "").lower()
            val = st.get("value", None)

            # defensive points allowed per game (common fields)
            if "points allowed" in disp or "opp points" in disp or "opponent points" in disp:
                try: def_ppg = float(val)
                except: pass

            # pace / possessions proxy sometimes appears as "pace" or "possessions per game"
            if "pace" in disp or "possessions" in disp:
                try: pace = float(val)
                except: pass

    stats_cache[team_id] = {"DEF_PPG_ALLOWED": def_ppg, "PACE_PROXY": pace}
    return stats_cache[team_id]

rows=[]
for tid in need_ids:
    s = fetch_team_stats(tid)
    rows.append({"team_id": tid, **s})

ctx = pd.DataFrame(rows)

# Merge: opponent context (we don’t know player team reliably in this CSV)
# Use game-level proxies: avg of both teams
df = df.merge(ctx.add_prefix("HOME_"), left_on="HOME_ESPN_ID", right_on="HOME_team_id", how="left")
df = df.merge(ctx.add_prefix("AWAY_"), left_on="AWAY_ESPN_ID", right_on="AWAY_team_id", how="left")

df["ESPN_DEF_GAME"]  = (df["HOME_DEF_PPG_ALLOWED"] + df["AWAY_DEF_PPG_ALLOWED"]) / 2.0
df["ESPN_PACE_GAME"] = (df["HOME_PACE_PROXY"] + df["AWAY_PACE_PROXY"]) / 2.0

# If pace missing from ESPN (common), fallback to a stable median so we still run
df["ESPN_PACE_GAME"] = df["ESPN_PACE_GAME"].fillna(df["ESPN_PACE_GAME"].median())

print("ESPN enrichment merged.")
print("DEF null:", df["ESPN_DEF_GAME"].isna().sum(), "/", len(df))
print("PACE null:", df["ESPN_PACE_GAME"].isna().sum(), "/", len(df))
display(df[["GAME","ESPN_DEF_GAME","ESPN_PACE_GAME"]].drop_duplicates().head(10))

ESPN enrichment merged.
DEF null: 71 / 71
PACE null: 71 / 71


,GAME,ESPN_DEF_GAME,ESPN_PACE_GAME
0,CLE at BKN,NaN,NaN
1,MIL at CHI,NaN,NaN
2,MEM at IND,NaN,NaN
12,POR at ATL,NaN,NaN
17,PHI at BOS,NaN,NaN
24,DET at ORL,NaN,NaN
31,SAC at LAL,NaN,NaN
39,OKC at DAL,NaN,NaN
52,NOP at LAC,NaN,NaN


In [ ]:
# Replace proxy context with ESPN context (symmetrical to NCAA)
df["OPP_DEF_USED"] = df["ESPN_DEF_GAME"]
df["PACE_USED"]    = df["ESPN_PACE_GAME"]

BASE_DEF  = float(df["OPP_DEF_USED"].median())
BASE_PACE = float(df["PACE_USED"].median())

# then use OPP_DEF_USED and PACE_USED in your shift calculations
# d_def  = df["OPP_DEF_USED"] - BASE_DEF
# d_pace = df["PACE_USED"] - BASE_PACE

In [ ]:
import pandas as pd, numpy as np, re, os

CSV_PATH = "NBA 3:1 props.csv"  # your file in cwd

# detect header row (same method you used)
raw = pd.read_csv(CSV_PATH, header=None)
best_i, best_score = 0, -1
required = {"PLAYER","GAME","PROP","OUTCOME","ODDS"}
for i in range(min(10, len(raw))):
    row = set([str(x).strip().upper() for x in raw.iloc[i].tolist() if pd.notna(x)])
    score = sum([1 for r in required if r in row])
    if score > best_score:
        best_score = score
        best_i = i

hdr = raw.iloc[best_i].tolist()
df = raw.iloc[best_i+1:].copy()
df.columns = [str(c).strip() for c in hdr]
df = df.dropna(how="all")

# normalize expected names
rename_map = {
    "ODDS": "AMERICAN_ODDS",
    "IM PROB %": "IM_PROB_%",
    "IM PROB%": "IM_PROB_%",
}
df = df.rename(columns=rename_map)

# keep only known columns if present
keep = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","IM_PROB_%","AVG","L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
df = df[[c for c in keep if c in df.columns]].copy()

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded df:", df.shape)
display(df.head(8))

Loaded df: (100, 16)


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,IM_PROB_%,AVG,L5,L10,2025,HM/AW,H2H,SIDE,LINE,BOOK_PROB,DEC_ODDS
2,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,52.4%,-1102,100%,100%,100%,100%,100%,over,0.5,0.523810,1.909091
3,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,48.8%,1052,100%,100%,100%,100%,100%,over,0.5,0.487805,2.050000
4,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,59.2%,-1452,100%,100%,100%,100%,100%,over,0.5,0.591837,1.689655
5,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,-129,56.3%,-1304,100%,90%,97.7%,100%,100%,under,16.5,0.563319,1.775194
6,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,55.8%,-1282,100%,90%,95.3%,100%,100%,under,13.5,0.557522,1.793651
7,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,-125,55.6%,-1254,100%,100%,90.2%,88%,100%,under,19.5,0.555556,1.800000
8,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,56.5%,-1448,100%,100%,86.3%,84%,100%,under,11.5,0.565217,1.769231
9,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,58.7%,-1839,100%,70%,100%,100%,100%,under,3.5,0.586777,1.704225


In [ ]:
import numpy as np, pandas as pd

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

# Convert all hit-rate columns to probabilities if present
hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

# Weighted blend (mirror NCAA style: recent + season + situational + DVP)
weights = {
    "L5":0.20, "L10":0.25, "2025":0.20, "HM/AW":0.10, "H2H":0.05,
    "DVP_L5":0.08, "DVP_L10":0.07, "DVP_2025":0.04, "DVP_HM/AW":0.01
}

num = 0.0
den = 0.0
for c,w in weights.items():
    if c in df.columns:
        v = df[c].astype(float)
        num += w * v.fillna(np.nan)
        den += w * v.notna().astype(float)

df["HIT_PROB"] = (num / den).clip(0.01, 0.99)

print("HIT_PROB null:", df["HIT_PROB"].isna().sum(), "/", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","HIT_PROB"]].head(10))

HIT_PROB null: 2 / 100


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIT_PROB
2,Anthony Edwards,MIN at DEN,Points + Rebounds + Assists,o0.5,-110,0.99000
3,Michael Porter Jr.,CLE at BKN,Rebounds + Assists,o0.5,105,0.99000
4,Jarrett Allen,CLE at BKN,Rebounds,o0.5,-145,0.99000
5,Rob Dillingham,MIL at CHI,Points + Rebounds + Assists,u16.5,-129,0.96300
6,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.95700
7,Walter Clayton Jr.,MEM at IND,Points + Rebounds + Assists,u19.5,-125,0.96050
8,Walter Clayton Jr.,MEM at IND,Points,u11.5,-130,0.94575
9,Scotty Pippen Jr.,MEM at IND,Rebounds,u3.5,-142,0.90625
10,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.91475
11,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.94075


In [ ]:
import requests, time, re
import pandas as pd, numpy as np

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})

def get_json(url, tries=4, timeout=25):
    for i in range(tries):
        try:
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception:
            if i == tries-1:
                raise
            time.sleep(1.5*(i+1))

ESPN_TEAM_URL  = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_STATS_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/statistics"

def parse_game_tokens(g):
    s = str(g).strip().upper()
    m = re.match(r"^\s*([A-Z]{2,4})\s+(?:AT|@)\s+([A-Z]{2,4})\s*$", s)
    return (m.group(1), m.group(2)) if m else (None, None)

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game_tokens))

teams_js = get_json(ESPN_TEAM_URL)
teams = []
for item in teams_js.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
    t = item.get("team",{})
    teams.append({"abbr": t.get("abbreviation"), "team_id": t.get("id")})
team_dir = pd.DataFrame(teams).dropna()
abbr_to_id = dict(zip(team_dir["abbr"], team_dir["team_id"]))

df["HOME_ESPN_ID"] = df["HOME_ABBR"].map(abbr_to_id)
df["AWAY_ESPN_ID"] = df["AWAY_ABBR"].map(abbr_to_id)

need_ids = pd.unique(pd.concat([df["HOME_ESPN_ID"], df["AWAY_ESPN_ID"]]).dropna()).tolist()

stats_cache = {}
def fetch_team_stats(team_id):
    if team_id in stats_cache:
        return stats_cache[team_id]
    js = get_json(ESPN_STATS_URL.format(team_id=team_id))
    def_ppg = np.nan
    pace = np.nan
    cats = js.get("categories", [])
    for c in cats:
        for st in c.get("stats", []):
            disp = (st.get("displayName","") or "").lower()
            val  = st.get("value", None)
            if ("points allowed" in disp) or ("opp points" in disp) or ("opponent points" in disp):
                try: def_ppg = float(val)
                except: pass
            if ("pace" in disp) or ("possessions" in disp):
                try: pace = float(val)
                except: pass
    stats_cache[team_id] = {"DEF_PPG_ALLOWED": def_ppg, "PACE_PROXY": pace}
    return stats_cache[team_id]

rows=[]
for tid in need_ids:
    s = fetch_team_stats(tid)
    rows.append({"team_id": tid, **s})
ctx = pd.DataFrame(rows)

df = df.merge(ctx.add_prefix("HOME_"), left_on="HOME_ESPN_ID", right_on="HOME_team_id", how="left")
df = df.merge(ctx.add_prefix("AWAY_"), left_on="AWAY_ESPN_ID", right_on="AWAY_team_id", how="left")

df["ESPN_DEF_GAME"]  = (df["HOME_DEF_PPG_ALLOWED"] + df["AWAY_DEF_PPG_ALLOWED"]) / 2.0
df["ESPN_PACE_GAME"] = (df["HOME_PACE_PROXY"] + df["AWAY_PACE_PROXY"]) / 2.0

# pace often missing -> fallback to median
df["ESPN_PACE_GAME"] = df["ESPN_PACE_GAME"].fillna(df["ESPN_PACE_GAME"].median())

print("ESPN DEF null:", df["ESPN_DEF_GAME"].isna().sum(), "/", len(df))
print("ESPN PACE null:", df["ESPN_PACE_GAME"].isna().sum(), "/", len(df))
display(df[["GAME","ESPN_DEF_GAME","ESPN_PACE_GAME"]].drop_duplicates().head(10))

ESPN DEF null: 100 / 100
ESPN PACE null: 100 / 100


,GAME,ESPN_DEF_GAME,ESPN_PACE_GAME
0,MIN at DEN,NaN,NaN
1,CLE at BKN,NaN,NaN
3,MIL at CHI,NaN,NaN
5,MEM at IND,NaN,NaN
16,POR at ATL,NaN,NaN
22,PHI at BOS,NaN,NaN
32,DET at ORL,NaN,NaN
34,NOP at LAC,NaN,NaN
42,SAC at LAL,NaN,NaN
54,OKC at DAL,NaN,NaN


In [ ]:
import numpy as np, pandas as pd, re
from scipy.stats import norm, poisson, nbinom

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p or "3" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

# STRICT gate: drop insane lines by market (prevents PRA o14.5 ever)
RANGES = {
    "points": (5.0,45.0), "rebounds":(2.0,20.0), "assists":(1.0,15.0),
    "three":(0.5,6.5), "blocks":(0.5,4.5), "steals":(0.5,4.5),
    "pra":(20.0,70.0), "pr":(10.0,55.0), "pa":(10.0,55.0), "ra":(8.0,35.0),
    "other":(1.0,80.0)
}
lo = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[0])
hi = df["MKT"].map(lambda k: RANGES.get(k, RANGES["other"])[1])
df = df[(df["LINE"].notna()) & (df["LINE"]>=lo) & (df["LINE"]<=hi)].copy()

# Distributions
def dist_for_market(mkey):
    if mkey in ["three","steals","blocks"]:
        return "poisson"
    if mkey in ["assists","rebounds"]:
        return "nbinom"
    return "normal"

def nb_params_from_mu_k(mu, k):
    mu = max(0.05, float(mu)); k = max(0.5, float(k))
    p = k/(k+mu); n = k
    return n,p

def poisson_prob_over(line, lam):
    cut = int(np.floor(line))
    return float(1 - poisson.cdf(cut, mu=lam))

def poisson_lam_from_pover(line, p_over):
    p_over = float(np.clip(p_over, 0.02, 0.98))
    lo, hi = 0.05, max(12.0, line+12.0)
    for _ in range(60):
        mid=(lo+hi)/2
        p=poisson_prob_over(line, mid)
        if p>p_over: hi=mid
        else: lo=mid
    return float((lo+hi)/2)

def nb_k_for_market(mkey): return 4.0 if mkey=="assists" else 5.0

def nb_prob_over(line, mu, k):
    n,p=nb_params_from_mu_k(mu,k)
    cut=int(np.floor(line))
    return float(1-nbinom.cdf(cut, n=n, p=p))

def nb_mu_from_pover(line, p_over, k):
    p_over=float(np.clip(p_over,0.02,0.98))
    lo,hi=0.05,max(18.0,line+18.0)
    for _ in range(60):
        mid=(lo+hi)/2
        p=nb_prob_over(line, mid, k)
        if p>p_over: hi=mid
        else: lo=mid
    return float((lo+hi)/2)

def normal_sigma(mkey, line):
    line=float(line)
    if mkey in ["pra","pr","pa","ra"]: return max(4.0, 0.17*line)
    if mkey=="points": return max(3.0, 0.18*line)
    return max(2.5, 0.18*max(line,1.0))

def normal_prob_over(line, mu, sigma):
    x=((line+0.5)-mu)/sigma
    return float(1-norm.cdf(x))

def normal_mu_from_pover(line, p_over, sigma):
    p_over=float(np.clip(p_over,0.02,0.98))
    z=norm.ppf(1-p_over)
    return float((line+0.5) - sigma*z)

def prob_from_mu(line, mu, dist_used, side, aux=np.nan, mkey="other"):
    if dist_used=="poisson":
        p_over=poisson_prob_over(line, mu)
    elif dist_used=="nbinom":
        k=float(aux) if pd.notna(aux) else nb_k_for_market(mkey)
        p_over=nb_prob_over(line, mu, k)
    else:
        sigma=float(aux) if pd.notna(aux) else normal_sigma(mkey, line)
        p_over=normal_prob_over(line, mu, sigma)
    return p_over if side=="over" else (1-p_over)

# invert HIT_PROB -> μ_base
dist_used=[]; aux=[]; mu_base=[]
for r in df.itertuples(index=False):
    dist=dist_for_market(r.MKT)
    dist_used.append(dist)
    p_over = r.HIT_PROB if r.SIDE=="over" else (1-r.HIT_PROB)
    if dist=="poisson":
        mu=poisson_lam_from_pover(r.LINE, p_over); mu_base.append(mu); aux.append(np.nan)
    elif dist=="nbinom":
        k=nb_k_for_market(r.MKT)
        mu=nb_mu_from_pover(r.LINE, p_over, k); mu_base.append(mu); aux.append(k)
    else:
        sig=normal_sigma(r.MKT, r.LINE)
        mu=normal_mu_from_pover(r.LINE, p_over, sig); mu_base.append(mu); aux.append(sig)

df["DIST_USED"]=dist_used
df["AUX_PARAM"]=aux
df["PROJ_MU_BASE"]=mu_base

# CTX-to-μ using ESPN DEF/PACE (like NCAA today)
df["OPP_DEF_USED"] = df["ESPN_DEF_GAME"]
df["PACE_USED"]    = df["ESPN_PACE_GAME"]
BASE_DEF  = float(df["OPP_DEF_USED"].median())
BASE_PACE = float(df["PACE_USED"].median())

BETA = {
    "points": (0.55, 0.35), "rebounds": (0.20, 0.15), "assists": (0.18, 0.14),
    "three": (0.10, 0.08), "steals": (0.06, 0.05), "blocks": (0.06, 0.05),
    "pra": (0.85, 0.55), "pr": (0.70, 0.45), "pa": (0.70, 0.45), "ra": (0.55, 0.35),
    "other": (0.35, 0.25),
}
MU_CAP = {"points":2.0,"rebounds":1.2,"assists":1.2,"three":0.8,"steals":0.5,"blocks":0.5,"pra":3.0,"pr":2.5,"pa":2.5,"ra":2.0,"other":1.5}

def clip(v, lo_, hi_):
    return float(max(lo_, min(hi_, v)))

mu_shift=[]; mu_ctx=[]
for r in df.itertuples(index=False):
    b_def,b_pace = BETA.get(r.MKT, BETA["other"])
    cap = MU_CAP.get(r.MKT, MU_CAP["other"])
    d_def  = (float(r.OPP_DEF_USED) - BASE_DEF) / 8.0
    d_pace = (float(r.PACE_USED) - BASE_PACE) / 2.5
    shift = (b_def*d_def) + (b_pace*d_pace)
    shift = clip(shift, -cap, cap)
    mu_shift.append(shift)
    mu_ctx.append(float(r.PROJ_MU_BASE) + shift)

df["PROJ_MU_SHIFT"]=mu_shift
df["PROJ_MU_CTX"]=mu_ctx

df["PROJ_PROB_CTX"] = [
    np.clip(prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT), 0.01, 0.99)
    for r in df.itertuples(index=False)
]

# Blend (same as NCAA today)
W_HIT, W_PROJ = 0.70, 0.30
df["BLEND_PROB"] = np.clip(W_HIT*df["HIT_PROB"] + W_PROJ*df["PROJ_PROB_CTX"], 0.01, 0.99)

# Mode A shrink toward book
SHRINK_TO_BOOK = 0.25
df["SAFE_PROB"] = np.clip((1-SHRINK_TO_BOOK)*df["BLEND_PROB"] + SHRINK_TO_BOOK*df["BOOK_PROB"], 0.01, 0.99)
df["EV_SAFE_1U"] = df["SAFE_PROB"]*(df["DEC_ODDS"]-1) - (1-df["SAFE_PROB"])

# Injury sensitivity (scenario swing)
UP_MULT  = {"points":1.06,"assists":1.07,"rebounds":1.04,"three":1.05,"pra":1.06,"pr":1.05,"pa":1.05,"ra":1.05,"other":1.05,"steals":1.04,"blocks":1.04}
DOWN_MULT= {"points":0.92,"assists":0.90,"rebounds":0.93,"three":0.92,"pra":0.91,"pr":0.92,"pa":0.92,"ra":0.92,"other":0.92,"steals":0.94,"blocks":0.94}

def apply_mu(mu, mkt, mode):
    if mode=="up": return float(mu)*UP_MULT.get(mkt,1.05)
    return float(mu)*DOWN_MULT.get(mkt,0.92)

p_up=[]; p_dn=[]
for r in df.itertuples(index=False):
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX, r.MKT, "up"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX, r.MKT, "down"), r.DIST_USED, r.SIDE, r.AUX_PARAM, r.MKT))

delta = np.clip(np.array(p_up) - np.array(p_dn), -1, 1)
df["INJ_SENS_ABS"] = np.abs(delta) * 100
df["MU_SHIFT_ABS"] = df["PROJ_MU_SHIFT"].abs()

df["SAFE_SCORE"] = (df["EV_SAFE_1U"]*100) - (df["INJ_SENS_ABS"]*1.25) - (df["MU_SHIFT_ABS"]*6.0)

# Elite selection + half Kelly
MIN_PROB=0.60; MIN_EV=0.08; MAX_PLAYS=5
elite = df[(df["SAFE_PROB"]>=MIN_PROB) & (df["EV_SAFE_1U"]>=MIN_EV)].copy()
elite = elite.sort_values("SAFE_SCORE", ascending=False).head(MAX_PLAYS).copy()
elite["RANK"]=range(1,len(elite)+1)

HALF_KELLY=0.50; MIN_U=0.25; MAX_U=1.00
def half_kelly_units(p, dec):
    b=dec-1; q=1-p
    f=(b*p-q)/b
    f=max(0.0,f)*HALF_KELLY
    u=min(max(f,0.0),MAX_U)
    if 0<u<MIN_U: u=MIN_U
    return float(round(u,2))

elite["UNITS"]=[half_kelly_units(p,d) for p,d in zip(elite["SAFE_PROB"], elite["DEC_ODDS"])]

display(elite[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

lines=["NBA ELITE PROP CARD","—"]
for r in elite.itertuples(index=False):
    lines.append(f"{int(r.RANK)}) {r.PLAYER} — {r.GAME}\n{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)}) — {r.UNITS:.2f}u")
print("\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
8,1,Rob Dillingham,MIL at CHI,Points,u8.5,-101,0.28
4,2,Rob Dillingham,MIL at CHI,Points + Assists,u13.5,-126,0.31
9,3,Walter Clayton Jr.,MEM at IND,Points + Rebounds,u14.5,-115,0.30
60,4,Day'Ron Sharpe,CLE at BKN,Rebounds,u8.5,113,0.25
38,5,Kobe Brown,MEM at IND,Assists,u1.5,130,0.25


NBA ELITE PROP CARD
—
1) Rob Dillingham — MIL at CHI
Points u8.5 (-101) — 0.28u
2) Rob Dillingham — MIL at CHI
Points + Assists u13.5 (-126) — 0.31u
3) Walter Clayton Jr. — MEM at IND
Points + Rebounds u14.5 (-115) — 0.30u
4) Day'Ron Sharpe — CLE at BKN
Rebounds u8.5 (113) — 0.25u
5) Kobe Brown — MEM at IND
Assists u1.5 (130) — 0.25u


In [ ]:
import pandas as pd, numpy as np, re

CSV_PATH = "/mnt/data/NBA prop 3:2.csv"

# Detect header row (robust for messy exports)
raw = pd.read_csv(CSV_PATH, header=None)
best_i, best_score = 0, -1
required = {"PLAYER","GAME","PROP","OUTCOME","ODDS"}
for i in range(min(15, len(raw))):
    row = set([str(x).strip().upper() for x in raw.iloc[i].tolist() if pd.notna(x)])
    score = sum([1 for r in required if r in row])
    if score > best_score:
        best_score = score
        best_i = i

hdr = raw.iloc[best_i].tolist()
df = raw.iloc[best_i+1:].copy()
df.columns = [str(c).strip() for c in hdr]
df = df.dropna(how="all").copy()

# Normalize expected names
rename_map = {
    "ODDS": "AMERICAN_ODDS",
    "IM PROB %": "IM_PROB_%",
    "IM PROB%": "IM_PROB_%",
}
df = df.rename(columns=rename_map)

# Keep only known columns if present
keep = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","IM_PROB_%","AVG","L5","L10","2025","HM/AW","H2H",
        "DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
df = df[[c for c in keep if c in df.columns]].copy()

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].apply(extract_american_odds)
df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded:", CSV_PATH)
print("df shape:", df.shape)
print("columns:", list(df.columns))
display(df.head(8))

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/NBA prop 3:2.csv'

CSV candidates found: []


FileNotFoundError: No CSV found. Upload NBA prop 3:2.csv to this Colab runtime (or mount Drive).

In [ ]:
from google.colab import files
uploaded = files.upload()

# Grab uploaded filename automatically
import os
CSV_PATH = list(uploaded.keys())[0]

print("Uploaded file:", CSV_PATH)


Saving NBA prop 3:2.csv to NBA prop 3:2 (1).csv
Uploaded file: NBA prop 3:2 (1).csv


In [ ]:
import pandas as pd, numpy as np, re, os

print("Using file:", CSV_PATH)

raw = pd.read_csv(CSV_PATH, header=None)

# -------- STEP 1: find real header row --------
header_index = None

for i in range(min(40, len(raw))):
    row_vals = [str(x).strip().upper() for x in raw.iloc[i].tolist()]
    if "PLAYER" in row_vals and "GAME" in row_vals and "PROP" in row_vals:
        header_index = i
        break

if header_index is None:
    raise Exception("Could not find header row containing PLAYER/GAME/PROP")

print("Detected header row index:", header_index)

# -------- STEP 2: rebuild df using real header --------
headers = [str(x).strip() for x in raw.iloc[header_index].tolist()]
df = raw.iloc[header_index+1:].copy()
df.columns = headers
df = df.dropna(how="all").copy()

print("Columns detected:", df.columns.tolist())

# -------- STEP 3: detect odds column --------
odds_col = None
for c in df.columns:
    sample = df[c].astype(str).head(30)
    hits = sample.str.contains(r"[+-]\d{2,4}", regex=True).sum()
    if hits > 5:  # strong signal of odds column
        odds_col = c
        break

if odds_col is None:
    raise Exception("Could not auto-detect odds column")

print("Detected odds column:", odds_col)

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    return int(m.group(1)) if m else np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df[odds_col].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

def american_to_prob(o):
    return (-o)/((-o)+100) if o < 0 else 100/(o+100)

def american_to_decimal(o):
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"] = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Rows loaded:", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","LINE","SIDE"]].head(10))

Using file: NBA prop 3:2 (1).csv


Exception: Could not find header row containing PLAYER/GAME/PROP

In [ ]:
import pandas as pd, numpy as np, re, os

print("Using file:", CSV_PATH)

raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)

# --- Header row detector: score each row by "header-likeness"
TOKENS = [
    "PLAYER","NAME","GAME","MATCHUP","PROP","MARKET","OUTCOME","O/U","LINE",
    "ODDS","PRICE","BET","BOOK","IM PROB","PROB","L5","L10","H2H","DVP","2025","HM","AW"
]

def row_score(row):
    vals = [str(x).strip().upper() for x in row]
    # remove empties
    vals = [v for v in vals if v not in ["", "NAN", "NONE"]]
    if not vals:
        return -999
    joined = " | ".join(vals)
    score = 0
    for t in TOKENS:
        if t in joined:
            score += 1
    # bonus if row has lots of distinct text labels (typical headers)
    alpha_cells = sum(1 for v in vals if re.search(r"[A-Z]", v))
    score += min(alpha_cells, 10) * 0.2
    return score

scores = []
for i in range(min(80, len(raw))):
    scores.append((i, row_score(raw.iloc[i].tolist())))

header_index, best = max(scores, key=lambda x: x[1])
print("Best header row index:", header_index, "score:", best)

headers = [str(x).strip() for x in raw.iloc[header_index].tolist()]
# If there are blanks, fill them with placeholders so pandas doesn't merge columns into DataFrames
headers = [h if h not in ["", "nan", "NaN", "None", None] else f"COL_{j}" for j,h in enumerate(headers)]

df = raw.iloc[header_index+1:].copy()
df.columns = headers
df = df.replace({"": np.nan}).dropna(how="all").copy()

print("Detected header sample:", headers[:20])
print("df shape:", df.shape)

# --- Auto-detect ODDS column by counting +105/-110 patterns
def odds_hits(series, n=60):
    s = series.astype(str).head(min(n, len(series)))
    return s.str.contains(r"([+-]\d{2,4})", regex=True).sum()

best_col, best_hits = None, -1
for c in df.columns:
    h = odds_hits(df[c])
    if h > best_hits:
        best_hits = h
        best_col = c

if best_hits < 3:
    # fallback: look for a column name with odds-like words
    name_hits = [c for c in df.columns if any(k in str(c).upper() for k in ["ODDS","PRICE","BET","LINE"])]
    best_col = name_hits[0] if name_hits else best_col

if best_col is None:
    raise Exception("Could not detect odds column in this CSV export.")

ODDS_COL = best_col
print("Detected ODDS_COL =", ODDS_COL, "pattern hits:", best_hits)

# --- Parse odds + outcome line
def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})$", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

# Find the OUTCOME column (it may not be literally called OUTCOME)
outcome_candidates = [c for c in df.columns if str(c).strip().upper() in ["OUTCOME","O/U","OU","TOTAL","SIDE"]]
if not outcome_candidates:
    # heuristic: pick a column where values look like o12.5 / u8.5
    best_o_col, best_o_hits = None, -1
    for c in df.columns:
        s = df[c].astype(str).head(80)
        hits = s.str.contains(r"^(o|u)\s*\d+(\.\d+)?$", case=False, regex=True).sum()
        if hits > best_o_hits:
            best_o_hits = hits
            best_o_col = c
    outcome_candidates = [best_o_col] if best_o_col is not None else []

if not outcome_candidates or outcome_candidates[0] is None:
    raise Exception("Could not detect OUTCOME column. This export is missing o/u lines.")

OUTCOME_COL = outcome_candidates[0]
print("Detected OUTCOME_COL =", OUTCOME_COL)

# Find PLAYER/GAME/PROP columns (again: not always exact names)
def pick_col(name_keywords):
    candidates = [c for c in df.columns if any(k in str(c).upper() for k in name_keywords)]
    return candidates[0] if candidates else None

PLAYER_COL = pick_col(["PLAYER","NAME"])
GAME_COL   = pick_col(["GAME","MATCHUP"])
PROP_COL   = pick_col(["PROP","MARKET"])

if PLAYER_COL is None or GAME_COL is None or PROP_COL is None:
    print("WARNING: Could not perfectly detect PLAYER/GAME/PROP column names.")
    print("Detected:", {"PLAYER_COL":PLAYER_COL,"GAME_COL":GAME_COL,"PROP_COL":PROP_COL})

# Normalize to expected names
if PLAYER_COL: df = df.rename(columns={PLAYER_COL:"PLAYER"})
if GAME_COL:   df = df.rename(columns={GAME_COL:"GAME"})
if PROP_COL:   df = df.rename(columns={PROP_COL:"PROP"})
df = df.rename(columns={OUTCOME_COL:"OUTCOME"})

df["AMERICAN_ODDS"] = df[ODDS_COL].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

df = df[df["AMERICAN_ODDS"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Final df shape:", df.shape)
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","LINE","SIDE"]].head(12))

Using file: NBA prop 3:2 (1).csv
Best header row index: 7 score: 3.0
Detected header sample: ['COL_0', 'Julian Strawther', 'DEN at UTA', 'Rebounds', 'u3.5', 'BET\n-137', '-1455', '57.8%', '80%', '60%', '78.4%', '70.6%', '100%', '0%', '20%', '33.3%', '32.3%']
df shape: (93, 17)
Detected ODDS_COL = BET
-137 pattern hits: 60
Detected OUTCOME_COL = u3.5
Detected: {'PLAYER_COL': None, 'GAME_COL': None, 'PROP_COL': None}
Final df shape: (93, 22)


/tmp/ipython-input-1748/2232817705.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({"": np.nan}).dropna(how="all").copy()
/tmp/ipython-input-1748/2232817705.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return s.str.contains(r"([+-]\d{2,4})", regex=True).sum()
/tmp/ipython-input-1748/2232817705.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return s.str.contains(r"([+-]\d{2,4})", regex=True).sum()
/tmp/ipython-input-1748/2232817705.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the group

KeyError: "['PLAYER', 'GAME', 'PROP'] not in index"

In [ ]:
import pandas as pd, numpy as np, re

CSV_PATH = "NBA prop 3:2.csv"   # or whatever your uploaded filename is

raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)

# Drop fully blank rows
raw = raw.replace("", np.nan).dropna(how="all").fillna("")

# Your file is "headerless" with a blank first column. Map by position.
# 0 blank, 1 player, 2 game, 3 prop, 4 outcome, 5 odds, 6 junk/extra, 7 im prob, 8..16 hit columns
colmap = {
    1: "PLAYER",
    2: "GAME",
    3: "PROP",
    4: "OUTCOME",
    5: "ODDS_RAW",
    7: "IM_PROB_%",
    8: "L5",
    9: "L10",
    10: "2025",
    11: "HM/AW",
    12: "H2H",
    13: "DVP_L5",
    14: "DVP_L10",
    15: "DVP_2025",
    16: "DVP_HM/AW",
}

df = raw.rename(columns=colmap)[list(colmap.values())].copy()

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    if m: return int(m.group(1))
    s2 = re.sub(r"[^\d\-\+]", "", s)
    m2 = re.match(r"^([+-])(\d{2,4})$", s2)
    if m2: return int(m2.group(1) + m2.group(2))
    return np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# keep only valid rows
df = df[df["AMERICAN_ODDS"].notna() & df["LINE"].notna() & df["SIDE"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)

def american_to_prob(o):
    o = float(o)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o = float(o)
    return 1 + (100/abs(o)) if o < 0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

print("Loaded rows:", len(df))
print("Columns:", list(df.columns))
display(df.head(10)[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","IM_PROB_%","L5","L10","2025"]])

Loaded rows: 100
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS_RAW', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP_L5', 'DVP_L10', 'DVP_2025', 'DVP_HM/AW', 'AMERICAN_ODDS', 'SIDE', 'LINE', 'BOOK_PROB', 'DEC_ODDS']


/tmp/ipython-input-1748/1156004536.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw = raw.replace("", np.nan).dropna(how="all").fillna("")


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,IM_PROB_%,L5,L10,2025
1,Reed Sheppard,HOU at WAS,Rebounds + Assists,u8.5,-148,59.7%,80%,70%,83.1%
2,Cody Williams,DEN at UTA,Three Pointers Made,u0.5,-145,59.2%,100%,90%,71.7%
3,Reed Sheppard,HOU at WAS,Assists,u4.5,-163,62.0%,80%,80%,79.7%
4,Reed Sheppard,HOU at WAS,Rebounds + Assists,u7.5,111,47.4%,80%,70%,78%
5,Tari Eason,HOU at WAS,Three Pointers Made,u2.5,-185,64.9%,100%,90%,67.6%
6,Gui Santos,LAC at GSW,Rebounds,u5.5,-172,63.2%,60%,60%,81.6%
7,Julian Strawther,DEN at UTA,Rebounds,u3.5,-137,57.8%,80%,60%,78.4%
8,Nikola Jokić,DEN at UTA,Points + Assists,u40.5,-117,53.9%,80%,90%,63.6%
9,Tari Eason,HOU at WAS,Points,u14.5,-125,55.6%,100%,90%,70.3%
10,Tari Eason,HOU at WAS,Points + Assists,u16.5,-125,55.6%,100%,90%,70.3%


In [ ]:
import numpy as np
import pandas as pd

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

# SAME weights as the prior run
weights = {
    "L5":0.20, "L10":0.25, "2025":0.20, "HM/AW":0.10, "H2H":0.05,
    "DVP_L5":0.08, "DVP_L10":0.07, "DVP_2025":0.04, "DVP_HM/AW":0.01
}

num = 0.0
den = 0.0
for c,w in weights.items():
    v = df[c].astype(float)
    num += w * v.fillna(np.nan)
    den += w * v.notna().astype(float)

df["HIT_PROB"] = (num / den).clip(0.01, 0.99)

print("HIT_PROB null:", df["HIT_PROB"].isna().sum(), "/", len(df))
display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","HIT_PROB"]].head(12))

HIT_PROB null: 0 / 100


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIT_PROB
1,Reed Sheppard,HOU at WAS,Rebounds + Assists,u8.5,-148,0.76312
2,Cody Williams,DEN at UTA,Three Pointers Made,u0.5,-145,0.71570
3,Reed Sheppard,HOU at WAS,Assists,u4.5,-163,0.74311
4,Reed Sheppard,HOU at WAS,Rebounds + Assists,u7.5,111,0.68889
5,Tari Eason,HOU at WAS,Three Pointers Made,u2.5,-185,0.80924
6,Gui Santos,LAC at GSW,Rebounds,u5.5,-172,0.68571
7,Julian Strawther,DEN at UTA,Rebounds,u3.5,-137,0.61795
8,Nikola Jokić,DEN at UTA,Points + Assists,u40.5,-117,0.82068
9,Tari Eason,HOU at WAS,Points,u14.5,-125,0.77508
10,Tari Eason,HOU at WAS,Points + Assists,u16.5,-125,0.77107


In [ ]:
import numpy as np

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        v = float(s)
        return v/100.0 if v > 1 else v
    except:
        return np.nan

hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]

for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob)

weights = {
    "L5":0.20,
    "L10":0.25,
    "2025":0.20,
    "HM/AW":0.10,
    "H2H":0.05,
    "DVP_L5":0.08,
    "DVP_L10":0.07,
    "DVP_2025":0.04,
    "DVP_HM/AW":0.01
}

num = 0
den = 0
for c,w in weights.items():
    num += w * df[c]
    den += w * df[c].notna()

df["HIT_PROB"] = (num/den).clip(0.01,0.99)

display(df[["PLAYER","PROP","OUTCOME","HIT_PROB"]].head())

,PLAYER,PROP,OUTCOME,HIT_PROB
1,Reed Sheppard,Rebounds + Assists,u8.5,0.76312
2,Cody Williams,Three Pointers Made,u0.5,0.71570
3,Reed Sheppard,Assists,u4.5,0.74311
4,Reed Sheppard,Rebounds + Assists,u7.5,0.68889
5,Tari Eason,Three Pointers Made,u2.5,0.80924


In [ ]:
from scipy.stats import norm, poisson, nbinom

def market_key(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p: return "three"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

def pick_dist(mkt, line):
    if mkt in ["three"]:
        return "poisson"
    if mkt in ["assists","rebounds"] and line <= 8.5:
        return "nbinom"
    if mkt in ["pra","pr","pa","ra"] or line >= 12:
        return "normal"
    return "normal"

df["DIST_USED"] = df.apply(lambda r: pick_dist(r.MKT, r.LINE), axis=1)

def prob_from_mu(line, mu, dist, side):
    if dist=="normal":
        sd = max(1.0, mu*0.35)
        return (1-norm.cdf(line, mu, sd)) if side=="over" else norm.cdf(line, mu, sd)
    if dist=="poisson":
        k=int(line)
        return (1-poisson.cdf(k, mu)) if side=="over" else poisson.cdf(k, mu)
    if dist=="nbinom":
        r=6.0
        p=r/(r+mu)
        k=int(line)
        return (1-nbinom.cdf(k, r, p)) if side=="over" else nbinom.cdf(k, r, p)

def invert_mu(target_p, line, dist, side):
    lo, hi = 0.05, max(2.0, line*3 + 10)
    for _ in range(50):
        mid = (lo+hi)/2
        pmid = prob_from_mu(line, mid, dist, side)
        if pmid < target_p:
            lo = mid if side=="over" else lo
            hi = hi if side=="over" else mid
        else:
            hi = mid if side=="over" else hi
            lo = lo if side=="over" else mid
    return (lo+hi)/2

df["PROJ_MU_BASE"] = df.apply(
    lambda r: invert_mu(r.HIT_PROB, r.LINE, r.DIST_USED, r.SIDE),
    axis=1
)

display(df[["PLAYER","PROP","OUTCOME","DIST_USED","HIT_PROB","PROJ_MU_BASE"]].head())

,PLAYER,PROP,OUTCOME,DIST_USED,HIT_PROB,PROJ_MU_BASE
1,Reed Sheppard,Rebounds + Assists,u8.5,normal,0.76312,6.796025
2,Cody Williams,Three Pointers Made,u0.5,poisson,0.71570,0.334494
3,Reed Sheppard,Assists,u4.5,nbinom,0.74311,3.284988
4,Reed Sheppard,Rebounds + Assists,u7.5,normal,0.68889,6.396876
5,Tari Eason,Three Pointers Made,u2.5,poisson,0.80924,1.498433


In [ ]:
# No ESPN layer for structure A (same as 3/1)
# Context shift derived from hit vs book delta

df["PROJ_MU_SHIFT"] = (df["HIT_PROB"] - df["BOOK_PROB"]) * df["PROJ_MU_BASE"]
df["PROJ_MU_CTX"]   = (df["PROJ_MU_BASE"] + df["PROJ_MU_SHIFT"]).clip(lower=0.05)

df["PROJ_PROB_CTX"] = df.apply(
    lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE),
    axis=1
).clip(0.01,0.99)

df["BLEND_PROB"] = (0.6*df["PROJ_PROB_CTX"] + 0.4*df["HIT_PROB"]).clip(0.01,0.99)

In [ ]:
# Injury sensitivity proxy
def apply_mu(mu, direction):
    bump = mu*0.07
    return mu + bump if direction=="up" else max(0.05, mu - bump)

p_up=[]
p_dn=[]

for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))

df["INJ_SENS_ABS"] = np.abs(np.array(p_up)-np.array(p_dn))*100

df["SAFE_PROB"] = (
    df["BLEND_PROB"] -
    (df["INJ_SENS_ABS"]/100)*0.15
).clip(0.01,0.99)

df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1

def half_kelly(p, dec):
    b=dec-1
    k=(p*b-(1-p))/b
    return max(0,k/2)

df["UNITS"] = [half_kelly(p,d) for p,d in zip(df["SAFE_PROB"],df["DEC_ODDS"])]
df["UNITS"] = df["UNITS"].clip(0,0.5)

df_card = df[(df["EV_SAFE_1U"]>0) & (df["SAFE_PROB"]>df["BOOK_PROB"]+0.03)]
df_card = df_card.sort_values("EV_SAFE_1U",ascending=False)

display(df_card[["PLAYER","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]].head(10))

,PLAYER,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
14,Bilal Coulibaly,Points + Rebounds + Assists,o15.5,-106,0.330376,0.835317,0.623351
58,Bilal Coulibaly,Points + Rebounds,o13.5,-101,0.298320,0.799323,0.590732
82,Keyonte George,Points + Assists,o23.5,115,0.230181,0.711356,0.529416
55,Keyonte George,Points + Rebounds + Assists,o25.5,100,0.237964,0.737964,0.475927
18,Bobby Portis,Points,o9.5,-125,0.266879,0.792781,0.427006
60,Bilal Coulibaly,Rebounds + Assists,o5.5,-118,0.240201,0.761652,0.407121
65,Bobby Portis,Points + Rebounds,o15.5,100,0.197271,0.697271,0.394542
33,Bilal Coulibaly,Points,o9.5,-113,0.222123,0.739082,0.393137
95,Keyonte George,Points + Rebounds,o20.5,-105,0.198242,0.705601,0.377603
72,Bilal Coulibaly,Points + Assists,o11.5,-120,0.212004,0.738185,0.353340


In [ ]:
top = df_card.head(5).copy().reset_index(drop=True)
top["RANK"] = range(1,len(top)+1)

print("NBA ELITE PROP CARD")
print("—")

for r in top.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

NBA ELITE PROP CARD
—
1) Bilal Coulibaly — HOU at WAS
Points + Rebounds + Assists o15.5 (-106) — 0.33u
2) Bilal Coulibaly — HOU at WAS
Points + Rebounds o13.5 (-101) — 0.30u
3) Keyonte George — DEN at UTA
Points + Assists o23.5 (115) — 0.23u
4) Keyonte George — DEN at UTA
Points + Rebounds + Assists o25.5 (100) — 0.24u
5) Bobby Portis — BOS at MIL
Points o9.5 (-125) — 0.27u


In [ ]:
import numpy as np

# --- Choose which George play to DROP ---
DROP_PROP_CONTAINS = "Points + Rebounds + Assists"   # drops George PRA, keeps George PA
DROP_PLAYER = "Keyonte George"

# Optional: enforce max 1 prop per player on the final card
ENFORCE_ONE_PER_PLAYER = True

df_rank = df_card.copy()

# 1) Start from a larger pool so replacements are available
pool = df_rank.sort_values("EV_SAFE_1U", ascending=False).copy()

# 2) Build an initial top 7 (gives us room to swap cleanly)
initial = pool.head(7).copy()

# 3) Drop the selected George play
mask_drop = (initial["PLAYER"].astype(str) == DROP_PLAYER) & (initial["PROP"].astype(str).str.contains(DROP_PROP_CONTAINS, case=False, regex=False))
initial = initial[~mask_drop].copy()

# 4) If there are still 2 George plays, keep only the best EV one
george_rows = initial[initial["PLAYER"].astype(str) == DROP_PLAYER].copy()
if len(george_rows) > 1:
    keep_idx = george_rows.sort_values("EV_SAFE_1U", ascending=False).index[0]
    initial = initial.drop(index=[i for i in george_rows.index if i != keep_idx])

# 5) If enforcing one per player, dedupe by player keeping best EV per player
if ENFORCE_ONE_PER_PLAYER:
    initial = (initial.sort_values("EV_SAFE_1U", ascending=False)
                     .drop_duplicates(subset=["PLAYER"], keep="first"))

# 6) Fill back up to 5 with next best non-included plays from full pool
selected = initial.sort_values("EV_SAFE_1U", ascending=False).copy()
need = 5 - len(selected)

if need > 0:
    already_players = set(selected["PLAYER"].astype(str))
    already_keys = set(zip(selected["PLAYER"].astype(str), selected["GAME"].astype(str), selected["PROP"].astype(str), selected["OUTCOME"].astype(str)))

    add_candidates = []
    for r in pool.itertuples():
        key = (str(r.PLAYER), str(r.GAME), str(r.PROP), str(r.OUTCOME))
        if key in already_keys:
            continue
        if ENFORCE_ONE_PER_PLAYER and str(r.PLAYER) in already_players:
            continue
        # don't add a second George play
        if str(r.PLAYER) == DROP_PLAYER:
            continue
        add_candidates.append(r)
        if len(add_candidates) >= need:
            break

    if len(add_candidates) < need:
        print("WARNING: Not enough replacement candidates to fill to 5 under current constraints.")
    else:
        add_df = pd.DataFrame([a._asdict() for a in add_candidates])
        selected = pd.concat([selected, add_df], ignore_index=True)

# 7) Final card + Discord print
selected = selected.sort_values("EV_SAFE_1U", ascending=False).head(5).copy().reset_index(drop=True)
selected["RANK"] = np.arange(1, len(selected)+1)

display(selected[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]])

print("NBA ELITE PROP CARD")
print("—")
for r in selected.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
0,1,Bilal Coulibaly,HOU at WAS,Points + Rebounds + Assists,o15.5,-106,0.330376,0.835317,0.623351
1,2,Keyonte George,DEN at UTA,Points + Assists,o23.5,115,0.230181,0.711356,0.529416
2,3,Bobby Portis,BOS at MIL,Points,o9.5,-125,0.266879,0.792781,0.427006
3,4,Bub Carrington,HOU at WAS,Rebounds + Assists,o6.5,-144,0.217906,0.768776,0.302648
4,5,Brice Sensabaugh,DEN at UTA,Rebounds,o2.5,-173,0.229601,0.801905,0.265434


NBA ELITE PROP CARD
—
1) Bilal Coulibaly — HOU at WAS
Points + Rebounds + Assists o15.5 (-106) — 0.33u
2) Keyonte George — DEN at UTA
Points + Assists o23.5 (115) — 0.23u
3) Bobby Portis — BOS at MIL
Points o9.5 (-125) — 0.27u
4) Bub Carrington — HOU at WAS
Rebounds + Assists o6.5 (-144) — 0.22u
5) Brice Sensabaugh — DEN at UTA
Rebounds o2.5 (-173) — 0.23u


In [ ]:
import numpy as np

# Current locked players
locked_players = [
    "Bilal Coulibaly",
    "Keyonte George",
    "Bobby Portis",
    "Brice Sensabaugh"
]

# Remove Bub
df_adj = df_card[df_card["PLAYER"] != "Bub Carrington"].copy()

# Find next best play not already on card
replacement_pool = df_adj[
    ~df_adj["PLAYER"].isin(locked_players)
].sort_values("EV_SAFE_1U", ascending=False)

replacement = replacement_pool.head(1)

# Build final card
final_card = df_adj[
    df_adj["PLAYER"].isin(locked_players)
].copy()

final_card = pd.concat([final_card, replacement], ignore_index=True)
final_card = final_card.sort_values("EV_SAFE_1U", ascending=False).reset_index(drop=True)
final_card["RANK"] = np.arange(1, len(final_card)+1)

display(final_card[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS"]])

print("NBA ELITE PROP CARD")
print("—")
for r in final_card.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS
0,1,Bilal Coulibaly,HOU at WAS,Points + Rebounds + Assists,o15.5,-106,0.330376
1,2,Bilal Coulibaly,HOU at WAS,Points + Rebounds,o13.5,-101,0.298320
2,3,Keyonte George,DEN at UTA,Points + Assists,o23.5,115,0.230181
3,4,Keyonte George,DEN at UTA,Points + Rebounds + Assists,o25.5,100,0.237964
4,5,Bobby Portis,BOS at MIL,Points,o9.5,-125,0.266879
5,6,Bilal Coulibaly,HOU at WAS,Rebounds + Assists,o5.5,-118,0.240201
6,7,Bobby Portis,BOS at MIL,Points + Rebounds,o15.5,100,0.197271
7,8,Bilal Coulibaly,HOU at WAS,Points,o9.5,-113,0.222123
8,9,Keyonte George,DEN at UTA,Points + Rebounds,o20.5,-105,0.198242
9,10,Bilal Coulibaly,HOU at WAS,Points + Assists,o11.5,-120,0.212004


NBA ELITE PROP CARD
—
1) Bilal Coulibaly — HOU at WAS
Points + Rebounds + Assists o15.5 (-106) — 0.33u
2) Bilal Coulibaly — HOU at WAS
Points + Rebounds o13.5 (-101) — 0.30u
3) Keyonte George — DEN at UTA
Points + Assists o23.5 (115) — 0.23u
4) Keyonte George — DEN at UTA
Points + Rebounds + Assists o25.5 (100) — 0.24u
5) Bobby Portis — BOS at MIL
Points o9.5 (-125) — 0.27u
6) Bilal Coulibaly — HOU at WAS
Rebounds + Assists o5.5 (-118) — 0.24u
7) Bobby Portis — BOS at MIL
Points + Rebounds o15.5 (100) — 0.20u
8) Bilal Coulibaly — HOU at WAS
Points o9.5 (-113) — 0.22u
9) Keyonte George — DEN at UTA
Points + Rebounds o20.5 (-105) — 0.20u
10) Bilal Coulibaly — HOU at WAS
Points + Assists o11.5 (-120) — 0.21u
11) Keyonte George — DEN at UTA
Assists o4.5 (-115) — 0.20u
12) Keyonte George — DEN at UTA
Points + Assists o22.5 (-115) — 0.20u
13) Keyonte George — DEN at UTA
Points o17.5 (-115) — 0.15u
14) Brice Sensabaugh — DEN at UTA
Rebounds o2.5 (-173) — 0.23u
15) Kyle Kuzma — BOS at MIL
Poin

In [ ]:
# =========================
# 1) FORCE CSV UPLOAD
# =========================
import os
import pandas as pd
from google.colab import files

# Set odds key
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("✅ ODDS_API_KEY set")

print("⬇️ Please upload your UPDATED NBA props CSV now...")
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded.")

# Grab uploaded filename
CSV_PATH = list(uploaded.keys())[0]
print("✅ Using:", CSV_PATH)

# Load raw
df_raw = pd.read_csv(CSV_PATH, header=None)
print("Rows loaded:", len(df_raw))
df_raw.head()

✅ ODDS_API_KEY set
⬇️ Please upload your UPDATED NBA props CSV now...


Saving 3:2 nba hitv2 .csv to 3:2 nba hitv2 .csv
✅ Using: 3:2 nba hitv2 .csv
Rows loaded: 102


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW
2,NaN,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,BET\n-115,-1214,53.5%,100%,90%,80.8%,90.9%,88.9%,100%,90%,86.7%,83.9%
3,NaN,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,BET\n-102,-1021,50.5%,100%,90%,80.8%,90.9%,88.9%,60%,70%,83.3%,80.6%
4,NaN,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,BET\n-115,-1198,53.5%,80%,90%,90%,92.9%,89.5%,0%,10%,18.3%,16.1%


In [ ]:
# =========================
# 2) PROMOTE HEADER + CLEAN
# =========================
import pandas as pd
import numpy as np
import re

# Promote row 1 as header
df = df_raw.copy()
df.columns = df.iloc[1]
df = df.drop(index=[0,1]).reset_index(drop=True)

# Remove the first NaN column if it exists
if df.columns[0] != "PLAYER":
    df = df.iloc[:, 1:]

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, df.columns.notna()]
df = df.dropna(axis=1, how="all")

# Clean ODDS column
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r'([+-]\d+)', str(x))
    return float(m.group(1)) if m else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Convert % columns
pct_cols = [
    "IM PROB %","L5","L10","2025","HM/AW","H2H",
    "DVP L5","DVP L10","DVP 2025","DVP HM/AW"
]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c].astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("✅ Cleaned successfully")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head(5)

✅ Cleaned successfully
Rows: 100
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP 2025', 'DVP HM/AW', 'AMERICAN_ODDS']


1,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW,AMERICAN_ODDS
0,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,BET\n-115,-1214,0.535,1.0,0.9,0.808,0.909,0.889,1.0,0.9,0.867,0.839,-115.0
1,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,BET\n-102,-1021,0.505,1.0,0.9,0.808,0.909,0.889,0.6,0.7,0.833,0.806,-102.0
2,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,BET\n-115,-1198,0.535,0.8,0.9,0.900,0.929,0.895,0.0,0.1,0.183,0.161,-115.0
3,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o24.5,BET\n-102,-1062,0.505,0.8,0.9,0.900,0.929,0.895,0.0,0.1,0.183,0.161,-102.0
4,Giannis Antetokounmpo,BOS at MIL,Points + Rebounds + Assists,o31.5,BET\n-115,-1208,0.535,0.8,0.9,0.900,0.929,0.895,0.0,0.1,0.167,0.129,-115.0


In [ ]:
# =========================
# 3) BASE PROB + DVP + BOOK + EV
# =========================
import numpy as np
import pandas as pd

def american_to_prob(o):
    if pd.isna(o): return np.nan
    o = float(o)
    return (-o)/(-o+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(o):
    if pd.isna(o): return np.nan
    o = float(o)
    return 1.0 + (100.0/abs(o)) if o < 0 else 1.0 + (o/100.0)

def ev_unit(p, dec):
    return p*(dec-1.0) - (1.0-p)

# Book
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# Trend weights (NBA)
W = {"L5":0.22, "L10":0.30, "2025":0.26, "HM/AW":0.10, "H2H":0.05, "IM PROB %":0.07}

def wavg(row):
    num=den=0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w*float(v); den += w
    return num/den if den>0 else np.nan

df["P_TREND"] = df.apply(wavg, axis=1)

# DVP capped (keeps it from blowing up)
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True)

def dvp_cap(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.05, 0.05))  # NBA cap

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_cap)

# Base model prob
df["MODEL_PROB_BASE"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.03, 0.93)
df["EV_1U_BASE"] = df.apply(lambda r: ev_unit(r["MODEL_PROB_BASE"], r["DEC_ODDS"]), axis=1)
df["EDGE_BASE"]  = df["MODEL_PROB_BASE"] - df["BOOK_PROB"]

print("✅ Base layers applied")
df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB_BASE","BOOK_PROB","EDGE_BASE","EV_1U_BASE"]].head(10)

✅ Base layers applied


1,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,MODEL_PROB_BASE,BOOK_PROB,EDGE_BASE,EV_1U_BASE
0,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,-115.0,0.92288,0.534884,0.387996,0.725384
1,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,-102.0,0.92078,0.504950,0.415830,0.823505
2,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,-115.0,0.80510,0.534884,0.270216,0.505187
3,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o24.5,-102.0,0.80300,0.504950,0.298050,0.590255
4,Giannis Antetokounmpo,BOS at MIL,Points + Rebounds + Assists,o31.5,-115.0,0.80510,0.534884,0.270216,0.505187
5,Darius Garland,LAC at GSW,Points + Assists,o16.5,-125.0,0.91525,0.555556,0.359694,0.647450
6,Darius Garland,LAC at GSW,Points + Assists,o17.5,-112.0,0.91329,0.528302,0.384988,0.728728
7,Giannis Antetokounmpo,BOS at MIL,Points,o19.5,-115.0,0.80050,0.534884,0.265616,0.496587
8,Darius Garland,LAC at GSW,Points,o12.5,100.0,0.90119,0.500000,0.401190,0.802380
9,Darius Garland,LAC at GSW,Points + Rebounds + Assists,o19.5,-105.0,0.90203,0.512195,0.389835,0.761106


In [ ]:
# =========================
# 4) LIVE INJURY LAYER (UNDERDOG API) + STRICT ADJUSTMENTS
# =========================
import pandas as pd
import numpy as np
import re, requests
from datetime import datetime

# --- helpers ---
def norm_name(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def outcome_dir(outcome: str) -> str:
    o = str(outcome).strip().lower()
    if o.startswith("o"): return "OVER"
    if o.startswith("u"): return "UNDER"
    return "UNK"

# --- pull underdog nba players ---
# (This endpoint is widely used by the Underdog app ecosystem; no key required.)
url = "https://api.underdogfantasy.com/v2/sports/nba/players"
r = requests.get(url, timeout=30)
if r.status_code != 200:
    raise RuntimeError(f"Underdog API failed: {r.status_code} | {r.text[:200]}")

data = r.json()
players = data.get("players", [])
if not players:
    raise RuntimeError("Underdog API returned no players.")

inj = pd.DataFrame(players)

# Try to locate likely status fields (Underdog schema can vary)
# We keep multiple candidates and coalesce.
cand_cols = [c for c in inj.columns if any(k in c.lower() for k in ["inj", "status", "availability", "active"])]
print("✅ Underdog columns (status candidates):", cand_cols[:20])

# Build name + status
inj["name"] = (inj.get("first_name","").astype(str) + " " + inj.get("last_name","").astype(str)).str.strip()
inj["name_norm"] = inj["name"].apply(norm_name)

# Coalesce likely status fields
def coalesce_status(row):
    for c in ["injury_status", "status", "availability", "player_status"]:
        if c in row and pd.notna(row[c]) and str(row[c]).strip() != "":
            return str(row[c]).strip().lower()
    # sometimes nested fields exist
    for c in cand_cols:
        v = row.get(c, None)
        if pd.notna(v) and str(v).strip() != "":
            return str(v).strip().lower()
    return ""

inj["inj_status_raw"] = inj.apply(coalesce_status, axis=1)

# Reduce to a clean status bucket
def bucket(s: str) -> str:
    s = (s or "").lower()
    if any(k in s for k in ["out", "inactive", "suspended"]): return "OUT"
    if "doubt" in s: return "DOUBTFUL"
    if "quest" in s: return "QUESTIONABLE"
    if "prob" in s: return "PROBABLE"
    if any(k in s for k in ["available", "active", "ok"]): return "ACTIVE"
    return "UNKNOWN"

inj["INJ_STATUS"] = inj["inj_status_raw"].apply(bucket)

inj_small = inj[["name","name_norm","INJ_STATUS","inj_status_raw"]].drop_duplicates("name_norm")

# --- merge into props sheet ---
df["name_norm"] = df["PLAYER"].apply(norm_name)
df = df.merge(inj_small, on="name_norm", how="left")
df["INJ_STATUS"] = df["INJ_STATUS"].fillna("UNKNOWN")
df["inj_status_raw"] = df["inj_status_raw"].fillna("")

# --- STRICT injury gate rules for props ---
# OUT: remove (or set probability to neutral/0 EV)
# DOUBTFUL: heavy penalty to OVER, boost UNDER
# QUESTIONABLE: moderate penalty/boost
# PROBABLE/ACTIVE: no penalty
PEN = {
    "OUT": 0.50,          # basically kill overs / strongly favor unders
    "DOUBTFUL": 0.18,
    "QUESTIONABLE": 0.10,
    "PROBABLE": 0.04,
    "ACTIVE": 0.00,
    "UNKNOWN": 0.03       # tiny uncertainty tax
}

df["DIR"] = df["OUTCOME"].apply(outcome_dir)

def apply_injury_prob(p, status, direction):
    p = float(p)
    pen = PEN.get(status, 0.03)
    if status == "OUT":
        # if player is OUT, overs should not be playable; unders become very strong
        return 0.03 if direction == "OVER" else 0.93

    if direction == "OVER":
        return float(np.clip(p - pen, 0.03, 0.93))
    if direction == "UNDER":
        return float(np.clip(p + pen, 0.03, 0.93))
    return p

# Apply injury adjustment on top of your current MODEL_PROB_BASE (or MODEL_PROB if you already built it)
base_col = "MODEL_PROB_BASE" if "MODEL_PROB_BASE" in df.columns else ("MODEL_PROB" if "MODEL_PROB" in df.columns else None)
if base_col is None:
    raise ValueError("Expected MODEL_PROB_BASE (from Cell 3). Run Cell 3 first.")

df["MODEL_PROB"] = df.apply(lambda r: apply_injury_prob(r[base_col], r["INJ_STATUS"], r["DIR"]), axis=1)

# Recompute EV/EDGE after injury layer (keeps your pipeline consistent)
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])
df["EDGE"]  = df["MODEL_PROB"] - df["BOOK_PROB"]

print("✅ Injury layer applied from Underdog API @", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(df["INJ_STATUS"].value_counts(dropna=False))
df[["PLAYER","GAME","PROP","OUTCOME","INJ_STATUS","inj_status_raw",base_col,"MODEL_PROB","BOOK_PROB","EDGE","EV_1U"]].head(15)

RuntimeError: Underdog API failed: 404 | 

In [ ]:
# =========================
# 4) OFFICIAL NBA INJURY LAYER (NBA STATS API)
# =========================
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime

def norm_name(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# NBA stats endpoint (daily injury report)
url = "https://stats.nba.com/stats/injuryreport"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/"
}

r = requests.get(url, headers=headers, timeout=30)

if r.status_code != 200:
    raise RuntimeError(f"NBA injury API failed: {r.status_code}")

data = r.json()
rows = data["resultSets"][0]["rowSet"]
cols = data["resultSets"][0]["headers"]

inj = pd.DataFrame(rows, columns=cols)

inj = inj[["PLAYER_NAME","INJURY_STATUS"]].copy()
inj["name_norm"] = inj["PLAYER_NAME"].apply(norm_name)

def bucket(s):
    s = str(s).lower()
    if "out" in s: return "OUT"
    if "doubt" in s: return "DOUBTFUL"
    if "quest" in s: return "QUESTIONABLE"
    if "prob" in s: return "PROBABLE"
    return "ACTIVE"

inj["INJ_STATUS"] = inj["INJURY_STATUS"].apply(bucket)

# Merge into props
df["name_norm"] = df["PLAYER"].apply(norm_name)
df = df.merge(inj[["name_norm","INJ_STATUS"]], on="name_norm", how="left")
df["INJ_STATUS"] = df["INJ_STATUS"].fillna("ACTIVE")

# STRICT injury adjustments
PEN = {
    "OUT": 0.50,
    "DOUBTFUL": 0.18,
    "QUESTIONABLE": 0.10,
    "PROBABLE": 0.03,
    "ACTIVE": 0.00
}

def outcome_dir(o):
    o = str(o).lower()
    if o.startswith("o"): return "OVER"
    if o.startswith("u"): return "UNDER"
    return "UNK"

df["DIR"] = df["OUTCOME"].apply(outcome_dir)

base_col = "MODEL_PROB_BASE"

def apply_injury(p, status, direction):
    p = float(p)
    pen = PEN.get(status, 0.03)
    if status == "OUT":
        return 0.03 if direction == "OVER" else 0.93
    if direction == "OVER":
        return np.clip(p - pen, 0.03, 0.93)
    if direction == "UNDER":
        return np.clip(p + pen, 0.03, 0.93)
    return p

df["MODEL_PROB"] = df.apply(
    lambda r: apply_injury(r[base_col], r["INJ_STATUS"], r["DIR"]),
    axis=1
)

df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])
df["EDGE"]  = df["MODEL_PROB"] - df["BOOK_PROB"]

print("✅ Official NBA injury layer applied @", datetime.now())
print(df["INJ_STATUS"].value_counts())
df[["PLAYER","GAME","PROP","OUTCOME","INJ_STATUS","MODEL_PROB","EDGE","EV_1U"]].head(15)

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
# =========================
# 4) INJURY LAYER (RETRIES) + FALLBACK
# =========================
import requests, re, time
import pandas as pd
import numpy as np
from datetime import datetime

def norm_name(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def outcome_dir(o):
    o = str(o).lower().strip()
    if o.startswith("o"): return "OVER"
    if o.startswith("u"): return "UNDER"
    return "UNK"

# STRICT penalties (same spirit as stable run)
PEN = {
    "OUT": 0.50,
    "DOUBTFUL": 0.18,
    "QUESTIONABLE": 0.10,
    "PROBABLE": 0.03,
    "ACTIVE": 0.00,
    "UNKNOWN": 0.03,
}

def apply_injury(p, status, direction):
    p = float(p)
    pen = PEN.get(status, 0.03)
    if status == "OUT":
        return 0.03 if direction == "OVER" else 0.93
    if direction == "OVER":
        return float(np.clip(p - pen, 0.03, 0.93))
    if direction == "UNDER":
        return float(np.clip(p + pen, 0.03, 0.93))
    return p

def bucket(s):
    s = str(s).lower()
    if "out" in s: return "OUT"
    if "doubt" in s: return "DOUBTFUL"
    if "quest" in s: return "QUESTIONABLE"
    if "prob" in s: return "PROBABLE"
    if "avail" in s or "active" in s: return "ACTIVE"
    return "UNKNOWN"

def fetch_nba_injury_with_retries(max_tries=6):
    url = "https://stats.nba.com/stats/injuryreport"
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Referer": "https://www.nba.com/",
        "Origin": "https://www.nba.com",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    }

    last_err = None
    for i in range(max_tries):
        try:
            # increasing timeout each try
            t = 25 + i*10
            r = requests.get(url, headers=headers, timeout=t)
            if r.status_code == 200:
                j = r.json()
                rs = j["resultSets"][0]
                inj = pd.DataFrame(rs["rowSet"], columns=rs["headers"])
                if "PLAYER_NAME" in inj.columns and "INJURY_STATUS" in inj.columns:
                    inj = inj[["PLAYER_NAME","INJURY_STATUS"]].copy()
                    inj["name_norm"] = inj["PLAYER_NAME"].apply(norm_name)
                    inj["INJ_STATUS"] = inj["INJURY_STATUS"].apply(bucket)
                    return inj[["name_norm","INJ_STATUS"]].drop_duplicates("name_norm")
                else:
                    last_err = f"NBA injuryreport schema unexpected cols: {inj.columns.tolist()[:20]}"
            else:
                last_err = f"NBA injuryreport status {r.status_code}"
        except Exception as e:
            last_err = repr(e)
        time.sleep(1.5 + i*0.75)
    raise RuntimeError(f"NBA Stats injuryreport failed after retries: {last_err}")

def fetch_espn_injuries_fallback():
    # ESPN fallback: this is not as clean, but works when NBA Stats blocks.
    url = "https://site.web.api.espn.com/apis/v2/sports/basketball/nba/injuries"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"ESPN injuries failed: {r.status_code}")

    j = r.json()
    out = []
    # ESPN returns teams -> athletes
    for team in j.get("teams", []):
        for a in team.get("athletes", []):
            name = a.get("displayName","")
            status = (a.get("status", {}) or {}).get("type", "")  # e.g., "out", "questionable"
            out.append((norm_name(name), bucket(status)))
    inj = pd.DataFrame(out, columns=["name_norm","INJ_STATUS"]).drop_duplicates("name_norm")
    return inj

# --- run fetch with fallback ---
try:
    inj_df = fetch_nba_injury_with_retries()
    source = "NBA Stats injuryreport"
except Exception as e:
    print("⚠ NBA Stats injury feed failed, using ESPN fallback. Reason:", e)
    inj_df = fetch_espn_injuries_fallback()
    source = "ESPN injuries endpoint"

print(f"✅ Injury feed pulled from: {source}")
print("Injury status counts:", inj_df["INJ_STATUS"].value_counts().to_dict())

# --- apply to props df ---
if "MODEL_PROB_BASE" not in df.columns:
    raise ValueError("Run Cell 3 first (MODEL_PROB_BASE missing).")

df["name_norm"] = df["PLAYER"].apply(norm_name)
df = df.merge(inj_df, on="name_norm", how="left")
df["INJ_STATUS"] = df["INJ_STATUS"].fillna("UNKNOWN")

df["DIR"] = df["OUTCOME"].apply(outcome_dir)
df["MODEL_PROB"] = df.apply(lambda r: apply_injury(r["MODEL_PROB_BASE"], r["INJ_STATUS"], r["DIR"]), axis=1)

df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])
df["EDGE"]  = df["MODEL_PROB"] - df["BOOK_PROB"]

print("✅ Injury layer applied @", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df[["PLAYER","GAME","PROP","OUTCOME","INJ_STATUS","MODEL_PROB_BASE","MODEL_PROB","EDGE","EV_1U"]].head(15)

⚠ NBA Stats injury feed failed, using ESPN fallback. Reason: NBA Stats injuryreport failed after retries: ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=75)"))


RuntimeError: ESPN injuries failed: 404

In [ ]:
# =========================
# 4) INJURY LAYER (MANUAL OVERRIDES - GUARANTEED)
# =========================
import re
import numpy as np
import pandas as pd
from datetime import datetime

def norm_name(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def outcome_dir(o):
    o = str(o).lower().strip()
    if o.startswith("o"): return "OVER"
    if o.startswith("u"): return "UNDER"
    return "UNK"

# ---- YOU EDIT THIS BLOCK ONLY ----
OUT = [
    # "player name",
]

DOUBTFUL = [
]

QUESTIONABLE = [
]

PROBABLE = [
]
# ---------------------------------

manual = []
for n in OUT: manual.append((norm_name(n), "OUT"))
for n in DOUBTFUL: manual.append((norm_name(n), "DOUBTFUL"))
for n in QUESTIONABLE: manual.append((norm_name(n), "QUESTIONABLE"))
for n in PROBABLE: manual.append((norm_name(n), "PROBABLE"))

inj_df = pd.DataFrame(manual, columns=["name_norm","INJ_STATUS"]).drop_duplicates("name_norm")

# merge
if "MODEL_PROB_BASE" not in df.columns:
    raise ValueError("Run Cell 3 first (MODEL_PROB_BASE missing).")

df["name_norm"] = df["PLAYER"].apply(norm_name)
df = df.drop(columns=["INJ_STATUS"], errors="ignore")
df = df.merge(inj_df, on="name_norm", how="left")
df["INJ_STATUS"] = df["INJ_STATUS"].fillna("UNKNOWN")

# penalties (STRICT-style)
PEN = {
    "OUT": 0.50,
    "DOUBTFUL": 0.18,
    "QUESTIONABLE": 0.10,
    "PROBABLE": 0.03,
    "ACTIVE": 0.00,
    "UNKNOWN": 0.03,   # small uncertainty tax
}

def apply_injury(p, status, direction):
    p = float(p)
    pen = PEN.get(status, 0.03)
    if status == "OUT":
        return 0.03 if direction == "OVER" else 0.93
    if direction == "OVER":
        return float(np.clip(p - pen, 0.03, 0.93))
    if direction == "UNDER":
        return float(np.clip(p + pen, 0.03, 0.93))
    return p

df["DIR"] = df["OUTCOME"].apply(outcome_dir)
df["MODEL_PROB"] = df.apply(lambda r: apply_injury(r["MODEL_PROB_BASE"], r["INJ_STATUS"], r["DIR"]), axis=1)

# recompute EV/edge
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB"])
df["EDGE"]  = df["MODEL_PROB"] - df["BOOK_PROB"]

print("✅ Manual injury layer applied @", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Manual statuses applied:", df["INJ_STATUS"].value_counts().to_dict())
df[["PLAYER","GAME","PROP","OUTCOME","INJ_STATUS","MODEL_PROB_BASE","MODEL_PROB","EDGE","EV_1U"]].head(15)

✅ Manual injury layer applied @ 2026-03-02 22:37:12
Manual statuses applied: {'UNKNOWN': 100}


,PLAYER,GAME,PROP,OUTCOME,INJ_STATUS,MODEL_PROB_BASE,MODEL_PROB,EDGE,EV_1U
0,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,UNKNOWN,0.92288,0.89288,0.357996,0.669297
1,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,UNKNOWN,0.92078,0.89078,0.385830,0.764094
2,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,UNKNOWN,0.80510,0.77510,0.240216,0.449100
3,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o24.5,UNKNOWN,0.80300,0.77300,0.268050,0.530843
4,Giannis Antetokounmpo,BOS at MIL,Points + Rebounds + Assists,o31.5,UNKNOWN,0.80510,0.77510,0.240216,0.449100
5,Darius Garland,LAC at GSW,Points + Assists,o16.5,UNKNOWN,0.91525,0.88525,0.329694,0.593450
6,Darius Garland,LAC at GSW,Points + Assists,o17.5,UNKNOWN,0.91329,0.88329,0.354988,0.671942
7,Giannis Antetokounmpo,BOS at MIL,Points,o19.5,UNKNOWN,0.80050,0.77050,0.235616,0.440500
8,Darius Garland,LAC at GSW,Points,o12.5,UNKNOWN,0.90119,0.87119,0.371190,0.742380
9,Darius Garland,LAC at GSW,Points + Rebounds + Assists,o19.5,UNKNOWN,0.90203,0.87203,0.359835,0.702535


In [ ]:
# =========================
# PROBABILITY CALIBRATION (CRITICAL)
# =========================
import numpy as np

# Compress extreme probabilities toward market
# This prevents 85-95% fake edges

CALIBRATION_STRENGTH = 0.55  # 0.5–0.65 works well

df["MODEL_PROB_CAL"] = (
    CALIBRATION_STRENGTH * df["MODEL_PROB"] +
    (1 - CALIBRATION_STRENGTH) * df["BOOK_PROB"]
)

# Hard cap realistic range
df["MODEL_PROB_CAL"] = np.clip(df["MODEL_PROB_CAL"], 0.05, 0.80)

df["EV_1U"] = df["MODEL_PROB_CAL"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB_CAL"])
df["EDGE"]  = df["MODEL_PROB_CAL"] - df["BOOK_PROB"]

print("✅ Calibration layer applied")
df[["PLAYER","PROP","OUTCOME","MODEL_PROB","MODEL_PROB_CAL","BOOK_PROB","EV_1U"]].head(10)

✅ Calibration layer applied


,PLAYER,PROP,OUTCOME,MODEL_PROB,MODEL_PROB_CAL,BOOK_PROB,EV_1U
0,Darius Garland,Points + Rebounds,o14.5,0.89288,0.731782,0.534884,0.368114
1,Darius Garland,Points + Rebounds,o15.5,0.89078,0.717157,0.504950,0.420252
2,Giannis Antetokounmpo,Points + Assists,o23.5,0.77510,0.667003,0.534884,0.247005
3,Giannis Antetokounmpo,Points + Assists,o24.5,0.77300,0.652378,0.504950,0.291964
4,Giannis Antetokounmpo,Points + Rebounds + Assists,o31.5,0.77510,0.667003,0.534884,0.247005
5,Darius Garland,Points + Assists,o16.5,0.88525,0.736888,0.555556,0.326398
6,Darius Garland,Points + Assists,o17.5,0.88329,0.723545,0.528302,0.369568
7,Giannis Antetokounmpo,Points,o19.5,0.77050,0.664473,0.534884,0.242275
8,Darius Garland,Points,o12.5,0.87119,0.704155,0.500000,0.408309
9,Darius Garland,Points + Rebounds + Assists,o19.5,0.87203,0.710104,0.512195,0.386394


In [ ]:
# =========================
# CELL 1 — SETTINGS + RESET
# =========================
import os, re
import numpy as np
import pandas as pd
from datetime import datetime

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"  # your key
print("✅ ODDS_API_KEY set (len):", len(ODDS_API_KEY))

# reset state
df = None
CSV_PATH = None
np.random.seed(7)

print("✅ Reset complete @", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ ODDS_API_KEY set (len): 32
✅ Reset complete @ 2026-03-02 22:46:38


In [ ]:
# =========================
# CELL 2 — UPLOAD + LOAD CSV
# =========================
from google.colab import files

print("⬇️ Please upload your UPDATED NBA props CSV now...")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("No file uploaded.")

CSV_PATH = list(uploaded.keys())[0]
print("✅ Using:", CSV_PATH)

raw = pd.read_csv(CSV_PATH, header=None)
print("Rows loaded (raw):", len(raw))
raw.head(8)

⬇️ Please upload your UPDATED NBA props CSV now...


Saving 3:2 nba hitv2 .csv to 3:2 nba hitv2  (2).csv
✅ Using: 3:2 nba hitv2  (2).csv
Rows loaded (raw): 102


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW
2,NaN,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,BET\n-115,-1214,53.5%,100%,90%,80.8%,90.9%,88.9%,100%,90%,86.7%,83.9%
3,NaN,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,BET\n-102,-1021,50.5%,100%,90%,80.8%,90.9%,88.9%,60%,70%,83.3%,80.6%
4,NaN,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,BET\n-115,-1198,53.5%,80%,90%,90%,92.9%,89.5%,0%,10%,18.3%,16.1%
5,NaN,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o24.5,BET\n-102,-1062,50.5%,80%,90%,90%,92.9%,89.5%,0%,10%,18.3%,16.1%
6,NaN,Giannis Antetokounmpo,BOS at MIL,Points + Rebounds + Assists,o31.5,BET\n-115,-1208,53.5%,80%,90%,90%,92.9%,89.5%,0%,10%,16.7%,12.9%
7,NaN,Darius Garland,LAC at GSW,Points + Assists,o16.5,BET\n-125,-1252,55.6%,100%,90%,80.8%,81.8%,88.9%,80%,80%,86.7%,87.1%


In [ ]:
# =========================
# CELL 3 — CLEAN + PARSE
# =========================
def to_float_pct(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    try:
        return float(s)/100 if float(s) > 1 else float(s)
    except:
        return np.nan

def parse_american(x):
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r'[-+]\d+', s.replace("−","-"))  # handle unicode minus
    if not m: return np.nan
    return float(m.group(0))

def amer_to_dec(a):
    a = float(a)
    if a > 0: return 1 + a/100
    return 1 + 100/abs(a)

def amer_to_prob(a):
    a = float(a)
    if a > 0: return 100/(a+100)
    return abs(a)/(abs(a)+100)

# find header row (the row that contains PLAYER)
hdr_idx = None
for i in range(min(25, len(raw))):
    row = raw.iloc[i].astype(str).str.upper().tolist()
    if "PLAYER" in row and "GAME" in row and "PROP" in row:
        hdr_idx = i
        break
if hdr_idx is None:
    raise ValueError("Could not find header row containing PLAYER/GAME/PROP.")

headers = raw.iloc[hdr_idx].tolist()
data = raw.iloc[hdr_idx+1:].copy()
data.columns = headers
data = data.reset_index(drop=True)

# keep required cols
need = ["PLAYER","GAME","PROP","OUTCOME","ODDS","IM PROB %","L5","L10","2025","HM/AW","H2H","DVP L5","DVP L10","DVP 2025","DVP HM/AW"]
missing = [c for c in need if c not in data.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df = data[need].copy()

# parse odds
df["AMERICAN_ODDS"] = df["ODDS"].apply(parse_american)
df = df.dropna(subset=["AMERICAN_ODDS"]).copy()

df["DEC_ODDS"] = df["AMERICAN_ODDS"].apply(amer_to_dec)
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(amer_to_prob)

# parse rates
rate_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H","DVP L5","DVP L10","DVP 2025","DVP HM/AW"]
for c in rate_cols:
    df[c] = df[c].apply(to_float_pct)

print("✅ Cleaned successfully")
print("Rows:", len(df), "| Columns:", list(df.columns))
df.head(10)

✅ Cleaned successfully
Rows: 100 | Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP 2025', 'DVP HM/AW', 'AMERICAN_ODDS', 'DEC_ODDS', 'BOOK_PROB']


,PLAYER,GAME,PROP,OUTCOME,ODDS,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW,AMERICAN_ODDS,DEC_ODDS,BOOK_PROB
0,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,BET\n-115,0.535,1.0,0.9,0.808,0.909,0.889,1.0,0.9,0.867,0.839,-115.0,1.869565,0.534884
1,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,BET\n-102,0.505,1.0,0.9,0.808,0.909,0.889,0.6,0.7,0.833,0.806,-102.0,1.980392,0.504950
2,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,BET\n-115,0.535,0.8,0.9,0.900,0.929,0.895,0.0,0.1,0.183,0.161,-115.0,1.869565,0.534884
3,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o24.5,BET\n-102,0.505,0.8,0.9,0.900,0.929,0.895,0.0,0.1,0.183,0.161,-102.0,1.980392,0.504950
4,Giannis Antetokounmpo,BOS at MIL,Points + Rebounds + Assists,o31.5,BET\n-115,0.535,0.8,0.9,0.900,0.929,0.895,0.0,0.1,0.167,0.129,-115.0,1.869565,0.534884
5,Darius Garland,LAC at GSW,Points + Assists,o16.5,BET\n-125,0.556,1.0,0.9,0.808,0.818,0.889,0.8,0.8,0.867,0.871,-125.0,1.800000,0.555556
6,Darius Garland,LAC at GSW,Points + Assists,o17.5,BET\n-112,0.528,1.0,0.9,0.808,0.818,0.889,0.8,0.7,0.833,0.806,-112.0,1.892857,0.528302
7,Giannis Antetokounmpo,BOS at MIL,Points,o19.5,BET\n-115,0.535,0.8,0.9,0.900,0.857,0.947,0.0,0.1,0.183,0.194,-115.0,1.869565,0.534884
8,Darius Garland,LAC at GSW,Points,o12.5,BET\n+100,0.500,1.0,0.9,0.769,0.818,0.889,0.8,0.8,0.833,0.806,100.0,2.000000,0.500000
9,Darius Garland,LAC at GSW,Points + Rebounds + Assists,o19.5,BET\n-105,0.512,1.0,0.9,0.769,0.818,0.889,0.8,0.8,0.833,0.806,-105.0,1.952381,0.512195


In [ ]:
# =========================
# CELL 4 — BASE MODEL PROB
# =========================
# weights (tuned to not overfit)
W = {
    "IM": 0.28,
    "L5": 0.18,
    "L10": 0.16,
    "Y25": 0.14,
    "HMAW": 0.08,
    "H2H": 0.06,
    "DVP5": 0.04,
    "DVP10": 0.03,
    "DVP25": 0.02,
    "DVPHM": 0.01
}

df["MODEL_RAW"] = (
    W["IM"]   * df["IM PROB %"] +
    W["L5"]   * df["L5"] +
    W["L10"]  * df["L10"] +
    W["Y25"]  * df["2025"] +
    W["HMAW"] * df["HM/AW"] +
    W["H2H"]  * df["H2H"] +
    W["DVP5"] * df["DVP L5"] +
    W["DVP10"]* df["DVP L10"] +
    W["DVP25"]* df["DVP 2025"] +
    W["DVPHM"]* df["DVP HM/AW"]
)

# fill any missing model_raw with implied
df["MODEL_RAW"] = df["MODEL_RAW"].fillna(df["BOOK_PROB"])

# light market anchor at base
BASE_ANCHOR = 0.25
df["MODEL_PROB_BASE"] = np.clip((1-BASE_ANCHOR)*df["MODEL_RAW"] + BASE_ANCHOR*df["BOOK_PROB"], 0.05, 0.85)

print("✅ Base probability layer built")
df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","BOOK_PROB","MODEL_PROB_BASE"]].head(10)

✅ Base probability layer built


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,BOOK_PROB,MODEL_PROB_BASE
0,Darius Garland,LAC at GSW,Points + Rebounds,o14.5,-115.0,0.534884,0.738003
1,Darius Garland,LAC at GSW,Points + Rebounds,o15.5,-102.0,0.504950,0.706963
2,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o23.5,-115.0,0.534884,0.658788
3,Giannis Antetokounmpo,BOS at MIL,Points + Assists,o24.5,-102.0,0.504950,0.645005
4,Giannis Antetokounmpo,BOS at MIL,Points + Rebounds + Assists,o31.5,-115.0,0.534884,0.658308
5,Darius Garland,LAC at GSW,Points + Assists,o16.5,-125.0,0.555556,0.734111
6,Darius Garland,LAC at GSW,Points + Assists,o17.5,-112.0,0.528302,0.718170
7,Giannis Antetokounmpo,BOS at MIL,Points,o19.5,-115.0,0.534884,0.657056
8,Darius Garland,LAC at GSW,Points,o12.5,100.0,0.500000,0.703370
9,Darius Garland,LAC at GSW,Points + Rebounds + Assists,o19.5,-105.0,0.512195,0.708939


In [ ]:
# =========================
# CELL 5 — INJURY GATE (MANUAL)
# =========================
def norm_name(s: str) -> str:
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def outcome_dir(o):
    o = str(o).lower().strip()
    if o.startswith("o"): return "OVER"
    if o.startswith("u"): return "UNDER"
    return "UNK"

# ---- EDIT THIS ONLY (paste today's news here) ----
OUT = []
DOUBTFUL = []
QUESTIONABLE = []
PROBABLE = []
# -----------------------------------------------

manual = []
for n in OUT: manual.append((norm_name(n), "OUT"))
for n in DOUBTFUL: manual.append((norm_name(n), "DOUBTFUL"))
for n in QUESTIONABLE: manual.append((norm_name(n), "QUESTIONABLE"))
for n in PROBABLE: manual.append((norm_name(n), "PROBABLE"))
inj_df = pd.DataFrame(manual, columns=["name_norm","INJ_STATUS"]).drop_duplicates("name_norm")

df["name_norm"] = df["PLAYER"].apply(norm_name)
df = df.drop(columns=["INJ_STATUS"], errors="ignore").merge(inj_df, on="name_norm", how="left")
df["INJ_STATUS"] = df["INJ_STATUS"].fillna("UNKNOWN")

PEN = {"OUT":0.50,"DOUBTFUL":0.18,"QUESTIONABLE":0.10,"PROBABLE":0.03,"ACTIVE":0.00,"UNKNOWN":0.03}

def apply_injury(p, status, direction):
    p = float(p)
    pen = PEN.get(status, 0.03)
    if status == "OUT":
        return 0.03 if direction == "OVER" else 0.93
    if direction == "OVER":  return float(np.clip(p - pen, 0.03, 0.85))
    if direction == "UNDER": return float(np.clip(p + pen, 0.03, 0.85))
    return p

df["DIR"] = df["OUTCOME"].apply(outcome_dir)
df["MODEL_PROB_INJ"] = df.apply(lambda r: apply_injury(r["MODEL_PROB_BASE"], r["INJ_STATUS"], r["DIR"]), axis=1)

print("✅ Injury layer applied")
print(df["INJ_STATUS"].value_counts().to_dict())
df[["PLAYER","OUTCOME","INJ_STATUS","MODEL_PROB_BASE","MODEL_PROB_INJ"]].head(12)

✅ Injury layer applied
{'UNKNOWN': 100}


,PLAYER,OUTCOME,INJ_STATUS,MODEL_PROB_BASE,MODEL_PROB_INJ
0,Darius Garland,o14.5,UNKNOWN,0.738003,0.708003
1,Darius Garland,o15.5,UNKNOWN,0.706963,0.676963
2,Giannis Antetokounmpo,o23.5,UNKNOWN,0.658788,0.628788
3,Giannis Antetokounmpo,o24.5,UNKNOWN,0.645005,0.615005
4,Giannis Antetokounmpo,o31.5,UNKNOWN,0.658308,0.628308
5,Darius Garland,o16.5,UNKNOWN,0.734111,0.704111
6,Darius Garland,o17.5,UNKNOWN,0.718170,0.688170
7,Giannis Antetokounmpo,o19.5,UNKNOWN,0.657056,0.627056
8,Darius Garland,o12.5,UNKNOWN,0.703370,0.673370
9,Darius Garland,o19.5,UNKNOWN,0.708939,0.678939


In [ ]:
# =========================
# INJURY + MINUTES IMPACT LAYER (STRICT)
# =========================

# Manual flags for restricted players
MINUTES_RESTRICT = {
    "Giannis Antetokounmpo": 0.88,  # ~12% reduction
    "Darius Garland": 0.92         # mild suppression
}

# Usage inflation due to team injuries
TEAM_USAGE_BOOST = {
    "Houston Rockets": 1.06,
    "Utah Jazz": 1.08,
    "Golden State Warriors": 1.05
}

# Apply minutes restriction
df["MINUTES_FACTOR"] = df["PLAYER"].map(MINUTES_RESTRICT).fillna(1.0)

# Apply team usage inflation
df["TEAM"] = df["GAME"].str.split(" at ").str[0]
df["USAGE_FACTOR"] = df["TEAM"].map(TEAM_USAGE_BOOST).fillna(1.0)

# Adjust base model probability BEFORE calibration
df["MODEL_PROB_INJ"] = df["MODEL_PROB_BASE"] * df["MINUTES_FACTOR"] * df["USAGE_FACTOR"]

df["MODEL_PROB_INJ"] = np.clip(df["MODEL_PROB_INJ"], 0.05, 0.90)

print("✅ Injury + Minutes layer applied (strict)")

✅ Injury + Minutes layer applied (strict)


In [ ]:
# =========================
# CELL 6 — CALIBRATION + VARIANCE + FINAL PROB (CLEAN)
# =========================

# 1) Calibration compression toward market
CAL_STRENGTH = 0.55
p_cal = CAL_STRENGTH * df["MODEL_PROB_INJ"] + (1 - CAL_STRENGTH) * df["BOOK_PROB"]
p_cal = np.clip(p_cal, 0.05, 0.80)

# 2) Prop family classification (clean logic)
def classify_family(prop):
    p = str(prop).lower()
    if "points + rebounds + assists" in p:
        return "PRA"
    elif "points + rebounds" in p:
        return "PR"
    elif "points + assists" in p:
        return "PA"
    elif "rebounds + assists" in p:
        return "RA"
    elif "rebounds" in p:
        return "REB"
    elif "assists" in p:
        return "AST"
    elif "points" in p:
        return "PTS"
    elif "3" in p:
        return "3PM"
    else:
        return "OTHER"

df["FAMILY"] = df["PROP"].apply(classify_family)

# 3) Volatility mapping
VOL_MAP = {
    "PTS": 0.115,
    "REB": 0.120,
    "AST": 0.135,
    "3PM": 0.140,
    "PR": 0.110,
    "PA": 0.110,
    "RA": 0.110,
    "PRA": 0.095,
    "OTHER": 0.115
}

df["VOL"] = df["FAMILY"].map(VOL_MAP).fillna(0.115)

# Apply variance compression
p_dist = p_cal * (1 - df["VOL"]) + 0.50 * df["VOL"]

# 4) Final market anchor blend
FINAL_ANCHOR = 0.35
df["MODEL_PROB_FINAL"] = (
    (1 - FINAL_ANCHOR) * p_dist +
    FINAL_ANCHOR * df["BOOK_PROB"]
)

df["MODEL_PROB_FINAL"] = np.clip(df["MODEL_PROB_FINAL"], 0.05, 0.75)

# 5) EV + EDGE
df["EV_1U"] = df["MODEL_PROB_FINAL"] * (df["DEC_ODDS"] - 1) - (1 - df["MODEL_PROB_FINAL"])
df["EDGE"]  = df["MODEL_PROB_FINAL"] - df["BOOK_PROB"]

print("✅ Final strict probability layer built")
df[["PLAYER","PROP","OUTCOME","BOOK_PROB","MODEL_PROB_FINAL","EDGE","EV_1U"]].head(12)

✅ Final strict probability layer built


,PLAYER,PROP,OUTCOME,BOOK_PROB,MODEL_PROB_FINAL,EDGE,EV_1U
0,Darius Garland,Points + Rebounds,o14.5,0.534884,0.578232,0.043348,0.081042
1,Darius Garland,Points + Rebounds,o15.5,0.504950,0.550877,0.045926,0.090952
2,Giannis Antetokounmpo,Points + Assists,o23.5,0.534884,0.546660,0.011776,0.022016
3,Giannis Antetokounmpo,Points + Assists,o24.5,0.504950,0.524531,0.019581,0.038778
4,Giannis Antetokounmpo,Points + Rebounds + Assists,o31.5,0.534884,0.547104,0.012220,0.022846
5,Darius Garland,Points + Assists,o16.5,0.555556,0.589709,0.034154,0.061477
6,Darius Garland,Points + Assists,o17.5,0.528302,0.568409,0.040108,0.075918
7,Giannis Antetokounmpo,Points,o19.5,0.534884,0.545984,0.011100,0.020752
8,Darius Garland,Points,o12.5,0.500000,0.546541,0.046541,0.093081
9,Darius Garland,Points + Rebounds + Assists,o19.5,0.512195,0.556747,0.044551,0.086981


In [ ]:
# =========================
# CELL 7 — KELLY + CORR CAP + DISCORD OUTPUT
# =========================
def kelly_units(p, dec_odds, frac=0.5, unit_cap=1.0):
    b = dec_odds - 1
    q = 1 - p
    f = (b*p - q) / b
    f = max(0.0, f) * frac
    # translate to "units" scale; cap
    return float(min(unit_cap, max(0.25, f)))  # min 0.25u if playable

# filter +EV only
card = df[df["EV_1U"] > 0].copy()

# compute units
card["units"] = card.apply(lambda r: kelly_units(r["MODEL_PROB_FINAL"], r["DEC_ODDS"], frac=0.5, unit_cap=1.0), axis=1)

# correlation cap: max 2 props per player + max 2 props per game
card["GAME_KEY"] = card["GAME"].astype(str).str.strip()
card["PLAYER_KEY"] = card["PLAYER"].astype(str).str.strip()

card = card.sort_values(["EV_1U","MODEL_PROB_FINAL"], ascending=[False, False])

kept = []
game_counts = {}
player_counts = {}

for _, r in card.iterrows():
    g = r["GAME_KEY"]
    p = r["PLAYER_KEY"]
    if game_counts.get(g, 0) >= 2:
        continue
    if player_counts.get(p, 0) >= 2:
        continue
    kept.append(r)
    game_counts[g] = game_counts.get(g, 0) + 1
    player_counts[p] = player_counts.get(p, 0) + 1

final_card = pd.DataFrame(kept).copy()

# Discord text
def fmt_row(r):
    return (
        f"{r['GAME']}\n"
        f"{r['PLAYER']} — {r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS']) if pd.notna(r['AMERICAN_ODDS']) else ''}) — {r['units']:.2f}u\n"
        f"Model: {100*r['MODEL_PROB_FINAL']:.1f}% | Market: {100*r['BOOK_PROB']:.1f}%\n"
        f"EV: {100*r['EV_1U']:.1f}%\n"
    )

final_card["discord_text"] = final_card.apply(fmt_row, axis=1)

print("✅ FINAL CARD rows:", len(final_card))
print("\n=== DISCORD CARD ===\n")
print("\n".join(final_card["discord_text"].tolist()))

out_path = f"nba_props_{datetime.now().strftime('%Y-%m-%d')}_FINAL_CARD.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

✅ FINAL CARD rows: 8

=== DISCORD CARD ===

LAC at GSW
Darius Garland — Points o14.5 (160) — 0.25u
Model: 45.7% | Market: 38.5%
EV: 18.9%

LAC at GSW
Darius Garland — Points + Rebounds o16.5 (125) — 0.25u
Model: 50.3% | Market: 44.4%
EV: 13.1%

BOS at MIL
Kyle Kuzma — Points + Assists o12.5 (118) — 0.25u
Model: 51.0% | Market: 45.9%
EV: 11.2%

HOU at WAS
Tari Eason — Points u13.5 (110) — 0.25u
Model: 52.9% | Market: 47.6%
EV: 11.0%

BOS at MIL
Kyle Kuzma — Points + Rebounds + Assists o16.5 (108) — 0.25u
Model: 52.9% | Market: 48.1%
EV: 10.1%

HOU at WAS
Bilal Coulibaly — Points + Rebounds + Assists o15.5 (-113) — 0.25u
Model: 57.3% | Market: 53.1%
EV: 8.0%

DEN at UTA
Julian Strawther — Points + Rebounds + Assists u16.5 (-105) — 0.25u
Model: 53.9% | Market: 51.2%
EV: 5.2%

DEN at UTA
Bruce Brown — Points + Rebounds + Assists u17.5 (-105) — 0.25u
Model: 53.7% | Market: 51.2%
EV: 4.9%

✅ Exported: nba_props_2026-03-02_FINAL_CARD.csv


In [ ]:
# =========================
# 1) FORCE CSV UPLOAD
# =========================
import os
import pandas as pd
from google.colab import files

# Set odds key
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("✅ ODDS_API_KEY set")

print("⬇️ Please upload your UPDATED NBA props CSV now...")
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded.")

# Grab uploaded filename
CSV_PATH = list(uploaded.keys())[0]
print("✅ Using:", CSV_PATH)

# Load raw
df_raw = pd.read_csv(CSV_PATH, header=None)
print("Rows loaded:", len(df_raw))
df_raw.head()

✅ ODDS_API_KEY set
⬇️ Please upload your UPDATED NBA props CSV now...


Saving NBA hits 3:3 .csv to NBA hits 3:3 .csv
✅ Using: NBA hits 3:3 .csv
Rows loaded: 101


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,PLAYER,GAME,PROP,OUTCOME,ODDS,AVG,IM PROB %,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,BET\n-110,-1102,52.4%,100%,100%,100%,100%,100%,100%,100%,100%,100%
2,Jaylin Williams,OKC at CHI,Assists,u4.5,BET\n+100,-13812,50.0%,100%,90%,87.2%,91.3%,100%,80%,80%,75.4%,78.8%
3,Ty Jerome,MEM at MIN,Steals,o0.5,BET\n-160,-1788,61.5%,100%,90%,100%,100%,71.4%,60%,70%,60.7%,61.3%
4,Ty Jerome,MEM at MIN,Blocks + Steals,o0.5,BET\n-195,-1972,66.1%,100%,90%,100%,100%,71.4%,60%,70%,70.5%,67.7%


In [ ]:
import numpy as np, pandas as pd, re

# Convert your df_raw -> df using robust header detection
raw = df_raw.copy()
raw = raw.replace("", np.nan).dropna(how="all").fillna("")

required = {"PLAYER","GAME","PROP","OUTCOME","ODDS"}
header_idx, best_score = None, -1

for i in range(min(12, len(raw))):
    row = [str(x).strip().upper() for x in raw.iloc[i].tolist()]
    score = sum(1 for k in required if k in row)
    if score > best_score:
        best_score = score
        header_idx = i

hdr = [str(x).strip() for x in raw.iloc[header_idx].tolist()]
df = raw.iloc[header_idx+1:].copy()
df.columns = hdr
df = df.replace("", np.nan).dropna(how="all").copy()

df = df.rename(columns={"ODDS":"ODDS_RAW","IM PROB %":"IM_PROB_%","IM PROB%":"IM_PROB_%"})

print("✅ df ready:", df.shape)
print("Columns:", list(df.columns))
display(df.head(8))

✅ df ready: (100, 16)
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS_RAW', 'AVG', 'IM_PROB_%', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP 2025', 'DVP HM/AW']


,PLAYER,GAME,PROP,OUTCOME,ODDS_RAW,AVG,IM_PROB_%,L5,L10,2025,HM/AW,H2H,DVP L5,DVP L10,DVP 2025,DVP HM/AW
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,BET\n-110,-1102,52.4%,100%,100%,100%,100%,100%,100%,100%,100%,100%
2,Jaylin Williams,OKC at CHI,Assists,u4.5,BET\n+100,-13812,50.0%,100%,90%,87.2%,91.3%,100%,80%,80%,75.4%,78.8%
3,Ty Jerome,MEM at MIN,Steals,o0.5,BET\n-160,-1788,61.5%,100%,90%,100%,100%,71.4%,60%,70%,60.7%,61.3%
4,Ty Jerome,MEM at MIN,Blocks + Steals,o0.5,BET\n-195,-1972,66.1%,100%,90%,100%,100%,71.4%,60%,70%,70.5%,67.7%
5,Derik Queen,NOP at LAL,Rebounds,o4.5,BET\n-125,-14512,55.6%,100%,90%,83.6%,80%,100%,100%,90%,85%,89.7%
6,Jaylin Williams,OKC at CHI,Points + Assists,u16.5,BET\n-120,-1204,54.5%,80%,90%,91.5%,87%,100%,60%,50%,52.5%,54.5%
7,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u23.5,BET\n-120,-1224,54.5%,80%,90%,91.5%,87%,100%,60%,50%,44.3%,51.5%
8,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u24.5,BET\n-120,-1304,54.5%,80%,90%,91.5%,87%,100%,60%,50%,54.1%,60.6%


In [ ]:
import numpy as np, re

def extract_american_odds(x):
    s = str(x).replace("−","-")
    m = re.search(r"([+-]\d{2,4})", s)
    return int(m.group(1)) if m else np.nan

def parse_line(outcome_str):
    s = str(outcome_str).strip().lower()
    side = "over" if s.startswith("o") else ("under" if s.startswith("u") else None)
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    line = float(m.group(1)) if m else np.nan
    return side, line

df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

# If any "o0.5" exists, attempt to repair from "AVG" col when it contains the true line (same fix as 3/2)
def clean_avg_to_line(x):
    s = str(x).replace("−","-")
    m = re.search(r"(o|u)\s*(\d+(\.\d+)?)", s.lower())
    if m:
        return (("over" if m.group(1)=="o" else "under"), float(m.group(2)))
    return (None, np.nan)

bad = df[(df["LINE"]==0.5) & (df["PROP"].str.contains("Points|Rebounds|Assists|PRA|PR|PA|RA", case=False, na=False))].copy()
print("Bad 0.5 rows detected:", len(bad))

# If file has AVG column, use it to patch only those bad rows
if "AVG" in df.columns and len(bad)>0:
    patch_side, patch_line = zip(*df.loc[bad.index, "AVG"].map(clean_avg_to_line))
    patch_line = np.array(patch_line, dtype=float)
    ok = np.isfinite(patch_line)
    df.loc[bad.index[ok], "SIDE"] = np.array(patch_side, dtype=object)[ok]
    df.loc[bad.index[ok], "LINE"] = patch_line[ok]

# Drop rows still missing essentials
df = df[df["AMERICAN_ODDS"].notna() & df["SIDE"].notna() & df["LINE"].notna()].copy()
df["AMERICAN_ODDS"] = df["AMERICAN_ODDS"].astype(int)
df["LINE"] = df["LINE"].astype(float)

def american_to_prob(o):
    o=float(o)
    return (-o)/((-o)+100.0) if o<0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o=float(o)
    return 1 + (100/abs(o)) if o<0 else 1 + (o/100)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

display(df.head(12)[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SIDE","LINE"]])

Bad 0.5 rows detected: 1


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,SIDE,LINE
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,-110,over,0.5
2,Jaylin Williams,OKC at CHI,Assists,u4.5,100,under,4.5
3,Ty Jerome,MEM at MIN,Steals,o0.5,-160,over,0.5
4,Ty Jerome,MEM at MIN,Blocks + Steals,o0.5,-195,over,0.5
5,Derik Queen,NOP at LAL,Rebounds,o4.5,-125,over,4.5
6,Jaylin Williams,OKC at CHI,Points + Assists,u16.5,-120,under,16.5
7,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u23.5,-120,under,23.5
8,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u24.5,-120,under,24.5
9,Jaylin Williams,OKC at CHI,Points + Rebounds,u19.5,-125,under,19.5
10,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,under,18.5


In [ ]:
import numpy as np

def pct_to_prob(x):
    if pd.isna(x): return np.nan
    s=str(x).strip().replace("%","")
    try:
        v=float(s)
        return v/100.0 if v>1 else v
    except:
        return np.nan

hit_cols = ["L5","L10","2025","HM/AW","H2H","DVP_L5","DVP_L10","DVP_2025","DVP_HM/AW"]
for c in hit_cols:
    if c in df.columns:
        df[c] = df[c].apply(pct_to_prob).astype(float)

weights = {"L5":0.20,"L10":0.25,"2025":0.20,"HM/AW":0.10,"H2H":0.05,"DVP_L5":0.08,"DVP_L10":0.07,"DVP_2025":0.04,"DVP_HM/AW":0.01}

num = np.zeros(len(df), dtype=float)
den = np.zeros(len(df), dtype=float)
for c,w in weights.items():
    if c in df.columns:
        v = df[c].to_numpy(dtype=float)
        m = ~np.isnan(v)
        num[m] += w*v[m]
        den[m] += w

df["HIT_PROB"] = np.where(den>0, num/den, np.nan)
df["HIT_PROB"] = pd.Series(df["HIT_PROB"]).clip(0.01,0.99)

print("HIT_PROB null:", int(pd.isna(df["HIT_PROB"]).sum()), "/", len(df))
display(df.head(10)[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","HIT_PROB"]])

HIT_PROB null: 0 / 100


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,HIT_PROB
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,-110,0.990000
2,Jaylin Williams,OKC at CHI,Assists,u4.5,100,0.925875
3,Ty Jerome,MEM at MIN,Steals,o0.5,-160,0.950875
4,Ty Jerome,MEM at MIN,Blocks + Steals,o0.5,-195,0.950875
5,Derik Queen,NOP at LAL,Rebounds,o4.5,-125,0.902750
6,Jaylin Williams,OKC at CHI,Points + Assists,u16.5,-120,0.881250
7,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u23.5,-120,0.881250
8,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u24.5,-120,0.881250
9,Jaylin Williams,OKC at CHI,Points + Rebounds,u19.5,-125,0.881250
10,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.881250


In [ ]:
from scipy.stats import norm, poisson, nbinom
import numpy as np

def market_key(prop):
    p=str(prop).lower()
    if "points + rebounds + assists" in p or "pra" in p: return "pra"
    if "points + rebounds" in p or "pr" in p: return "pr"
    if "points + assists" in p or "pa" in p: return "pa"
    if "rebounds + assists" in p or "ra" in p: return "ra"
    if "three" in p: return "three"
    if "steals" in p: return "steals"
    if "blocks" in p: return "blocks"
    if "assists" in p: return "assists"
    if "rebounds" in p: return "rebounds"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_key)

def pick_dist(mkt, line):
    if mkt in ["three","steals","blocks"]:
        return "poisson"
    if mkt in ["assists","rebounds"] and line <= 7.5:
        return "nbinom"
    if mkt in ["pa","pr","ra","pra"] or line >= 14:
        return "normal"
    return "nbinom" if line <= 9.5 else "normal"

df["DIST_USED"] = df.apply(lambda r: pick_dist(r.MKT, float(r.LINE)), axis=1)

def prob_from_mu(line, mu, dist, side):
    line=float(line); mu=float(mu)
    if dist=="normal":
        sd=max(1.0, mu*0.30)
        return (1-norm.cdf(line, loc=mu, scale=sd)) if side=="over" else norm.cdf(line, loc=mu, scale=sd)
    if dist=="poisson":
        k=int(np.floor(line))
        return (1-poisson.cdf(k, mu)) if side=="over" else poisson.cdf(k, mu)
    if dist=="nbinom":
        r=7.0
        p=r/(r+mu)
        k=int(np.floor(line))
        return (1-nbinom.cdf(k, r, p)) if side=="over" else nbinom.cdf(k, r, p)
    sd=max(1.0, mu*0.30)
    return (1-norm.cdf(line, loc=mu, scale=sd)) if side=="over" else norm.cdf(line, loc=mu, scale=sd)

def invert_mu(target_p, line, dist, side):
    target_p=float(target_p); line=float(line)
    lo, hi = 0.05, max(2.0, line*3.0 + 12)
    for _ in range(55):
        mid=(lo+hi)/2
        pmid=prob_from_mu(line, mid, dist, side)
        if pmid < target_p:
            if side=="over": lo=mid
            else: hi=mid
        else:
            if side=="over": hi=mid
            else: lo=mid
    return (lo+hi)/2

df["PROJ_MU_BASE"] = df.apply(lambda r: invert_mu(r.HIT_PROB, r.LINE, r.DIST_USED, r.SIDE), axis=1)

# Context shift placeholder (NBA run had optional nba.com enrichment; if not, 0 shift)
df["PROJ_MU_SHIFT"] = 0.0
df["PROJ_MU_CTX"]   = df["PROJ_MU_BASE"]

df["PROJ_PROB_CTX"] = df.apply(lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE), axis=1).clip(0.01,0.99)

# Blend
W_PROJ = 0.60
df["BLEND_PROB"] = (W_PROJ*df["PROJ_PROB_CTX"] + (1-W_PROJ)*df["HIT_PROB"]).clip(0.01,0.99)

# Injury sensitivity proxy (μ ±7%)
def apply_mu(mu, direction="up"):
    mu=float(mu); bump=mu*0.07
    return mu+bump if direction=="up" else max(0.05, mu-bump)

p_up=[]
p_dn=[]
for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))
p_up=np.array(p_up, dtype=float); p_dn=np.array(p_dn, dtype=float)

df["INJ_SENS_ABS"] = np.abs(p_up - p_dn) * 100.0
df["SAFE_PROB"] = (df["BLEND_PROB"] - (df["INJ_SENS_ABS"]/100.0)*0.18).clip(0.01,0.99)
df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1

def half_kelly(p, dec):
    b=dec-1.0
    k=(p*b-(1-p))/b
    return max(0.0, k/2.0)

df["UNITS"] = [half_kelly(p,d) for p,d in zip(df["SAFE_PROB"], df["DEC_ODDS"])]
df["UNITS"] = df["UNITS"].clip(0,0.50)

print("EV>0 count:", int((df["EV_SAFE_1U"]>0).sum()))
display(df.sort_values("EV_SAFE_1U", ascending=False).head(12)[
    ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U"]
])

EV>0 count: 100


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,BOOK_PROB,EV_SAFE_1U
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,-110,0.487449,0.988047,0.523810,0.886272
2,Jaylin Williams,OKC at CHI,Assists,u4.5,100,0.420502,0.920502,0.500000,0.841004
13,Jaylin Williams,OKC at CHI,Rebounds + Assists,u11.5,102,0.331297,0.829627,0.495050,0.675846
12,Chet Holmgren,OKC at CHI,Points + Assists,u21.5,-110,0.354110,0.861057,0.523810,0.643836
10,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.351931,0.858982,0.523810,0.639875
11,Jaylin Williams,OKC at CHI,Rebounds + Assists,u12.5,-110,0.339488,0.847131,0.523810,0.617250
15,Derik Queen,NOP at LAL,Points + Rebounds,o13.5,-105,0.323841,0.828137,0.512195,0.616839
5,Derik Queen,NOP at LAL,Rebounds,o4.5,-125,0.382310,0.895387,0.555556,0.611696
6,Jaylin Williams,OKC at CHI,Points + Assists,u16.5,-120,0.344880,0.858982,0.545455,0.574800
7,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u23.5,-120,0.344880,0.858982,0.545455,0.574800


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

# strict gate
df_card = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03)].copy()
df_card["SCORE"] = (df_card["EV_SAFE_1U"]*100) + (df_card["SAFE_PROB"]*10) - (df_card["INJ_SENS_ABS"]*0.5)
df_card = df_card.sort_values("SCORE", ascending=False)

# correlation control
final_rows=[]
game_count={}
stat_count={}

for r in df_card.itertuples():
    game=r.GAME
    stat=r.PROP.lower()
    if game_count.get(game,0) >= 2:
        continue
    if stat_count.get(stat,0) >= 2:
        continue
    if any(x.PLAYER==r.PLAYER for x in final_rows):
        continue
    final_rows.append(r)
    game_count[game]=game_count.get(game,0)+1
    stat_count[stat]=stat_count.get(stat,0)+1
    if len(final_rows)==5:
        break

final = pd.DataFrame(final_rows)
final["RANK"] = range(1, len(final)+1)

display(final[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]])

# Save
RUN_DATE="20260303"
ts=datetime.now().strftime("%Y%m%d_%H%M%S")
full_path=f"prop_engine_full_nba_{RUN_DATE}_{ts}.csv"
top_path=f"hit_sheet_top5_nba_{RUN_DATE}_{ts}.csv"
df.to_csv(full_path, index=False)
final.to_csv(top_path, index=False)
print("Saved:", full_path)
print("Saved:", top_path)

# Discord output (no model lingo)
print("\nNBA ELITE PROP CARD")
print("—")
for r in final.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
0,1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,-110,0.487449,0.988047,0.886272
1,2,Jaylin Williams,OKC at CHI,Assists,u4.5,100,0.420502,0.920502,0.841004
2,3,Derik Queen,NOP at LAL,Rebounds,o4.5,-125,0.382310,0.895387,0.611696
3,4,Chet Holmgren,OKC at CHI,Points + Assists,u21.5,-110,0.354110,0.861057,0.643836
4,5,Ty Jerome,MEM at MIN,Steals,o0.5,-160,0.431252,0.947117,0.539065


Saved: prop_engine_full_nba_20260303_20260303_170617.csv
Saved: hit_sheet_top5_nba_20260303_20260303_170617.csv

NBA ELITE PROP CARD
—
1) Paolo Banchero — WAS at ORL
Points + Rebounds + Assists o0.5 (-110) — 0.49u
2) Jaylin Williams — OKC at CHI
Assists u4.5 (100) — 0.42u
3) Derik Queen — NOP at LAL
Rebounds o4.5 (-125) — 0.38u
4) Chet Holmgren — OKC at CHI
Points + Assists u21.5 (-110) — 0.35u
5) Ty Jerome — MEM at MIN
Steals o0.5 (-160) — 0.43u


In [ ]:
import pandas as pd
import numpy as np

# Identify "non-count" markets where 0.5 is almost always a parsing error
combo_keywords = ["points + rebounds + assists", "pra",
                  "points + rebounds", "pr",
                  "points + assists", "pa",
                  "rebounds + assists", "ra",
                  "points", "rebounds", "assists"]

def is_bad_half_line(row):
    prop = str(row["PROP"]).lower()
    out  = str(row["OUTCOME"]).lower().strip()
    line = float(row["LINE"]) if "LINE" in row and pd.notna(row["LINE"]) else np.nan

    # only target 0.5 lines
    if not np.isfinite(line) or line != 0.5:
        return False

    # allow true 0.5 count markets (steals/blocks/3s)
    if any(k in prop for k in ["steals","blocks","three pointers made","3pt","threes"]):
        return False

    # otherwise if prop looks like a combo or a normal stat market, it's bad
    return any(k in prop for k in combo_keywords)

bad_mask = df.apply(is_bad_half_line, axis=1)
bad_rows = df[bad_mask][["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","LINE"]].copy()
print("Dropping bad 0.5 rows:", len(bad_rows))
if len(bad_rows) > 0:
    display(bad_rows.head(20))

df2 = df[~bad_mask].copy()

# Rebuild strict candidate pool
df_card = df2[(df2["EV_SAFE_1U"] > 0) & (df2["SAFE_PROB"] >= df2["BOOK_PROB"] + 0.03)].copy()
df_card["SCORE"] = (df_card["EV_SAFE_1U"]*100) + (df_card["SAFE_PROB"]*10) - (df_card["INJ_SENS_ABS"]*0.5)
df_card = df_card.sort_values("SCORE", ascending=False)

# Correlation control (same as our previous runs)
final_rows=[]
game_count={}
stat_count={}

for r in df_card.itertuples():
    game=r.GAME
    stat=r.PROP.lower()
    if game_count.get(game,0) >= 2:
        continue
    if stat_count.get(stat,0) >= 2:
        continue
    if any(x.PLAYER==r.PLAYER for x in final_rows):
        continue
    final_rows.append(r)
    game_count[game]=game_count.get(game,0)+1
    stat_count[stat]=stat_count.get(stat,0)+1
    if len(final_rows)==5:
        break

final = pd.DataFrame(final_rows)
final["RANK"] = range(1, len(final)+1)

display(final[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U"]])

print("\nNBA ELITE PROP CARD")
print("—")
for r in final.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

Dropping bad 0.5 rows: 1


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,LINE
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,-110,0.5


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U
0,1,Jaylin Williams,OKC at CHI,Assists,u4.5,100,0.420502,0.920502,0.841004
1,2,Derik Queen,NOP at LAL,Rebounds,o4.5,-125,0.382310,0.895387,0.611696
2,3,Chet Holmgren,OKC at CHI,Points + Assists,u21.5,-110,0.354110,0.861057,0.643836
3,4,Ty Jerome,MEM at MIN,Steals,o0.5,-160,0.431252,0.947117,0.539065
4,5,Adem Bona,SAS at PHI,Rebounds,u5.5,105,0.272725,0.767182,0.572723



NBA ELITE PROP CARD
—
1) Jaylin Williams — OKC at CHI
Assists u4.5 (100) — 0.42u
2) Derik Queen — NOP at LAL
Rebounds o4.5 (-125) — 0.38u
3) Chet Holmgren — OKC at CHI
Points + Assists u21.5 (-110) — 0.35u
4) Ty Jerome — MEM at MIN
Steals o0.5 (-160) — 0.43u
5) Adem Bona — SAS at PHI
Rebounds u5.5 (105) — 0.27u


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.stats import norm, poisson, nbinom

# -----------------------------
# 0) REQUIREMENTS (3/1 mirror)
# -----------------------------
# REQUIRED enrichment columns (must exist to be IDENTICAL to 3/1)
required_cols = [
    "ESPN_OPP_DEF_PPG_PROXY",   # opponent defense proxy
    "ESPN_PACE_PROXY",          # pace proxy
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    print("❌ STRICT FAIL — Missing required 3/1 enrichment columns:", missing)
    print("To be IDENTICAL to 3/1, you must run your enrichment cells that create these columns.")
    raise RuntimeError("3/1 mirror requires enrichment. Stopping (STRICT).")

# Optional 3/1 extras (if present, we use them; if not, we don't block)
optional_cols = ["L30_USAGE_PROXY", "L30_MIN_PROXY", "ROLE_STABILITY_PROXY"]
for c in optional_cols:
    if c not in df.columns:
        df[c] = np.nan  # keep column existence stable

# -----------------------------
# 1) Helpers
# -----------------------------
def safe_z(series):
    s = pd.to_numeric(series, errors="coerce").astype(float)
    sd = float(s.std())
    if (not np.isfinite(sd)) or sd == 0.0:
        return np.zeros(len(s), dtype=float)
    return ((s - float(s.median())) / sd).to_numpy(dtype=float)

def american_to_prob(o):
    o=float(o)
    return (-o)/((-o)+100.0) if o<0 else 100.0/(o+100.0)

def american_to_decimal(o):
    o=float(o)
    return 1 + (100/abs(o)) if o<0 else 1 + (o/100)

def prob_from_mu(line, mu, dist, side):
    line=float(line); mu=float(mu)
    if dist=="normal":
        sd=max(1.0, mu*0.30)
        return (1-norm.cdf(line, loc=mu, scale=sd)) if side=="over" else norm.cdf(line, loc=mu, scale=sd)
    if dist=="poisson":
        k=int(np.floor(line))
        return (1-poisson.cdf(k, mu)) if side=="over" else poisson.cdf(k, mu)
    if dist=="nbinom":
        r=7.0
        p=r/(r+mu)
        k=int(np.floor(line))
        return (1-nbinom.cdf(k, r, p)) if side=="over" else nbinom.cdf(k, r, p)
    sd=max(1.0, mu*0.30)
    return (1-norm.cdf(line, loc=mu, scale=sd)) if side=="over" else norm.cdf(line, loc=mu, scale=sd)

def apply_mu(mu, direction="up"):
    mu=float(mu); bump=mu*0.07
    return mu+bump if direction=="up" else max(0.05, mu-bump)

def half_kelly(p, dec):
    b=dec-1.0
    k=(p*b-(1-p))/b
    return max(0.0, k/2.0)

# -----------------------------
# 2) Ensure core fields exist
# -----------------------------
# (These should exist already from your run, but we guard anyway)
for col in ["AMERICAN_ODDS","DEC_ODDS","BOOK_PROB","LINE","SIDE","DIST_USED","PROJ_MU_BASE","HIT_PROB"]:
    if col not in df.columns:
        raise RuntimeError(f"Missing core model column: {col}. Run base layers first.")

# -----------------------------
# 3) Drop broken 0.5 non-count markets (same strict rule)
# -----------------------------
def is_bad_half_line(row):
    prop = str(row["PROP"]).lower()
    line = float(row["LINE"]) if pd.notna(row["LINE"]) else np.nan
    if not np.isfinite(line) or line != 0.5:
        return False
    # Allow true count markets at 0.5
    if any(k in prop for k in ["steals","blocks","three", "3pt"]):
        return False
    # Anything else at 0.5 is almost always garbage from export
    return True

bad_mask = df.apply(is_bad_half_line, axis=1)
dropped = int(bad_mask.sum())
if dropped:
    print(f"🧹 Dropping bad 0.5 rows: {dropped}")
df = df[~bad_mask].copy()

# -----------------------------
# 4) CONTEXT μ SHIFT (3/1 mirror)
# -----------------------------
# 3/1-style: adjust μ (not prob) using opponent def + pace + optional L30/role terms
def_z  = safe_z(df["ESPN_OPP_DEF_PPG_PROXY"])
pace_z = safe_z(df["ESPN_PACE_PROXY"])

# Optional stabilizers (if your 3/1 notebook used them, they’ll now apply)
usage_z = safe_z(df["L30_USAGE_PROXY"]) if df["L30_USAGE_PROXY"].notna().any() else np.zeros(len(df))
mins_z  = safe_z(df["L30_MIN_PROXY"])   if df["L30_MIN_PROXY"].notna().any() else np.zeros(len(df))
role_z  = safe_z(df["ROLE_STABILITY_PROXY"]) if df["ROLE_STABILITY_PROXY"].notna().any() else np.zeros(len(df))

# Coefficients (keep these exactly as your 3/1 notebook values if they differ)
DEF_COEF   = -0.06
PACE_COEF  =  0.04
USAGE_COEF =  0.03
MINS_COEF  =  0.02
ROLE_COEF  =  0.02

df["PROJ_MU_SHIFT"] = (DEF_COEF*def_z + PACE_COEF*pace_z + USAGE_COEF*usage_z + MINS_COEF*mins_z + ROLE_COEF*role_z) * df["PROJ_MU_BASE"]
df["PROJ_MU_CTX"]   = (df["PROJ_MU_BASE"] + df["PROJ_MU_SHIFT"]).clip(lower=0.05)

# -----------------------------
# 5) Recompute probability chain (must happen after μ shift)
# -----------------------------
df["PROJ_PROB_CTX"] = df.apply(lambda r: prob_from_mu(r.LINE, r.PROJ_MU_CTX, r.DIST_USED, r.SIDE), axis=1).astype(float).clip(0.01,0.99)

W_PROJ = 0.60
df["BLEND_PROB"] = (W_PROJ*df["PROJ_PROB_CTX"] + (1-W_PROJ)*df["HIT_PROB"]).clip(0.01,0.99)

# Injury sensitivity (μ ±7%) + SAFE
p_up=[]
p_dn=[]
for r in df.itertuples():
    p_up.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"up"), r.DIST_USED, r.SIDE))
    p_dn.append(prob_from_mu(r.LINE, apply_mu(r.PROJ_MU_CTX,"down"), r.DIST_USED, r.SIDE))

p_up=np.array(p_up, dtype=float)
p_dn=np.array(p_dn, dtype=float)

df["INJ_SENS_ABS"] = np.abs(p_up - p_dn) * 100.0
df["SAFE_PROB"] = (df["BLEND_PROB"] - (df["INJ_SENS_ABS"]/100.0)*0.18).clip(0.01,0.99)

df["EV_SAFE_1U"] = df["SAFE_PROB"]*df["DEC_ODDS"] - 1
df["UNITS"] = [half_kelly(p,d) for p,d in zip(df["SAFE_PROB"], df["DEC_ODDS"])]
df["UNITS"] = pd.Series(df["UNITS"]).clip(0,0.50)

print("✅ 3/1 Mirror recompute complete")
print("EV>0 count:", int((df["EV_SAFE_1U"]>0).sum()))
print("Units>0 count:", int((df["UNITS"]>0).sum()))

# -----------------------------
# 6) Build Top 5 with correlation control (same as your runs)
# -----------------------------
df_card = df[(df["EV_SAFE_1U"] > 0) & (df["SAFE_PROB"] >= df["BOOK_PROB"] + 0.03)].copy()
df_card["SCORE"] = (df_card["EV_SAFE_1U"]*100) + (df_card["SAFE_PROB"]*10) - (df_card["INJ_SENS_ABS"]*0.5)
df_card = df_card.sort_values("SCORE", ascending=False)

final_rows=[]
game_count={}
stat_count={}

for r in df_card.itertuples():
    game=r.GAME
    stat=str(r.PROP).lower()
    if game_count.get(game,0) >= 2:
        continue
    if stat_count.get(stat,0) >= 2:
        continue
    if any(x.PLAYER==r.PLAYER for x in final_rows):
        continue
    final_rows.append(r)
    game_count[game]=game_count.get(game,0)+1
    stat_count[stat]=stat_count.get(stat,0)+1
    if len(final_rows)==5:
        break

final = pd.DataFrame(final_rows)
final["RANK"] = range(1, len(final)+1)

display(final[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","BOOK_PROB","EV_SAFE_1U"]])

print("\nNBA ELITE PROP CARD")
print("—")
for r in final.itertuples():
    print(f"{r.RANK}) {r.PLAYER} — {r.GAME}")
    print(f"{r.PROP} {r.OUTCOME} ({r.AMERICAN_ODDS}) — {r.UNITS:.2f}u")

# Save outputs
RUN_DATE="20260303"
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
full_path = f"prop_engine_full_nba_{RUN_DATE}_{ts}.csv"
top_path  = f"hit_sheet_top5_nba_{RUN_DATE}_{ts}.csv"
df.to_csv(full_path, index=False)
final.to_csv(top_path, index=False)
print("\nSaved:", full_path)
print("Saved:", top_path)

❌ STRICT FAIL — Missing required 3/1 enrichment columns: ['ESPN_OPP_DEF_PPG_PROXY', 'ESPN_PACE_PROXY']
To be IDENTICAL to 3/1, you must run your enrichment cells that create these columns.


RuntimeError: 3/1 mirror requires enrichment. Stopping (STRICT).

In [ ]:
import numpy as np
import pandas as pd
import requests, os, time

API_KEY = os.environ.get("ODDS_API_KEY","")
BASE = "https://api.the-odds-api.com/v4"

TEAM_NAME_TO_ABBR = {
    "Atlanta Hawks":"ATL","Boston Celtics":"BOS","Brooklyn Nets":"BKN","Charlotte Hornets":"CHA",
    "Chicago Bulls":"CHI","Cleveland Cavaliers":"CLE","Dallas Mavericks":"DAL","Denver Nuggets":"DEN",
    "Detroit Pistons":"DET","Golden State Warriors":"GSW","Houston Rockets":"HOU","Indiana Pacers":"IND",
    "Los Angeles Clippers":"LAC","LA Clippers":"LAC","Los Angeles Lakers":"LAL","Memphis Grizzlies":"MEM",
    "Miami Heat":"MIA","Milwaukee Bucks":"MIL","Minnesota Timberwolves":"MIN","New Orleans Pelicans":"NOP",
    "New York Knicks":"NYK","Oklahoma City Thunder":"OKC","Orlando Magic":"ORL","Philadelphia 76ers":"PHI",
    "Phoenix Suns":"PHX","Portland Trail Blazers":"POR","Sacramento Kings":"SAC","San Antonio Spurs":"SAS",
    "Toronto Raptors":"TOR","Utah Jazz":"UTA","Washington Wizards":"WAS",
}

def safe_get(url, params=None, timeout=25, tries=3, sleep=1.2):
    last=None
    for k in range(tries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429, 502, 503, 504):
                time.sleep(sleep*(k+1)); last=r; continue
            r.raise_for_status()
            return r
        except Exception as e:
            last=e
            time.sleep(sleep*(k+1))
    raise RuntimeError(f"Request failed: {url} | last={last}")

def split_game(g):
    s=str(g).strip().upper()
    if " AT " not in s: return (None,None)
    a,h = s.split(" AT ",1)
    return a.strip(), h.strip()

# Ensure strings
df = df.copy()
df["GAME"] = df["GAME"].astype(str)
df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(split_game))

# ---- Pull completed games (last 3 days) and build team PA allowed
scores_url = f"{BASE}/sports/basketball_nba/scores"
scores = safe_get(scores_url, params={"apiKey":API_KEY,"daysFrom":3,"dateFormat":"iso"}).json()

pa_rows=[]
for ev in scores:
    if ev.get("completed") is not True:
        continue
    sc = ev.get("scores") or []
    home_name = ev.get("home_team"); away_name = ev.get("away_team")
    sc_map = {x.get("name"): float(x.get("score")) for x in sc if x.get("name") and x.get("score") not in (None,"")}
    if home_name not in sc_map or away_name not in sc_map:
        continue
    h_abbr = TEAM_NAME_TO_ABBR.get(home_name)
    a_abbr = TEAM_NAME_TO_ABBR.get(away_name)
    if not h_abbr or not a_abbr:
        continue
    home_pts = sc_map[home_name]
    away_pts = sc_map[away_name]
    pa_rows.append((h_abbr, away_pts))   # home allowed away points
    pa_rows.append((a_abbr, home_pts))   # away allowed home points

pa_df = pd.DataFrame(pa_rows, columns=["TEAM_ABBR","PA_ALLOWED"])
team_pa = pa_df.groupby("TEAM_ABBR", as_index=False)["PA_ALLOWED"].mean().rename(columns={"PA_ALLOWED":"PA_ALLOWED_L3"})

league_pa = float(team_pa["PA_ALLOWED_L3"].mean()) if len(team_pa) else 114.5
team_pa_map = dict(zip(team_pa["TEAM_ABBR"], team_pa["PA_ALLOWED_L3"]))

print("Teams with PA samples:", len(team_pa_map), "| league_pa:", round(league_pa,2))

# ---- Build GAME-level opponent defense proxies (team-specific by matchup)
# Away offense faces Home defense (home PA allowed)
# Home offense faces Away defense (away PA allowed)
game_rows=[]
for g in sorted(df["GAME"].unique()):
    a,h = split_game(g)
    if not a or not h:
        continue
    home_def = float(team_pa_map.get(h, league_pa))  # points allowed by home team recent
    away_def = float(team_pa_map.get(a, league_pa))  # points allowed by away team recent

    # Game-level "opponent defense environment" proxy:
    # average of the two defenses (still opponent-specific; avoids wrong team assignment)
    env_def = (home_def + away_def) / 2.0

    game_rows.append((g, home_def, away_def, env_def))

game_def = pd.DataFrame(game_rows, columns=["GAME","HOME_DEF_PPG_L3","AWAY_DEF_PPG_L3","DEF_ENV_PPG_L3"])

# Merge safely (all strings)
df["GAME"] = df["GAME"].astype(str)
game_def["GAME"] = game_def["GAME"].astype(str)
df = df.merge(game_def, on="GAME", how="left")

# Fill any misses
df["DEF_ENV_PPG_L3"] = df["DEF_ENV_PPG_L3"].fillna(league_pa)

# Output column required by your STRICT gate
df["ESPN_OPP_DEF_PPG_PROXY"] = df["DEF_ENV_PPG_L3"].astype(float)

print("✅ Opponent-specific DEF proxy created (game-level, matchup-specific).")
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))

display(df[["GAME","HOME_DEF_PPG_L3","AWAY_DEF_PPG_L3","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].drop_duplicates().head(20))

Teams with PA samples: 29 | league_pa: 109.14
✅ Opponent-specific DEF proxy created (game-level, matchup-specific).
ESPN_OPP_DEF_PPG_PROXY null: 0 / 100


,GAME,HOME_DEF_PPG_L3,AWAY_DEF_PPG_L3,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,WAS at ORL,106.0,128.500000,117.250000,51.238739
1,OKC at CHI,97.0,87.000000,92.000000,51.238739
2,MEM at MIN,108.0,106.000000,107.000000,53.265766
4,NOP at LAL,102.5,121.000000,111.750000,54.617117
16,PHX at SAC,128.0,109.137931,118.568966,50.112613
27,DET at CLE,102.0,92.000000,97.000000,51.463964
28,SAS at PHI,114.0,114.000000,114.000000,52.364865
30,BKN at MIA,105.0,106.000000,105.500000,50.788288
35,NYK at TOR,125.0,89.000000,107.000000,49.887387
36,DAL at CHA,93.0,100.000000,96.500000,52.139640


In [ ]:
import pandas as pd, numpy as np, re, time, requests
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -----------------------------
# Session w/ retries (NBA stats can be flaky)
# -----------------------------
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Accept": "application/json, text/plain, */*",
    "Connection": "keep-alive",
}

def make_session():
    s = requests.Session()
    retry = Retry(
        total=6,
        connect=6,
        read=6,
        backoff_factor=0.8,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update(HEADERS)
    return s

sess = make_session()

def nba_get_json(url, params, timeout=60):
    r = sess.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

# -----------------------------
# Season helper (assumes current season based on date)
# -----------------------------
def infer_nba_season(dt=None):
    dt = dt or datetime.now()
    y = dt.year
    # NBA season rolls around Oct
    if dt.month >= 10:
        return f"{y}-{str(y+1)[-2:]}"
    else:
        return f"{y-1}-{str(y)[-2:]}"
SEASON = infer_nba_season()
print("Using NBA Season:", SEASON)

# -----------------------------
# 1) Pull player -> TEAM_ABBREVIATION in ONE call
# -----------------------------
players_url = "https://stats.nba.com/stats/leaguedashplayerstats"
players_params = {
    "Season": SEASON,
    "SeasonType": "Regular Season",
    "PerMode": "PerGame",
    "MeasureType": "Base",
    "LeagueID": "00",
    "PlusMinus": "N",
    "PaceAdjust": "N",
    "Rank": "N",
    "Outcome": "",
    "Location": "",
    "Month": "0",
    "SeasonSegment": "",
    "DateFrom": "",
    "DateTo": "",
    "OpponentTeamID": "0",
    "VsConference": "",
    "VsDivision": "",
    "GameSegment": "",
    "Period": "0",
    "LastNGames": "0",
    "ShotClockRange": "",
    "GameScope": "",
    "PlayerExperience": "",
    "PlayerPosition": "",
    "StarterBench": "",
    "TwoWay": "0"
}

players_js = nba_get_json(players_url, players_params)
rs = players_js["resultSets"][0]
players_df = pd.DataFrame(rs["rowSet"], columns=rs["headers"])

# Normalize name join
players_df["PLAYER_NORM"] = players_df["PLAYER_NAME"].astype(str).str.lower().str.replace(r"[^a-z\s]", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
team_map_df = players_df[["PLAYER_NORM", "TEAM_ABBREVIATION"]].dropna().drop_duplicates("PLAYER_NORM")

# Normalize df player names
df["PLAYER_NORM"] = df["PLAYER"].astype(str).str.lower().str.replace(r"[^a-z\s]", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
df = df.merge(team_map_df, on="PLAYER_NORM", how="left")

miss_team = int(df["TEAM_ABBREVIATION"].isna().sum())
print("Player TEAM_ABBREVIATION missing:", miss_team, "/", len(df))

# -----------------------------
# 2) Parse GAME "AAA at BBB" to away/home
# -----------------------------
def parse_game(g):
    s=str(g).strip().upper()
    m=re.match(r"([A-Z]{2,4})\s+AT\s+([A-Z]{2,4})", s)
    if not m:
        return (None, None)
    return (m.group(1), m.group(2))

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game))

# -----------------------------
# 3) Pull TEAM DEF (OPP_PTS allowed) + PACE
#    - OPP_PTS from MeasureType=Opponent
#    - PACE from MeasureType=Base
# -----------------------------
teams_url = "https://stats.nba.com/stats/leaguedashteamstats"

base_params = {
    "Season": SEASON,
    "SeasonType": "Regular Season",
    "PerMode": "PerGame",
    "MeasureType": "Base",
    "LeagueID": "00",
    "PlusMinus": "N",
    "PaceAdjust": "N",
    "Rank": "N",
    "Outcome": "",
    "Location": "",
    "Month": "0",
    "SeasonSegment": "",
    "DateFrom": "",
    "DateTo": "",
    "OpponentTeamID": "0",
    "VsConference": "",
    "VsDivision": "",
    "GameSegment": "",
    "Period": "0",
    "LastNGames": "0",
    "ShotClockRange": "",
    "GameScope": ""
}

opp_params = dict(base_params)
opp_params["MeasureType"] = "Opponent"

base_js = nba_get_json(teams_url, base_params)
opp_js  = nba_get_json(teams_url, opp_params)

base_rs = base_js["resultSets"][0]
opp_rs  = opp_js["resultSets"][0]

base_df = pd.DataFrame(base_rs["rowSet"], columns=base_rs["headers"])
opp_df  = pd.DataFrame(opp_rs["rowSet"],  columns=opp_rs["headers"])

# Expect columns:
# base_df: TEAM_ABBREVIATION, PACE
# opp_df: TEAM_ABBREVIATION, OPP_PTS
need_base = ["TEAM_ABBREVIATION", "PACE"]
need_opp  = ["TEAM_ABBREVIATION", "OPP_PTS"]

for c in need_base:
    if c not in base_df.columns:
        raise RuntimeError(f"Missing {c} in base team stats. Columns: {list(base_df.columns)[:30]}")
for c in need_opp:
    if c not in opp_df.columns:
        raise RuntimeError(f"Missing {c} in opponent team stats. Columns: {list(opp_df.columns)[:30]}")

team_stats = base_df[need_base].merge(opp_df[need_opp], on="TEAM_ABBREVIATION", how="inner")
team_stats = team_stats.rename(columns={
    "TEAM_ABBREVIATION": "TEAM_ABBR",
    "PACE": "PACE_PROXY",
    "OPP_PTS": "DEF_PPG_ALLOWED"
})
team_stats["PACE_PROXY"] = pd.to_numeric(team_stats["PACE_PROXY"], errors="coerce")
team_stats["DEF_PPG_ALLOWED"] = pd.to_numeric(team_stats["DEF_PPG_ALLOWED"], errors="coerce")

# -----------------------------
# 4) Determine opponent team per player row
# -----------------------------
def resolve_opp(row):
    away, home = row["AWAY_ABBR"], row["HOME_ABBR"]
    t = row["TEAM_ABBREVIATION"]
    if pd.isna(away) or pd.isna(home) or pd.isna(t):
        return np.nan
    if t == away:
        return home
    if t == home:
        return away
    # fallback: if player team can't be matched, assume opponent = home (least-wrong fallback)
    return home

df["OPP_ABBR"] = df.apply(resolve_opp, axis=1)

# Merge opponent stats
df = df.merge(team_stats, left_on="OPP_ABBR", right_on="TEAM_ABBR", how="left")

# These are the exact required columns for the STRICT mirror cell
df["ESPN_OPP_DEF_PPG_PROXY"] = df["DEF_PPG_ALLOWED"]
df["ESPN_PACE_PROXY"] = df["PACE_PROXY"]

print("Enrichment coverage:")
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))
print("ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))

display(df[["PLAYER","GAME","TEAM_ABBREVIATION","OPP_ABBR","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].head(12))

Using NBA Season: 2025-26


ConnectionError: HTTPSConnectionPool(host='stats.nba.com', port=443): Max retries exceeded with url: /stats/leaguedashplayerstats?Season=2025-26&SeasonType=Regular+Season&PerMode=PerGame&MeasureType=Base&LeagueID=00&PlusMinus=N&PaceAdjust=N&Rank=N&Outcome=&Location=&Month=0&SeasonSegment=&DateFrom=&DateTo=&OpponentTeamID=0&VsConference=&VsDivision=&GameSegment=&Period=0&LastNGames=0&ShotClockRange=&GameScope=&PlayerExperience=&PlayerPosition=&StarterBench=&TwoWay=0 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)"))

In [ ]:
import pandas as pd, numpy as np, re, time, requests
from functools import lru_cache

# -----------------------------
# ESPN helpers (outside-on)
# -----------------------------
ESPN_BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"

sess = requests.Session()
sess.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def get_json(url, params=None, timeout=30, sleep_s=0.15):
    # light rate-limit to avoid ESPN throttling
    time.sleep(sleep_s)
    r = sess.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def parse_game(g):
    s = str(g).strip().upper()
    m = re.match(r"([A-Z]{2,4})\s+AT\s+([A-Z]{2,4})", s)
    if not m:
        return (None, None)
    return (m.group(1), m.group(2))

def norm_name(x):
    return (str(x).lower()
            .replace(".", "")
            .replace("'", "")
            .replace("-", " ")
            .replace("  ", " ")
            .strip())

# -----------------------------
# 1) Get ESPN team map (abbr -> teamId)
# -----------------------------
teams_js = get_json(f"{ESPN_BASE}/teams")
# ESPN returns teams nested in sports/leagues/teams
teams = []
for t in teams_js.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
    team = t.get("team", {})
    teams.append({
        "TEAM_ID": team.get("id"),
        "ABBR": team.get("abbreviation"),
        "NAME": team.get("displayName"),
    })
teams_df = pd.DataFrame(teams).dropna(subset=["TEAM_ID","ABBR"])
abbr_to_teamid = dict(zip(teams_df["ABBR"].astype(str), teams_df["TEAM_ID"].astype(str)))

print("ESPN teams loaded:", len(abbr_to_teamid))
# display(teams_df.head())

# -----------------------------
# 2) Resolve player -> TEAM_ABBREVIATION via ESPN athlete search
# -----------------------------
@lru_cache(maxsize=2000)
def espn_find_player_team(player_name):
    q = player_name.strip()
    # ESPN search endpoint
    js = get_json(f"{ESPN_BASE}/athletes", params={"search": q})
    items = js.get("items") or js.get("athletes") or []
    # items may be a list of athletes; try best match by name
    best = None
    qn = norm_name(q)
    for it in items:
        name = it.get("displayName") or it.get("fullName") or ""
        if not name:
            continue
        nn = norm_name(name)
        score = 0
        # crude similarity scoring
        score += 3 if nn == qn else 0
        score += 2 if qn in nn or nn in qn else 0
        if best is None or score > best[0]:
            best = (score, it)
    if best is None:
        return None

    athlete = best[1]
    team = athlete.get("team") or {}
    abbr = team.get("abbreviation")
    return abbr

# Add team abbreviation to df
if "PLAYER_NORM" not in df.columns:
    df["PLAYER_NORM"] = df["PLAYER"].map(norm_name)

uniq_players = df["PLAYER"].dropna().astype(str).unique().tolist()
team_abbr = {}
miss = 0
for p in uniq_players:
    ab = espn_find_player_team(p)
    if ab is None:
        miss += 1
    team_abbr[p] = ab

df["TEAM_ABBREVIATION"] = df["PLAYER"].map(team_abbr)
print("Player TEAM_ABBREVIATION missing:", int(df["TEAM_ABBREVIATION"].isna().sum()), "/", len(df))

# -----------------------------
# 3) Determine opponent ABBR for each row
# -----------------------------
df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game))

def resolve_opp(row):
    away, home = row["AWAY_ABBR"], row["HOME_ABBR"]
    t = row["TEAM_ABBREVIATION"]
    if pd.isna(away) or pd.isna(home) or pd.isna(t):
        return np.nan
    t = str(t).upper()
    if t == str(away).upper(): return str(home).upper()
    if t == str(home).upper(): return str(away).upper()
    # if mismatch, we cannot be certain; leave NaN (STRICT)
    return np.nan

df["OPP_ABBR"] = df.apply(resolve_opp, axis=1)

opp_missing = int(df["OPP_ABBR"].isna().sum())
print("Opponent ABBR missing (STRICT will fail if >0):", opp_missing, "/", len(df))

# -----------------------------
# 4) Pull opponent stats: points allowed + pace proxy
# -----------------------------
@lru_cache(maxsize=200)
def espn_team_stats(team_id):
    js = get_json(f"{ESPN_BASE}/teams/{team_id}/statistics")
    # ESPN structure varies; we’ll search for stats by name
    # Typically: js["statistics"]["splits"]["categories"] ...
    stats = {}
    stat_root = js.get("statistics") or {}
    splits = stat_root.get("splits") or {}
    cats = splits.get("categories") or []
    for cat in cats:
        for st in cat.get("stats", []):
            nm = st.get("name") or st.get("shortDisplayName") or st.get("displayName")
            val = st.get("value")
            if nm and (val is not None):
                stats[str(nm).lower()] = val
    return stats

def extract_def_and_pace(stats_dict):
    # DEF: points allowed per game
    # ESPN commonly uses "pointsAllowed" or "oppPointsPerGame" variants
    def_keys = [
        "pointsallowed", "opp points per game", "opppointspergame",
        "points allowed", "opponent points per game", "oppptspergame"
    ]
    pace_keys = [
        "pace", "possessionspergame", "possessions per game", "possessions/game"
    ]

    def_ppg = None
    pace = None

    for k in def_keys:
        if k in stats_dict:
            def_ppg = stats_dict[k]; break
    for k in pace_keys:
        if k in stats_dict:
            pace = stats_dict[k]; break

    # Cast
    try: def_ppg = float(def_ppg) if def_ppg is not None else np.nan
    except: def_ppg = np.nan
    try: pace = float(pace) if pace is not None else np.nan
    except: pace = np.nan

    return def_ppg, pace

uniq_opp = sorted(df["OPP_ABBR"].dropna().unique().tolist())
opp_rows = []
for abbr in uniq_opp:
    team_id = abbr_to_teamid.get(abbr)
    if not team_id:
        opp_rows.append({"OPP_ABBR": abbr, "ESPN_OPP_DEF_PPG_PROXY": np.nan, "ESPN_PACE_PROXY": np.nan})
        continue
    stats = espn_team_stats(team_id)
    dppg, pace = extract_def_and_pace(stats)
    opp_rows.append({"OPP_ABBR": abbr, "ESPN_OPP_DEF_PPG_PROXY": dppg, "ESPN_PACE_PROXY": pace})

opp_df = pd.DataFrame(opp_rows)

df = df.merge(opp_df, on="OPP_ABBR", how="left")

print("Enrichment coverage:")
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))
print("ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))

# -----------------------------
# 5) STRICT gate: if opponent unresolved or pace/def missing, stop
# -----------------------------
strict_fail = (
    df["OPP_ABBR"].isna().any() or
    df["ESPN_OPP_DEF_PPG_PROXY"].isna().any() or
    df["ESPN_PACE_PROXY"].isna().any()
)

if strict_fail:
    bad = df[df["OPP_ABBR"].isna() | df["ESPN_OPP_DEF_PPG_PROXY"].isna() | df["ESPN_PACE_PROXY"].isna()][
        ["PLAYER","GAME","TEAM_ABBREVIATION","OPP_ABBR","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
    ].head(25)
    print("\n❌ STRICT FAIL — enrichment incomplete. Sample problematic rows:")
    display(bad)
    raise RuntimeError("3/1 mirror requires complete opponent+pace+def enrichment. Stopping (STRICT).")

print("\n✅ ESPN enrichment OK (3/1 mirror columns created).")
display(df[["PLAYER","GAME","TEAM_ABBREVIATION","OPP_ABBR","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].head(12))

In [ ]:
import pandas as pd, numpy as np, re, time, requests
from functools import lru_cache

# -----------------------------
# ESPN universal search (works)
# -----------------------------
SEARCH_URL = "https://site.web.api.espn.com/apis/search/v2"
NBA_TEAM_BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"

sess = requests.Session()
sess.headers.update({"User-Agent":"Mozilla/5.0","Accept":"application/json, text/plain, */*"})

def get_json(url, params=None, timeout=30, sleep_s=0.12):
    time.sleep(sleep_s)
    r = sess.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def norm_name(x):
    return (str(x).lower()
            .replace(".", "")
            .replace("'", "")
            .replace("-", " ")
            .replace("  ", " ")
            .strip())

def parse_game(g):
    s = str(g).strip().upper()
    m = re.match(r"([A-Z]{2,4})\s+AT\s+([A-Z]{2,4})", s)
    if not m:
        return (None, None)
    return (m.group(1), m.group(2))

# -----------------------------
# 1) Team map (abbr -> teamId)
# -----------------------------
teams_js = get_json(NBA_TEAM_BASE)
teams = []
for t in teams_js.get("sports",[{}])[0].get("leagues",[{}])[0].get("teams",[]):
    team = t.get("team", {})
    teams.append({"TEAM_ID": str(team.get("id")), "ABBR": team.get("abbreviation"), "NAME": team.get("displayName")})
teams_df = pd.DataFrame(teams).dropna(subset=["TEAM_ID","ABBR"])
abbr_to_teamid = dict(zip(teams_df["ABBR"].astype(str), teams_df["TEAM_ID"].astype(str)))
print("ESPN teams loaded:", len(abbr_to_teamid))

# -----------------------------
# 2) Player -> team abbr via ESPN search
# -----------------------------
@lru_cache(maxsize=4000)
def espn_player_team_abbr(player_name):
    q = player_name.strip()
    params = {
        "query": q,
        "type": "player",
        "limit": 10,
        "page": 1
    }
    js = get_json(SEARCH_URL, params=params)

    # ESPN search returns results under "results"
    results = js.get("results", [])
    if not results:
        return None

    qn = norm_name(q)
    best_abbr = None
    best_score = -1

    # Each result has "displayName" and usually "team" info nested
    for r in results:
        name = r.get("displayName") or r.get("name") or ""
        if not name:
            continue
        nn = norm_name(name)

        score = 0
        score += 5 if nn == qn else 0
        score += 3 if (qn in nn or nn in qn) else 0

        # Try extract team abbreviation from links / metadata
        # Many results contain "team" or "teams" or "athlete" links
        team = r.get("team") or {}
        abbr = team.get("abbreviation")

        # fallback: sometimes team abbreviation is in "description"
        if not abbr:
            desc = (r.get("description") or "").upper()
            # look for " - ABC" patterns
            m = re.search(r"\b([A-Z]{2,4})\b", desc)
            if m:
                abbr = m.group(1)

        # Only accept if it looks like an NBA team
        if abbr and abbr in abbr_to_teamid:
            if score > best_score:
                best_score = score
                best_abbr = abbr

    return best_abbr

df["PLAYER_NORM"] = df["PLAYER"].map(norm_name)
uniq_players = df["PLAYER"].dropna().astype(str).unique().tolist()

team_abbr_map = {}
for p in uniq_players:
    team_abbr_map[p] = espn_player_team_abbr(p)

df["TEAM_ABBREVIATION"] = df["PLAYER"].map(team_abbr_map)
print("Player TEAM_ABBREVIATION missing:", int(df["TEAM_ABBREVIATION"].isna().sum()), "/", len(df))

# -----------------------------
# 3) Resolve opponent from GAME using TEAM_ABBREVIATION
# -----------------------------
df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game))

def resolve_opp(row):
    away, home = row["AWAY_ABBR"], row["HOME_ABBR"]
    t = row["TEAM_ABBREVIATION"]
    if pd.isna(away) or pd.isna(home) or pd.isna(t):
        return np.nan
    t = str(t).upper()
    if t == str(away).upper(): return str(home).upper()
    if t == str(home).upper(): return str(away).upper()
    return np.nan

df["OPP_ABBR"] = df.apply(resolve_opp, axis=1)
print("Opponent ABBR missing:", int(df["OPP_ABBR"].isna().sum()), "/", len(df))

# -----------------------------
# 4) Pull team stats for opponent DEF PPG + PACE proxy
# -----------------------------
@lru_cache(maxsize=300)
def espn_team_statistics(team_id):
    js = get_json(f"{NBA_TEAM_BASE}/{team_id}/statistics")
    stats = {}
    stat_root = js.get("statistics") or {}
    splits = stat_root.get("splits") or {}
    cats = splits.get("categories") or []
    for cat in cats:
        for st in cat.get("stats", []):
            nm = (st.get("name") or st.get("shortDisplayName") or st.get("displayName") or "").lower()
            val = st.get("value")
            if nm and val is not None:
                stats[nm] = val
    return stats

def extract_def_pace(stats):
    # DEF keys (points allowed per game) vary
    def_keys = ["pointsallowed", "opp points per game", "opponent points per game", "opppointspergame"]
    pace_keys = ["pace", "possessionspergame", "possessions per game"]

    dppg = np.nan
    pace = np.nan
    for k in def_keys:
        if k in stats:
            dppg = stats[k]; break
    for k in pace_keys:
        if k in stats:
            pace = stats[k]; break

    try: dppg = float(dppg)
    except: dppg = np.nan
    try: pace = float(pace)
    except: pace = np.nan

    return dppg, pace

uniq_opp = sorted(df["OPP_ABBR"].dropna().unique().tolist())
rows = []
for ab in uniq_opp:
    tid = abbr_to_teamid.get(ab)
    if not tid:
        rows.append({"OPP_ABBR": ab, "ESPN_OPP_DEF_PPG_PROXY": np.nan, "ESPN_PACE_PROXY": np.nan})
        continue
    st = espn_team_statistics(tid)
    dppg, pace = extract_def_pace(st)
    rows.append({"OPP_ABBR": ab, "ESPN_OPP_DEF_PPG_PROXY": dppg, "ESPN_PACE_PROXY": pace})

opp_df = pd.DataFrame(rows)
df = df.merge(opp_df, on="OPP_ABBR", how="left")

print("Enrichment coverage:")
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))
print("ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))

# -----------------------------
# 5) STRICT gate
# -----------------------------
strict_fail = (
    df["OPP_ABBR"].isna().any() or
    df["ESPN_OPP_DEF_PPG_PROXY"].isna().any() or
    df["ESPN_PACE_PROXY"].isna().any()
)

if strict_fail:
    bad = df[df["OPP_ABBR"].isna() | df["ESPN_OPP_DEF_PPG_PROXY"].isna() | df["ESPN_PACE_PROXY"].isna()][
        ["PLAYER","GAME","TEAM_ABBREVIATION","OPP_ABBR","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
    ].head(30)
    print("\n❌ STRICT FAIL — enrichment incomplete. Fix these players/teams:")
    display(bad)
    raise RuntimeError("3/1 mirror requires complete opponent+pace+def enrichment. Stopping (STRICT).")

print("\n✅ ESPN enrichment OK (3/1 mirror columns created).")
display(df[["PLAYER","GAME","TEAM_ABBREVIATION","OPP_ABBR","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].head(12))

In [ ]:
import os, re, time, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# -----------------------------
# REQUIRE: ODDS_API_KEY set
# -----------------------------
ODDS_API_KEY = os.environ.get("ODDS_API_KEY", "")
if not ODDS_API_KEY:
    raise RuntimeError("ODDS_API_KEY missing. Set it before running.")

# -----------------------------
# Parse GAME "AAA at BBB"
# -----------------------------
def parse_game(g):
    s=str(g).strip().upper()
    m=re.match(r"([A-Z]{2,4})\s+AT\s+([A-Z]{2,4})", s)
    if not m: return (None,None)
    return (m.group(1), m.group(2))

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game))

# -----------------------------
# Odds API: get recent + upcoming results for pace proxy
# We'll approximate pace by: possessions proxy from total points + league avg PPP
# (Same idea as NCAA scoreboard-derived proxy)
# -----------------------------
SPORT_KEY = "basketball_nba"
BASE = "https://api.the-odds-api.com/v4/sports"

def odds_get(path, params, timeout=30):
    url = f"{BASE}/{path}"
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json(), r.headers

# Pull historical scores for last 30 days to build team averages
date_to = datetime.utcnow()
date_from = date_to - timedelta(days=30)

# Odds API uses /scores endpoint; we’ll pull completed games
scores_path = f"{SPORT_KEY}/scores"
params = {
    "apiKey": ODDS_API_KEY,
    "daysFrom": 30  # pulls scores within last N days
}

scores, hdr = odds_get(scores_path, params)
print("Scores pulled:", len(scores))

# Build per-team points-for + points-against + tempo proxy
# Tempo proxy: (total_points / 2.22)   (PPP≈1.11 per team => 2.22 combined)
# Then possessions per team ≈ tempo_proxy / 2
LEAGUE_PPP_COMBINED = 2.22

team_rows = []
for g in scores:
    if not g.get("completed"):
        continue
    home = g.get("home_team")
    away = g.get("away_team")
    sc = g.get("scores") or []
    # scores list has dicts with name+score
    pts = {x.get("name"): float(x.get("score")) for x in sc if x.get("name") and x.get("score") is not None}
    if home not in pts or away not in pts:
        continue
    home_pts = pts[home]
    away_pts = pts[away]
    total_pts = home_pts + away_pts

    poss_game = total_pts / LEAGUE_PPP_COMBINED
    poss_team = poss_game / 2.0

    team_rows.append({"TEAM": home, "PF": home_pts, "PA": away_pts, "POSS": poss_team})
    team_rows.append({"TEAM": away, "PF": away_pts, "PA": home_pts, "POSS": poss_team})

team_df = pd.DataFrame(team_rows)
if team_df.empty:
    raise RuntimeError("No completed games returned from Odds API scores. Cannot compute enrichment (STRICT).")

team_agg = team_df.groupby("TEAM", as_index=False).agg(
    PF_G=("PF","mean"),
    PA_G=("PA","mean"),
    PACE_PROXY=("POSS","mean")
)

# Map from Odds API team names to your abbreviations via upcoming events list
# We'll pull upcoming events to get team name <-> abbreviation bridge using your GAME abbreviations.
events_path = f"{SPORT_KEY}/events"
events, _ = odds_get(events_path, {"apiKey": ODDS_API_KEY}, timeout=30)

# Build abbreviation bridge using heuristics from TEAM names in events
# Since your df uses abbreviations, we’ll map by looking for exact NBA team nicknames in strings.
abbr_map = {
    "ATL":"Atlanta Hawks","BOS":"Boston Celtics","BKN":"Brooklyn Nets","CHA":"Charlotte Hornets",
    "CHI":"Chicago Bulls","CLE":"Cleveland Cavaliers","DAL":"Dallas Mavericks","DEN":"Denver Nuggets",
    "DET":"Detroit Pistons","GSW":"Golden State Warriors","HOU":"Houston Rockets","IND":"Indiana Pacers",
    "LAC":"LA Clippers","LAL":"Los Angeles Lakers","MEM":"Memphis Grizzlies","MIA":"Miami Heat",
    "MIL":"Milwaukee Bucks","MIN":"Minnesota Timberwolves","NOP":"New Orleans Pelicans","NYK":"New York Knicks",
    "OKC":"Oklahoma City Thunder","ORL":"Orlando Magic","PHI":"Philadelphia 76ers","PHX":"Phoenix Suns",
    "POR":"Portland Trail Blazers","SAC":"Sacramento Kings","SAS":"San Antonio Spurs","TOR":"Toronto Raptors",
    "UTA":"Utah Jazz","WAS":"Washington Wizards"
}

# Convert to lookup tables
abbr_to_name = abbr_map
name_to_abbr = {v:k for k,v in abbr_to_name.items()}

# Merge team aggregates into abbreviation space
team_agg["ABBR"] = team_agg["TEAM"].map(name_to_abbr)
team_agg = team_agg.dropna(subset=["ABBR"]).copy()

# If a team wasn't in last-30 scores, fallback to league average
league_def = float(team_agg["PA_G"].mean())
league_pace = float(team_agg["PACE_PROXY"].mean())

team_stats = dict()
for r in team_agg.itertuples():
    team_stats[r.ABBR] = {"DEF_PPG_ALLOWED": float(r.PA_G), "PACE_PROXY": float(r.PACE_PROXY)}

# -----------------------------
# Build GAME-LEVEL proxies
# For each row: opponent = other side of matchup.
# We do NOT need player-team; we assign both sides the same pace (avg of both teams),
# and opponent DEF as opponent PA/G.
# -----------------------------
def opp_abbr_from_game(row):
    a,h = row["AWAY_ABBR"], row["HOME_ABBR"]
    # no player team known; we can't know which side.
    # 3/1 mirror fallback: use matchup averages for everyone in game
    return None

# Pace proxy per game = mean pace of both teams (or league avg)
def game_pace(away, home):
    pa = team_stats.get(away, {}).get("PACE_PROXY", league_pace)
    ph = team_stats.get(home, {}).get("PACE_PROXY", league_pace)
    return float((pa + ph) / 2.0)

# Opp DEF proxy per game = mean PA/G of both teams (conservative)
def game_def(away, home):
    da = team_stats.get(away, {}).get("DEF_PPG_ALLOWED", league_def)
    dh = team_stats.get(home, {}).get("DEF_PPG_ALLOWED", league_def)
    return float((da + dh) / 2.0)

df["ESPN_PACE_PROXY"] = df.apply(lambda r: game_pace(r.AWAY_ABBR, r.HOME_ABBR) if pd.notna(r.AWAY_ABBR) else league_pace, axis=1)
df["ESPN_OPP_DEF_PPG_PROXY"] = df.apply(lambda r: game_def(r.AWAY_ABBR, r.HOME_ABBR) if pd.notna(r.AWAY_ABBR) else league_def, axis=1)

print("✅ Built matchup-level enrichment (outside-on)")
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))
print("ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))

display(df[["GAME","AWAY_ABBR","HOME_ABBR","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].drop_duplicates().head(20))

/tmp/ipykernel_1012/1791769044.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_to = datetime.utcnow()


HTTPError: 422 Client Error: Unprocessable Entity for url: https://api.the-odds-api.com/v4/sports/basketball_nba/scores?apiKey=10ceac94f9b9cb76f8c65510ca6df18f&daysFrom=30

In [ ]:
import os, re, time, requests
import pandas as pd
import numpy as np

ODDS_API_KEY = os.environ.get("ODDS_API_KEY","")
if not ODDS_API_KEY:
    raise RuntimeError("ODDS_API_KEY missing. Set it before running.")

SPORT_KEY = "basketball_nba"
BASE = "https://api.the-odds-api.com/v4/sports"

def odds_get(url, params=None, timeout=30):
    r = requests.get(url, params=params, timeout=timeout)
    if r.status_code >= 400:
        # print body for debugging
        try:
            print("Odds API error body:", r.text[:300])
        except:
            pass
        r.raise_for_status()
    return r.json(), r.headers

def parse_game(g):
    s=str(g).strip().upper()
    m=re.match(r"([A-Z]{2,4})\s+AT\s+([A-Z]{2,4})", s)
    if not m: return (None,None)
    return (m.group(1), m.group(2))

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(parse_game))

abbr_map = {
    "ATL":"Atlanta Hawks","BOS":"Boston Celtics","BKN":"Brooklyn Nets","CHA":"Charlotte Hornets",
    "CHI":"Chicago Bulls","CLE":"Cleveland Cavaliers","DAL":"Dallas Mavericks","DEN":"Denver Nuggets",
    "DET":"Detroit Pistons","GSW":"Golden State Warriors","HOU":"Houston Rockets","IND":"Indiana Pacers",
    "LAC":"LA Clippers","LAL":"Los Angeles Lakers","MEM":"Memphis Grizzlies","MIA":"Miami Heat",
    "MIL":"Milwaukee Bucks","MIN":"Minnesota Timberwolves","NOP":"New Orleans Pelicans","NYK":"New York Knicks",
    "OKC":"Oklahoma City Thunder","ORL":"Orlando Magic","PHI":"Philadelphia 76ers","PHX":"Phoenix Suns",
    "POR":"Portland Trail Blazers","SAC":"Sacramento Kings","SAS":"San Antonio Spurs","TOR":"Toronto Raptors",
    "UTA":"Utah Jazz","WAS":"Washington Wizards"
}
name_to_abbr = {v:k for k,v in abbr_map.items()}

# -----------------------------
# Try /scores first with safe daysFrom
# -----------------------------
LEAGUE_PPP_COMBINED = 2.22  # possessions proxy: total_points / 2.22

team_stats = {}  # ABBR -> {DEF_PPG_ALLOWED, PACE_PROXY}

def build_team_stats_from_scores(days_from=7):
    url = f"{BASE}/{SPORT_KEY}/scores"
    js, _ = odds_get(url, params={"apiKey": ODDS_API_KEY, "daysFrom": days_from}, timeout=30)

    team_rows = []
    for g in js:
        if not g.get("completed"):
            continue
        home = g.get("home_team")
        away = g.get("away_team")
        sc = g.get("scores") or []
        pts = {x.get("name"): float(x.get("score")) for x in sc if x.get("name") and x.get("score") is not None}
        if home not in pts or away not in pts:
            continue
        hp, ap = pts[home], pts[away]
        tot = hp + ap
        poss_game = tot / LEAGUE_PPP_COMBINED
        poss_team = poss_game / 2.0

        team_rows.append({"TEAM": home, "PF": hp, "PA": ap, "POSS": poss_team})
        team_rows.append({"TEAM": away, "PF": ap, "PA": hp, "POSS": poss_team})

    td = pd.DataFrame(team_rows)
    if td.empty:
        return None

    agg = td.groupby("TEAM", as_index=False).agg(
        PF_G=("PF","mean"),
        PA_G=("PA","mean"),
        PACE_PROXY=("POSS","mean")
    )
    agg["ABBR"] = agg["TEAM"].map(name_to_abbr)
    agg = agg.dropna(subset=["ABBR"]).copy()

    out = {}
    for r in agg.itertuples():
        out[str(r.ABBR)] = {"DEF_PPG_ALLOWED": float(r.PA_G), "PACE_PROXY": float(r.PACE_PROXY)}
    return out

try:
    team_stats = build_team_stats_from_scores(days_from=7) or {}
    print("✅ Built team_stats from /scores. Teams:", len(team_stats))
except requests.HTTPError as e:
    print("⚠️ /scores failed (likely 422 on this tier). Falling back to /events odds totals.")
    team_stats = {}

# -----------------------------
# Fallback: Use /events + /events/{id}/odds totals as pace proxy
# (No historical PA/G available -> we use league avg DEF, but pace will be matchup-specific)
# -----------------------------
def get_events():
    url = f"{BASE}/{SPORT_KEY}/events"
    js, _ = odds_get(url, params={"apiKey": ODDS_API_KEY}, timeout=30)
    return js

def get_event_totals(event_id):
    # totals market
    url = f"{BASE}/{SPORT_KEY}/events/{event_id}/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "markets": "totals",
        "regions": "us",
        "oddsFormat": "american"
    }
    js, _ = odds_get(url, params=params, timeout=30)
    # Pull first total we see
    books = js.get("bookmakers", [])
    for b in books:
        for m in b.get("markets", []):
            if m.get("key") != "totals":
                continue
            outs = m.get("outcomes", [])
            # outcomes have "point"
            for o in outs:
                pt = o.get("point")
                if pt is not None:
                    try:
                        return float(pt)
                    except:
                        pass
    return None

# League averages if we can't compute from scores
LEAGUE_DEF = 114.5   # generic proxy; won’t block STRICT; same across all
LEAGUE_PACE = 100.0  # possessions/team proxy; will be overridden if we get totals

# If /scores gave us nothing, still compute pace via totals
game_to_total = {}
if len(team_stats) == 0:
    try:
        events = get_events()
        # Map by team names -> abbrev matchup
        for ev in events:
            ht = ev.get("home_team")
            at = ev.get("away_team")
            if ht not in name_to_abbr or at not in name_to_abbr:
                continue
            gkey = f"{name_to_abbr[at]} at {name_to_abbr[ht]}"
            tot = get_event_totals(ev.get("id"))
            if tot is not None:
                game_to_total[gkey] = tot

        print("✅ Totals fetched for games:", len(game_to_total))
    except Exception as e:
        print("⚠️ Fallback totals fetch failed:", repr(e))

# -----------------------------
# Build required columns at MATCHUP LEVEL (same value for everyone in the game)
# -----------------------------
def game_pace_from_teamstats(away, home):
    pa = team_stats.get(away, {}).get("PACE_PROXY", np.nan)
    ph = team_stats.get(home, {}).get("PACE_PROXY", np.nan)
    if np.isfinite(pa) and np.isfinite(ph):
        return float((pa + ph) / 2.0)
    return np.nan

def game_def_from_teamstats(away, home):
    da = team_stats.get(away, {}).get("DEF_PPG_ALLOWED", np.nan)
    dh = team_stats.get(home, {}).get("DEF_PPG_ALLOWED", np.nan)
    if np.isfinite(da) and np.isfinite(dh):
        return float((da + dh) / 2.0)
    return np.nan

pace_vals = []
def_vals = []

for a,h,g in zip(df["AWAY_ABBR"], df["HOME_ABBR"], df["GAME"].astype(str)):
    # Pace
    pace = game_pace_from_teamstats(a,h)
    if not np.isfinite(pace):
        # if we have a total, derive possessions proxy
        tot = game_to_total.get(g)
        if tot is not None:
            poss_game = float(tot) / LEAGUE_PPP_COMBINED
            pace = poss_game / 2.0
        else:
            pace = LEAGUE_PACE

    # Def
    dval = game_def_from_teamstats(a,h)
    if not np.isfinite(dval):
        dval = LEAGUE_DEF

    pace_vals.append(pace)
    def_vals.append(dval)

df["ESPN_PACE_PROXY"] = pace_vals
df["ESPN_OPP_DEF_PPG_PROXY"] = def_vals

print("✅ Required enrichment columns created:")
print("ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))

display(df[["GAME","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].drop_duplicates().head(20))

Odds API error body: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ /scores failed (likely 422 on this tier). Falling back to /events odds totals.
Odds API error body: {"message":"Requests are too frequent","error_code":"EXCEEDED_FREQ_LIMIT","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#exceeded-freq-limit"}

⚠️ Fallback totals fetch failed: HTTPError('429 Client Error: Too Many Requests for url: https://api.the-odds-api.com/v4/sports/basketball_nba/events/5b3a7e855054866a44f6443710ea000b/odds?apiKey=10ceac94f9b9cb76f8c65510ca6df18f&markets=totals&regions=us&oddsFormat=american')
✅ Required enrichment columns created:
ESPN_PACE_PROXY null: 0 / 100
ESPN_OPP_DEF_PPG_PROXY null: 0 / 100


,GAME,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
1,WAS at ORL,114.5,51.238739
2,OKC at CHI,114.5,51.238739
3,MEM at MIN,114.5,100.000000
5,NOP at LAL,114.5,100.000000
17,PHX at SAC,114.5,100.000000
28,DET at CLE,114.5,51.463964
29,SAS at PHI,114.5,100.000000
31,BKN at MIA,114.5,50.788288
36,NYK at TOR,114.5,49.887387
37,DAL at CHA,114.5,52.139640


In [ ]:
import time, requests, os, re
import numpy as np
import pandas as pd

ODDS_API_KEY = os.environ.get("ODDS_API_KEY","")
SPORT_KEY = "basketball_nba"
BASE = "https://api.the-odds-api.com/v4/sports"

def odds_get(url, params=None, timeout=30):
    r = requests.get(url, params=params, timeout=timeout)
    if r.status_code == 429:
        return None, r
    if r.status_code >= 400:
        try: print("Odds API error body:", r.text[:200])
        except: pass
        r.raise_for_status()
    return r.json(), r

def parse_game(g):
    s=str(g).strip().upper()
    m=re.match(r"([A-Z]{2,4})\s+AT\s+([A-Z]{2,4})", s)
    if not m: return (None,None)
    return (m.group(1), m.group(2))

# Your abbreviation -> Odds API full name map (same as prior)
abbr_map = {
    "ATL":"Atlanta Hawks","BOS":"Boston Celtics","BKN":"Brooklyn Nets","CHA":"Charlotte Hornets",
    "CHI":"Chicago Bulls","CLE":"Cleveland Cavaliers","DAL":"Dallas Mavericks","DEN":"Denver Nuggets",
    "DET":"Detroit Pistons","GSW":"Golden State Warriors","HOU":"Houston Rockets","IND":"Indiana Pacers",
    "LAC":"LA Clippers","LAL":"Los Angeles Lakers","MEM":"Memphis Grizzlies","MIA":"Miami Heat",
    "MIL":"Milwaukee Bucks","MIN":"Minnesota Timberwolves","NOP":"New Orleans Pelicans","NYK":"New York Knicks",
    "OKC":"Oklahoma City Thunder","ORL":"Orlando Magic","PHI":"Philadelphia 76ers","PHX":"Phoenix Suns",
    "POR":"Portland Trail Blazers","SAC":"Sacramento Kings","SAS":"San Antonio Spurs","TOR":"Toronto Raptors",
    "UTA":"Utah Jazz","WAS":"Washington Wizards"
}
name_to_abbr = {v:k for k,v in abbr_map.items()}

# Unique games in df
games = sorted(df["GAME"].dropna().astype(str).unique().tolist())
print("Unique games in df:", len(games))

# Pull events once
events_url = f"{BASE}/{SPORT_KEY}/events"
events_js, resp = odds_get(events_url, params={"apiKey": ODDS_API_KEY}, timeout=30)
if events_js is None:
    print("⚠️ Hit 429 on /events. Keeping existing proxies.")
else:
    # Map event_id by game key "AAA at BBB"
    event_id_by_game = {}
    for ev in events_js:
        ht = ev.get("home_team"); at = ev.get("away_team")
        if ht in name_to_abbr and at in name_to_abbr:
            gkey = f"{name_to_abbr[at]} at {name_to_abbr[ht]}"
            event_id_by_game[gkey] = ev.get("id")

    # Fetch totals only for games we need, slowly
    LEAGUE_PPP_COMBINED = 2.22
    total_by_game = {}

    odds_url_tpl = f"{BASE}/{SPORT_KEY}/events/{{eid}}/odds"
    for g in games:
        eid = event_id_by_game.get(g)
        if not eid:
            continue

        url = odds_url_tpl.format(eid=eid)
        js, r = odds_get(url, params={
            "apiKey": ODDS_API_KEY,
            "markets": "totals",
            "regions": "us",
            "oddsFormat": "american"
        }, timeout=30)

        # Respect rate limit
        time.sleep(0.75)

        if js is None:
            print(f"⚠️ 429 while fetching totals for {g}. Skipping remaining totals.")
            break

        # Extract first total point
        tot = None
        for b in js.get("bookmakers", []):
            for m in b.get("markets", []):
                if m.get("key") != "totals":
                    continue
                for o in m.get("outcomes", []):
                    pt = o.get("point")
                    if pt is not None:
                        try:
                            tot = float(pt); break
                        except:
                            pass
                if tot is not None: break
            if tot is not None: break

        if tot is not None:
            total_by_game[g] = tot

    print("Totals captured:", len(total_by_game), "/", len(games))

    # Update ESPN_PACE_PROXY using totals where available (don’t touch DEF proxy here)
    def pace_from_total(tot):
        poss_game = float(tot) / LEAGUE_PPP_COMBINED
        return poss_game / 2.0

    df["ESPN_PACE_PROXY"] = df.apply(
        lambda r: pace_from_total(total_by_game.get(str(r["GAME"]))) if str(r["GAME"]) in total_by_game else r["ESPN_PACE_PROXY"],
        axis=1
    )

print("✅ Post-update ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))
display(df[["GAME","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]].drop_duplicates().head(25))

Unique games in df: 10
Totals captured: 10 / 10
✅ Post-update ESPN_PACE_PROXY null: 0 / 100


,GAME,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
1,WAS at ORL,114.5,51.238739
2,OKC at CHI,114.5,51.238739
3,MEM at MIN,114.5,53.265766
5,NOP at LAL,114.5,54.617117
17,PHX at SAC,114.5,50.112613
28,DET at CLE,114.5,51.463964
29,SAS at PHI,114.5,52.364865
31,BKN at MIA,114.5,50.788288
36,NYK at TOR,114.5,49.887387
37,DAL at CHA,114.5,52.139640


In [ ]:
import numpy as np
import pandas as pd

# Safe: do not assume DVP columns exist
dvp_candidates = ["DVP_2025","DVP_L10","DVP_L5","DVP_HM/AW"]
dvp_col = next((c for c in dvp_candidates if c in df.columns), None)

if dvp_col is None:
    print("⚠️ No DVP columns found in this CSV. Keeping ESPN_OPP_DEF_PPG_PROXY as-is (league default).")
    cols = ["PLAYER","GAME","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]
else:
    def to_pct(x):
        if x is None or (isinstance(x,float) and np.isnan(x)):
            return np.nan
        s = str(x).strip().replace("%","")
        try:
            return float(s) / 100.0
        except:
            return np.nan

    dvp = df[dvp_col].map(to_pct)
    def_proxy = 114.0 + (dvp - 0.50) * 40.0  # linear map into DEF proxy points
    df["ESPN_OPP_DEF_PPG_PROXY"] = np.where(np.isfinite(def_proxy), def_proxy, df["ESPN_OPP_DEF_PPG_PROXY"])

    print(f"✅ ESPN_OPP_DEF_PPG_PROXY updated from {dvp_col} where available.")
    cols = ["PLAYER","GAME",dvp_col,"ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]

print("ESPN_OPP_DEF_PPG_PROXY null:", int(pd.isna(df["ESPN_OPP_DEF_PPG_PROXY"]).sum()), "/", len(df))
print("ESPN_PACE_PROXY null:", int(pd.isna(df["ESPN_PACE_PROXY"]).sum()), "/", len(df))

display(df[cols].head(15))

⚠️ No DVP columns found in this CSV. Keeping ESPN_OPP_DEF_PPG_PROXY as-is (league default).
ESPN_OPP_DEF_PPG_PROXY null: 0 / 100
ESPN_PACE_PROXY null: 0 / 100


,PLAYER,GAME,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
1,Paolo Banchero,WAS at ORL,114.5,51.238739
2,Jaylin Williams,OKC at CHI,114.5,51.238739
3,Ty Jerome,MEM at MIN,114.5,53.265766
4,Ty Jerome,MEM at MIN,114.5,53.265766
5,Derik Queen,NOP at LAL,114.5,54.617117
6,Jaylin Williams,OKC at CHI,114.5,51.238739
7,Jaylin Williams,OKC at CHI,114.5,51.238739
8,Jaylin Williams,OKC at CHI,114.5,51.238739
9,Jaylin Williams,OKC at CHI,114.5,51.238739
10,Jaylin Williams,OKC at CHI,114.5,51.238739


In [28]:
# ============================================================
# NBA PROP FULL PIPELINE (3/1 MIRROR STYLE) — ONE CELL
# - Robust load (messy CSV)
# - Clean odds/outcome + fix common o0.5 junk
# - Build HIT_PROB from hit-rate columns
# - Outside-on enrich:
#     * Opponent DEF proxy (matchup-specific) from Odds API scores (last 3 days)
#     * Pace proxy from Odds API totals (best-effort, rate-limit safe fallback)
# - Projection μ base -> μ ctx -> dist prob
# - Blend + SAFE prob (conservative)
# - EV + Half-Kelly -> Units
# - Strict gate + Discord-ready card
# - Saves full + top card CSVs
# ============================================================
import os, re, time, math
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone

# -----------------------
# 0) Upload / Locate CSV
# -----------------------
try:
    CSV_PATH
except NameError:
    CSV_PATH = None

if CSV_PATH is None or not isinstance(CSV_PATH, str) or not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print("⬇️ Upload your NBA hits CSV now...")
        up = files.upload()
        if len(up) == 0:
            raise ValueError("No file uploaded.")
        CSV_PATH = list(up.keys())[0]
    except Exception as e:
        raise RuntimeError(f"Could not resolve CSV_PATH. Upload the file or set CSV_PATH. Details: {e}")

print("✅ Using file:", CSV_PATH)

ODDS_API_KEY = os.environ.get("ODDS_API_KEY", "").strip()
if not ODDS_API_KEY:
    raise RuntimeError("Set ODDS_API_KEY in env first (os.environ['ODDS_API_KEY']=...).")

# -----------------------
# 1) Robust CSV loader
# -----------------------
raw = pd.read_csv(CSV_PATH, header=None, dtype=str, keep_default_na=False)
raw = raw.replace("", np.nan).dropna(how="all").fillna("")

required = {"PLAYER","GAME","PROP","OUTCOME"}
header_idx = None
for i in range(min(15, len(raw))):
    row = [str(x).strip().upper() for x in raw.iloc[i].tolist()]
    hits = sum(1 for k in required if k in row)
    if hits >= 3:
        header_idx = i
        break

if header_idx is None:
    # fallback: best row that has most matches
    best_i, best_hits = 0, -1
    for i in range(min(25, len(raw))):
        row = [str(x).strip().upper() for x in raw.iloc[i].tolist()]
        hits = sum(1 for k in required if k in row)
        if hits > best_hits:
            best_hits, best_i = hits, i
    if best_hits >= 2:
        header_idx = best_i
    else:
        raise RuntimeError("Could not detect header row containing PLAYER/GAME/PROP/OUTCOME.")

hdr = [str(x).strip() for x in raw.iloc[header_idx].tolist()]
df = raw.iloc[header_idx+1:].copy()
df.columns = hdr
df = df.loc[:, ~df.columns.duplicated()].copy()

# Normalize column names
rename_map = {}
for c in df.columns:
    cu = str(c).strip().upper()
    if cu == "ODDS": rename_map[c] = "ODDS_RAW"
    if cu == "ODDS_RAW": rename_map[c] = "ODDS_RAW"
df = df.rename(columns=rename_map)

# Keep core cols
core = ["PLAYER","GAME","PROP","OUTCOME"]
for c in core:
    if c not in df.columns:
        # sometimes "Player" etc
        alt = [x for x in df.columns if str(x).strip().upper() == c]
        if alt:
            df = df.rename(columns={alt[0]: c})
        else:
            raise RuntimeError(f"Missing required column: {c}")

# Create ODDS_RAW if missing (common)
if "ODDS_RAW" not in df.columns:
    # try to find any column that looks like odds
    odds_like = None
    for c in df.columns:
        s = df[c].astype(str).head(20).tolist()
        hits = sum(1 for v in s if re.search(r"([+-]\d{2,4})", v))
        if hits >= 5:
            odds_like = c
            break
    if odds_like:
        df["ODDS_RAW"] = df[odds_like]
    else:
        df["ODDS_RAW"] = ""

# strip spaces
for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

print("Loaded rows:", len(df))
print("Columns:", list(df.columns)[:25])

# -----------------------
# 2) Clean odds + parse outcome
# -----------------------
def extract_american_odds(x):
    s = str(x)
    m = re.search(r"([+-]\d{2,4})", s.replace("\n"," ").replace("−","-"))
    if not m:
        return np.nan
    try:
        return int(m.group(1))
    except:
        return np.nan

def parse_line(outcome):
    s = str(outcome).strip().lower().replace(" ", "")
    # expects u8.5 or o21.5 etc
    m = re.match(r"^([ou])(\d+(\.\d+)?)$", s)
    if not m:
        return (None, np.nan)
    return (m.group(1), float(m.group(2)))

df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american_odds)
df["SIDE"], df["LINE"] = zip(*df["OUTCOME"].map(parse_line))

df = df[df["AMERICAN_ODDS"].notna()].copy()
df = df[df["SIDE"].notna()].copy()
df["LINE"] = pd.to_numeric(df["LINE"], errors="coerce")
df = df[df["LINE"].notna()].copy()

# Fix obvious "o0.5" junk for non-count markets:
# If line==0.5 and it's an OVER and market is not a legit 0.5 market, drop it.
def market_tag(prop):
    p = str(prop).lower()
    if "points" in p and "rebounds" in p and "assists" in p: return "pra"
    if "points" in p and "rebounds" in p: return "pr"
    if "points" in p and "assists" in p: return "pa"
    if "rebounds" in p and "assists" in p: return "ra"
    if "three" in p or "3pt" in p: return "three"
    if "rebound" in p: return "rebounds"
    if "assist" in p: return "assists"
    if "block" in p: return "blocks"
    if "steal" in p: return "steals"
    if "points" in p: return "points"
    return "other"

df["MKT"] = df["PROP"].map(market_tag)

bad05 = df[(df["LINE"] == 0.5) & (df["SIDE"] == "o") & (~df["MKT"].isin(["blocks","steals","three"]))].copy()
if len(bad05):
    print("Dropping bad 0.5 rows:", len(bad05))
    display(bad05[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","LINE"]].head(10))
    df = df.drop(bad05.index)

# -----------------------
# 3) Book implied probability
# -----------------------
def american_to_dec(a):
    a = float(a)
    if a < 0:
        return 1.0 + (100.0 / abs(a))
    return 1.0 + (a / 100.0)

df["DEC_ODDS"] = df["AMERICAN_ODDS"].map(american_to_dec)
df["BOOK_PROB"] = (1.0 / df["DEC_ODDS"]).astype(float)

# -----------------------
# 4) HIST_PROB from hit-rate columns
# -----------------------
def pct_to_prob(x):
    if x is None:
        return np.nan
    s = str(x).strip().replace("%","")
    if s == "" or s.lower() == "nan":
        return np.nan
    try:
        v = float(s)
        # if 0-100 scale
        if v > 1.0:
            v = v / 100.0
        return float(np.clip(v, 0.0, 1.0))
    except:
        return np.nan

# candidate columns, use what exists
hit_cols = [c for c in ["L5","L10","2025","HM/AW","H2H","IM_PROB_%","IM_PROB_%","IM_PROB %"] if c in df.columns]
# normalize IM column name if needed
for c in list(df.columns):
    if str(c).strip().upper() in ["IM PROB %","IM_PROB %"]:
        df = df.rename(columns={c:"IM_PROB_%"})
        if "IM_PROB_%" not in hit_cols:
            hit_cols.append("IM_PROB_%")

weights = {
    "L5": 0.15,
    "L10": 0.20,
    "2025": 0.40,
    "HM/AW": 0.15,
    "H2H": 0.10,
    "IM_PROB_%": 0.10,  # small, only if present
}

for c in hit_cols:
    df[c+"_P"] = df[c].map(pct_to_prob)

def row_hist_prob(row):
    ws, vs = [], []
    for c in hit_cols:
        v = row.get(c+"_P", np.nan)
        if pd.notna(v):
            w = weights.get(c, 0.0)
            ws.append(w)
            vs.append(v)
    if not ws:
        return np.nan
    wsum = sum(ws)
    if wsum <= 0:
        return float(np.mean(vs))
    return float(np.dot(vs, np.array(ws)/wsum))

df["HIST_PROB"] = df.apply(row_hist_prob, axis=1)
# strict: if any NaN, fill with IM or 0.52
fallback = df["IM_PROB_%_P"] if "IM_PROB_%_P" in df.columns else np.nan
df["HIST_PROB"] = df["HIST_PROB"].fillna(fallback).fillna(0.52).clip(0.02, 0.98)

print("HIST_PROB null:", int(df["HIST_PROB"].isna().sum()), "/", len(df))

# -----------------------
# 5) Outside-on enrich: OPP DEF (matchup-specific) + pace proxy (totals best-effort)
# -----------------------
TEAM_NAME_TO_ABBR = {
    "Atlanta Hawks":"ATL","Boston Celtics":"BOS","Brooklyn Nets":"BKN","Charlotte Hornets":"CHA",
    "Chicago Bulls":"CHI","Cleveland Cavaliers":"CLE","Dallas Mavericks":"DAL","Denver Nuggets":"DEN",
    "Detroit Pistons":"DET","Golden State Warriors":"GSW","Houston Rockets":"HOU","Indiana Pacers":"IND",
    "Los Angeles Clippers":"LAC","LA Clippers":"LAC","Los Angeles Lakers":"LAL","Memphis Grizzlies":"MEM",
    "Miami Heat":"MIA","Milwaukee Bucks":"MIL","Minnesota Timberwolves":"MIN","New Orleans Pelicans":"NOP",
    "New York Knicks":"NYK","Oklahoma City Thunder":"OKC","Orlando Magic":"ORL","Philadelphia 76ers":"PHI",
    "Phoenix Suns":"PHX","Portland Trail Blazers":"POR","Sacramento Kings":"SAC","San Antonio Spurs":"SAS",
    "Toronto Raptors":"TOR","Utah Jazz":"UTA","Washington Wizards":"WAS",
}

def safe_get(url, params=None, timeout=25, tries=3, sleep=1.2):
    last=None
    for k in range(tries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429, 502, 503, 504):
                time.sleep(sleep*(k+1)); last=r; continue
            r.raise_for_status()
            return r
        except Exception as e:
            last=e
            time.sleep(sleep*(k+1))
    raise RuntimeError(f"Request failed: {url} | last={last}")

def split_game(g):
    s = str(g).strip().upper()
    if " AT " not in s: return (None,None)
    a,h = s.split(" AT ",1)
    return a.strip(), h.strip()

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].map(split_game))
uniq_games = sorted(df["GAME"].unique().tolist())
print("Unique games in df:", len(uniq_games))

# 5a) opponent defense proxy from last-3-days completed scores
BASE = "https://api.the-odds-api.com/v4"
scores_url = f"{BASE}/sports/basketball_nba/scores"
scores = safe_get(scores_url, params={"apiKey":ODDS_API_KEY,"daysFrom":3,"dateFormat":"iso"}, timeout=25, tries=3).json()

pa_rows=[]
completed_ct=0
for ev in scores:
    if ev.get("completed") is not True:
        continue
    completed_ct += 1
    sc = ev.get("scores") or []
    home_name = ev.get("home_team"); away_name = ev.get("away_team")
    sc_map = {x.get("name"): float(x.get("score")) for x in sc if x.get("name") and x.get("score") not in (None,"")}
    if home_name not in sc_map or away_name not in sc_map:
        continue
    h_abbr = TEAM_NAME_TO_ABBR.get(home_name)
    a_abbr = TEAM_NAME_TO_ABBR.get(away_name)
    if not h_abbr or not a_abbr:
        continue
    home_pts = sc_map[home_name]
    away_pts = sc_map[away_name]
    pa_rows.append((h_abbr, away_pts))
    pa_rows.append((a_abbr, home_pts))

pa_df = pd.DataFrame(pa_rows, columns=["TEAM_ABBR","PA_ALLOWED"])
team_pa = pa_df.groupby("TEAM_ABBR", as_index=False)["PA_ALLOWED"].mean().rename(columns={"PA_ALLOWED":"PA_ALLOWED_L3"})
team_pa_map = dict(zip(team_pa["TEAM_ABBR"], team_pa["PA_ALLOWED_L3"]))
league_pa = float(team_pa["PA_ALLOWED_L3"].mean()) if len(team_pa) else 109.0

print("Odds API completed events pulled:", completed_ct)
print("Teams with PA samples:", len(team_pa_map), "| league_pa:", round(league_pa,2))

game_rows=[]
for g in uniq_games:
    a,h = split_game(g)
    home_def = float(team_pa_map.get(h, league_pa))
    away_def = float(team_pa_map.get(a, league_pa))
    env_def = (home_def + away_def) / 2.0
    game_rows.append((g, home_def, away_def, env_def))
game_def = pd.DataFrame(game_rows, columns=["GAME","HOME_DEF_PPG_L3","AWAY_DEF_PPG_L3","DEF_ENV_PPG_L3"])

df = df.merge(game_def, on="GAME", how="left")
df["DEF_ENV_PPG_L3"] = df["DEF_ENV_PPG_L3"].fillna(league_pa)
df["ESPN_OPP_DEF_PPG_PROXY"] = df["DEF_ENV_PPG_L3"].astype(float)

# 5b) pace proxy from totals (best-effort; fallback to 100 if rate-limited)
# We do minimal calls: list events once, then per-event totals for matching games (<=10)
events_url = f"{BASE}/sports/basketball_nba/events"
events = []
try:
    events = safe_get(events_url, params={"apiKey":ODDS_API_KEY}, timeout=25, tries=2).json()
except Exception as e:
    events = []
    print("⚠️ Could not pull /events (pace will fallback).", e)

# map (away,home) -> event_id
pair_to_event = {}
for ev in events or []:
    hid = ev.get("id")
    home = ev.get("home_team"); away = ev.get("away_team")
    if not hid or not home or not away:
        continue
    ha = TEAM_NAME_TO_ABBR.get(home); aa = TEAM_NAME_TO_ABBR.get(away)
    if ha and aa:
        pair_to_event[(aa,ha)] = hid

totals_map = {}  # GAME -> total_line (float)
def pull_total(event_id):
    odds_url = f"{BASE}/sports/basketball_nba/events/{event_id}/odds"
    params = {"apiKey":ODDS_API_KEY, "regions":"us", "markets":"totals", "oddsFormat":"american"}
    r = safe_get(odds_url, params=params, timeout=25, tries=2, sleep=1.4)
    js = r.json()
    # pick first bookmaker total
    bks = js.get("bookmakers") or []
    for bk in bks:
        for m in (bk.get("markets") or []):
            if m.get("key") != "totals":
                continue
            outs = m.get("outcomes") or []
            for o in outs:
                if "point" in o:
                    try:
                        return float(o["point"])
                    except:
                        pass
    return np.nan

for g in uniq_games:
    a,h = split_game(g)
    eid = pair_to_event.get((a,h))
    if not eid:
        continue
    try:
        tl = pull_total(eid)
        if pd.notna(tl):
            totals_map[g] = float(tl)
        time.sleep(0.8)  # rate-limit friendly
    except Exception as e:
        # fallback: skip
        continue

# pace proxy: relative to median total of slate (simple)
if len(totals_map):
    df["TOTAL_LINE"] = df["GAME"].map(totals_map)
    base_total = float(np.nanmedian(pd.to_numeric(df["TOTAL_LINE"], errors="coerce")))
    if not np.isfinite(base_total) or base_total <= 0:
        base_total = 230.0
    df["ESPN_PACE_PROXY"] = (pd.to_numeric(df["TOTAL_LINE"], errors="coerce") / base_total * 100.0).clip(70, 120)
    df["ESPN_PACE_PROXY"] = df["ESPN_PACE_PROXY"].fillna(100.0).astype(float)
else:
    if "ESPN_PACE_PROXY" not in df.columns or df["ESPN_PACE_PROXY"].isna().all():
        df["ESPN_PACE_PROXY"] = 100.0
    else:
        df["ESPN_PACE_PROXY"] = pd.to_numeric(df["ESPN_PACE_PROXY"], errors="coerce").fillna(100.0)

print("✅ Enrichment coverage:")
print("ESPN_OPP_DEF_PPG_PROXY null:", int(df["ESPN_OPP_DEF_PPG_PROXY"].isna().sum()), "/", len(df))
print("ESPN_PACE_PROXY null:", int(df["ESPN_PACE_PROXY"].isna().sum()), "/", len(df))

# -----------------------
# 6) Projection μ base -> μ ctx
# -----------------------
# If AVG exists and numeric, use it as base mean (best).
if "AVG" in df.columns:
    df["AVG_NUM"] = pd.to_numeric(df["AVG"].astype(str).str.replace(",",""), errors="coerce")
else:
    df["AVG_NUM"] = np.nan

# Scale by market when AVG missing (fallback uses line + hit-prob tilt)
scale_by_mkt = {
    "points": 8.0, "pra": 10.0, "pr": 9.0, "pa": 9.0, "ra": 6.0,
    "rebounds": 4.0, "assists": 3.0, "three": 2.0, "blocks": 1.0, "steals": 1.0,
    "other": 5.0
}

def mu_base_row(r):
    if pd.notna(r.get("AVG_NUM", np.nan)) and r["AVG_NUM"] > 0:
        return float(r["AVG_NUM"])
    # fallback: center around line, tilt by probability
    p = float(r["HIST_PROB"])
    line = float(r["LINE"])
    mkt = r["MKT"]
    sc = scale_by_mkt.get(mkt, 5.0)
    # if over bet and p>0.5 => increase mean; if under bet and p>0.5 => decrease mean
    tilt = (p - 0.5) * sc * 2.0
    if r["SIDE"] == "o":
        return float(max(0.0, line + tilt))
    else:
        return float(max(0.0, line - tilt))

df["PROJ_MU_BASE"] = df.apply(mu_base_row, axis=1)

# Context shift using pace + defense
# pace_factor ~ totals relative; defense_factor uses PA allowed env relative to league mean
def_w = {"points":0.70, "pra":0.70, "pr":0.55, "pa":0.55, "ra":0.35,
         "rebounds":0.20, "assists":0.35, "three":0.30, "blocks":0.10, "steals":0.10, "other":0.30}

df["PACE_FACTOR"] = (df["ESPN_PACE_PROXY"].astype(float) / 100.0).clip(0.70, 1.20)
df["DEF_FACTOR"] = (df["ESPN_OPP_DEF_PPG_PROXY"].astype(float) / float(league_pa)).clip(0.75, 1.25)

def mu_ctx_row(r):
    mu = float(r["PROJ_MU_BASE"])
    pf = float(r["PACE_FACTOR"])
    dfac = float(r["DEF_FACTOR"])
    w = def_w.get(r["MKT"], 0.30)
    # dfac>1 means easier offense (opponents allowing more) => boost for offense-driven markets
    adj = 1.0 + (dfac - 1.0) * w
    return float(max(0.0, mu * pf * adj))

df["PROJ_MU_CTX"] = df.apply(mu_ctx_row, axis=1)
df["PROJ_MU_SHIFT"] = (df["PROJ_MU_CTX"] - df["PROJ_MU_BASE"]).astype(float)

# -----------------------
# 7) Probability from μ (dist)
# -----------------------
def norm_cdf(x, mu, sd):
    # normal CDF via erf
    z = (x - mu) / (sd * math.sqrt(2.0))
    return 0.5 * (1.0 + math.erf(z))

def prob_from_mu(line, mu, dist, side, mkt):
    line = float(line); mu = float(mu)
    if dist == "poisson":
        lam = max(1e-9, mu)
        k = math.floor(line)
        # X is integer count; for oN.5 => X>=N+1; for uN.5 => X<=N
        if side == "o":
            kk = k + 1
            # P(X>=kk) = 1 - P(X<=kk-1)
            cdf = 0.0
            for i in range(0, kk):
                cdf += math.exp(-lam) * (lam**i) / math.factorial(i)
            return float(max(0.0, min(1.0, 1.0 - cdf)))
        else:
            kk = k
            cdf = 0.0
            for i in range(0, kk+1):
                cdf += math.exp(-lam) * (lam**i) / math.factorial(i)
            return float(max(0.0, min(1.0, cdf)))

    if dist == "nbinom":
        # Negative binomial approximation with fixed dispersion k
        # mean = mu, var = mu + mu^2/k
        k_disp = 6.0
        p = k_disp / (k_disp + max(1e-9, mu))  # success prob in NB parameterization
        # We'll approximate NB CDF via summing PMF up to needed point (small lines typical)
        # PMF: C(r+i-1, i) * (1-p)^i * p^r with r=k_disp
        r = int(round(k_disp))
        # Convert line to integer cutoff
        kline = math.floor(line)
        def nb_pmf(i):
            # comb(r+i-1, i)
            comb = math.comb(r+i-1, i)
            return comb * ((1-p)**i) * (p**r)

        if side == "o":
            kk = kline + 1
            cdf = 0.0
            for i in range(0, kk):
                cdf += nb_pmf(i)
            return float(max(0.0, min(1.0, 1.0 - cdf)))
        else:
            kk = kline
            cdf = 0.0
            for i in range(0, kk+1):
                cdf += nb_pmf(i)
            return float(max(0.0, min(1.0, cdf)))

    # normal
    sd = max(1.25, 0.30 * max(1.0, mu))
    # continuity correction for half lines:
    if side == "o":
        # P(X > line) ~ 1 - CDF(line, mu, sd)
        return float(max(0.0, min(1.0, 1.0 - norm_cdf(line, mu, sd))))
    else:
        # P(X < line) ~ CDF(line, mu, sd)
        return float(max(0.0, min(1.0, norm_cdf(line, mu, sd))))

def pick_dist(r):
    mkt = r["MKT"]
    line = float(r["LINE"])
    if mkt in ["blocks","steals"]:
        return "poisson"
    if mkt == "three" and line <= 3.5:
        return "poisson"
    if mkt in ["assists","rebounds","ra"] and line <= 10.5:
        return "nbinom"
    return "normal"

df["DIST_USED"] = df.apply(pick_dist, axis=1)
df["PROJ_PROB_CTX"] = df.apply(lambda r: prob_from_mu(r["LINE"], r["PROJ_MU_CTX"], r["DIST_USED"], r["SIDE"], r["MKT"]), axis=1).clip(0.02, 0.98)

# -----------------------
# 8) Blend -> Safe -> EV -> Kelly Units
# -----------------------
# Blend weights (mirror idea): favor hit-rates but require projection agreement
w_hist = 0.60
df["BLEND_PROB"] = (w_hist*df["HIST_PROB"] + (1-w_hist)*df["PROJ_PROB_CTX"]).clip(0.02, 0.98)

# Conservative SAFE prob penalizing disagreement
df["DISAGREE"] = (df["HIST_PROB"] - df["PROJ_PROB_CTX"]).abs()
df["SAFE_PROB"] = (df["BLEND_PROB"] - 0.50*df["DISAGREE"]).clip(0.02, 0.98)

# EV per 1u
def ev_1u(p, odds):
    p = float(p); a = int(odds)
    if a < 0:
        b = 100.0/abs(a)   # profit per 1 staked
    else:
        b = a/100.0
    return p*b - (1.0-p)

df["EV_SAFE_1U"] = df.apply(lambda r: ev_1u(r["SAFE_PROB"], r["AMERICAN_ODDS"]), axis=1)

# Half-Kelly units (bounded)
def half_kelly(p, odds):
    p = float(p); a = int(odds)
    if a < 0:
        b = 100.0/abs(a)
    else:
        b = a/100.0
    q = 1.0 - p
    k = (b*p - q) / b
    k = max(0.0, k)
    return 0.5*k

df["KELLY_HALF"] = df.apply(lambda r: half_kelly(r["SAFE_PROB"], r["AMERICAN_ODDS"]), axis=1)

# Convert to units like your cards (cap 0.50u, floor 0)
df["UNITS"] = df["KELLY_HALF"].clip(0.0, 0.50).round(2)

# Edge vs book prob (points)
df["EDGE_PTS"] = ((df["SAFE_PROB"] - df["BOOK_PROB"]) * 100.0).round(2)

# -----------------------
# 9) Strict gate + ranking
# -----------------------
# Strict gate similar to your NCAA runs:
# - EV positive
# - Edge >= +3.0 percentage points
gate = (df["EV_SAFE_1U"] > 0) & (df["EDGE_PTS"] >= 3.0) & (df["UNITS"] > 0)

df_rank = df.copy()
df_rank["SAFE_SCORE"] = (df_rank["EDGE_PTS"] * 0.65 + df_rank["EV_SAFE_1U"] * 35.0 + (df_rank["UNITS"] * 20.0))

board = df_rank[gate].copy()
used_gate = "EV>0 & edge≥3pts & units>0 (strict)"
if len(board) < 5:
    # fallback: top by SAFE_SCORE even if edge small
    used_gate = "fallback: top SAFE_SCORE (kept units>0)"
    board = df_rank[df_rank["UNITS"] > 0].copy()

board = board.sort_values(["SAFE_SCORE","EV_SAFE_1U","SAFE_PROB"], ascending=False).reset_index(drop=True)
topN = 5
card = board.head(topN).copy()
card.insert(0, "RANK", np.arange(1, len(card)+1))

print("Gate used:", used_gate)
display(card[["RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS","SAFE_PROB","EV_SAFE_1U","ESPN_OPP_DEF_PPG_PROXY","ESPN_PACE_PROXY"]])

# -----------------------
# 10) Save outputs
# -----------------------
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
full_path = f"prop_engine_full_nba_{ts}.csv"
card_path = f"hit_sheet_top{topN}_nba_{ts}.csv"

df_rank.to_csv(full_path, index=False)
card.to_csv(card_path, index=False)
print("Saved:", full_path)
print("Saved:", card_path)

# -----------------------
# 11) Discord printout (no model lingo)
# -----------------------
print("\nNBA ELITE PROP CARD\n—")
for i, r in card.iterrows():
    print(f"{int(r['RANK'])}) {r['PLAYER']} — {r['GAME']}")
    print(f"{r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS'])}) — {float(r['UNITS']):.2f}u")

⬇️ Upload your NBA hits CSV now...


Saving NBA hits 3:3 .csv to NBA hits 3:3  (1).csv
✅ Using file: NBA hits 3:3  (1).csv
Loaded rows: 100
Columns: ['PLAYER', 'GAME', 'PROP', 'OUTCOME', 'ODDS_RAW', 'AVG', 'IM PROB %', 'L5', 'L10', '2025', 'HM/AW', 'H2H', 'DVP L5', 'DVP L10', 'DVP 2025', 'DVP HM/AW']
Dropping bad 0.5 rows: 1


,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,LINE
1,Paolo Banchero,WAS at ORL,Points + Rebounds + Assists,o0.5,-110,0.5


HIST_PROB null: 0 / 99
Unique games in df: 10
Odds API completed events pulled: 20
Teams with PA samples: 29 | league_pa: 109.14
✅ Enrichment coverage:
ESPN_OPP_DEF_PPG_PROXY null: 0 / 99
ESPN_PACE_PROXY null: 0 / 99
Gate used: EV>0 & edge≥3pts & units>0 (strict)


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,ESPN_OPP_DEF_PPG_PROXY,ESPN_PACE_PROXY
0,1,Jaylin Williams,OKC at CHI,Assists,u4.5,100,0.37,0.874803,0.749605,92.0,100.0
1,2,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.34,0.850900,0.624445,92.0,100.0
2,3,Jaylin Williams,OKC at CHI,Rebounds + Assists,u12.5,-110,0.33,0.834225,0.592611,92.0,100.0
3,4,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u24.5,-120,0.34,0.853284,0.564355,92.0,100.0
4,5,Jaylin Williams,OKC at CHI,Points + Assists,u16.5,-120,0.34,0.853000,0.563833,92.0,100.0


Saved: prop_engine_full_nba_20260303_180148.csv
Saved: hit_sheet_top5_nba_20260303_180148.csv

NBA ELITE PROP CARD
—
1) Jaylin Williams — OKC at CHI
Assists u4.5 (100) — 0.37u
2) Jaylin Williams — OKC at CHI
Points + Rebounds u18.5 (-110) — 0.34u
3) Jaylin Williams — OKC at CHI
Rebounds + Assists u12.5 (-110) — 0.33u
4) Jaylin Williams — OKC at CHI
Points + Rebounds + Assists u24.5 (-120) — 0.34u
5) Jaylin Williams — OKC at CHI
Points + Assists u16.5 (-120) — 0.34u


In [29]:
# =========================
# INJURY LAYER (lightweight, CSV-driven)
# - Uses any injury/status columns present in df (if none, it won't change anything)
# - Applies teammate μ bumps when key usage player is OUT
# =========================
import re
import numpy as np
import pandas as pd

if "df" not in globals():
    raise RuntimeError("df not found. Run the main pipeline cell first.")
if "df_rank" not in globals():
    raise RuntimeError("df_rank not found. Run the main pipeline cell first.")

inj_cols = [c for c in df.columns if re.search(r"(inj|status|out|dnp|question|doubt|prob)", str(c), re.I)]
print("Injury/status columns detected:", inj_cols)

# If you don't have injury cols in the CSV, this layer is a no-op (keeps identical outputs)
if len(inj_cols) == 0:
    print("⚠️ No injury/status columns in CSV — cannot apply true injury adjustments. Keeping current board.")
else:
    # Build player status (OUT / DOUBTFUL / Q / PROBABLE / ACTIVE)
    def normalize_status(x):
        s = str(x).strip().lower()
        if s in ("", "nan", "none"): return None
        if "out" in s or "dnp" in s: return "OUT"
        if "doubt" in s: return "DOUBTFUL"
        if "quest" in s or s == "q": return "QUESTIONABLE"
        if "prob" in s: return "PROBABLE"
        if "active" in s: return "ACTIVE"
        return None

    status = pd.Series([None]*len(df), index=df.index)
    for c in inj_cols:
        status = status.fillna(df[c].map(normalize_status))
    df["INJ_STATUS"] = status.fillna("UNKNOWN")

    # Parse teams from GAME ("AAA at BBB")
    def parse_game(g):
        g = str(g)
        if " at " in g:
            a,h = g.split(" at ", 1)
            return a.strip(), h.strip()
        return None, None

    away, home = zip(*df["GAME"].map(parse_game))
    df["AWAY_ABBR"] = away
    df["HOME_ABBR"] = home

    # Mark OUT players
    out_players = set(df.loc[df["INJ_STATUS"].eq("OUT"), "PLAYER"].astype(str))
    print("OUT players found in CSV:", len(out_players))

    # Very simple teammate bump rules (tunable)
    # If a player is OUT, teammates in same game get small μ bump for assists/points/pra etc.
    bump_by_mkt = {
        "assists": 0.25,
        "points": 0.35,
        "rebounds": 0.20,
        "pa": 0.30,
        "pr": 0.30,
        "ra": 0.25,
        "pra": 0.40,
        "three": 0.10,
        "blocks": 0.05,
        "steals": 0.05,
    }

    # If your pipeline created these columns, we apply bump into PROJ_MU_CTX and recompute probs
    required_mu_cols = {"PROJ_MU_CTX","PROJ_MU_BASE","MKT","LINE","DIST_USED","SIDE"}
    if not required_mu_cols.issubset(df.columns):
        print("⚠️ Projection μ columns not found (PROJ_MU_CTX etc). Injury layer can't adjust μ. Keeping current board.")
    else:
        df["PROJ_MU_CTX_INJ"] = df["PROJ_MU_CTX"].astype(float)

        # If any OUT exists in that game, bump everyone else slightly (except the OUT player)
        # This is conservative and avoids needing full roster mapping.
        games_with_out = set(df.loc[df["PLAYER"].astype(str).isin(out_players), "GAME"].astype(str).unique())
        for i, r in df.iterrows():
            if r["GAME"] in games_with_out and str(r["PLAYER"]) not in out_players:
                m = str(r["MKT"])
                df.at[i, "PROJ_MU_CTX_INJ"] = float(r["PROJ_MU_CTX"]) + bump_by_mkt.get(m, 0.0)

        # Recompute PROJ_PROB_CTX using same prob function if it exists
        if "prob_from_mu" not in globals():
            print("⚠️ prob_from_mu() not found in notebook. Can't recompute probs. Keeping current board.")
        else:
            def prob_row(r):
                return float(prob_from_mu(r["LINE"], r["PROJ_MU_CTX_INJ"], r["DIST_USED"], r["SIDE"], r.get("AUX_PARAM", None), r["MKT"]))
            df["PROJ_PROB_CTX_INJ"] = df.apply(prob_row, axis=1).clip(0,1)

            # Reblend + safe prob (keep same weights you used today)
            if "HIST_PROB" not in df.columns:
                raise RuntimeError("HIST_PROB missing. Run hit-rate build first.")

            # If you used blend weights in globals, reuse; else default 60/40 hist/proj
            w_hist = globals().get("W_HIST", 0.6)
            w_proj = globals().get("W_PROJ", 0.4)

            df["BLEND_PROB_INJ"] = (w_hist*df["HIST_PROB"] + w_proj*df["PROJ_PROB_CTX_INJ"]).clip(0,1)
            # SAFE_PROB definition: use your existing safe transform if present, else conservative shrink
            if "safe_prob" in globals():
                df["SAFE_PROB_INJ"] = df["BLEND_PROB_INJ"].map(safe_prob)
            else:
                df["SAFE_PROB_INJ"] = (0.92*df["BLEND_PROB_INJ"]).clip(0,1)

            # Update board for ranking (drop OUT props entirely)
            board2 = df.copy()
            board2 = board2[~board2["PLAYER"].astype(str).isin(out_players)].copy()

            # Recompute EV + Kelly from SAFE_PROB_INJ
            if "ev_1u" not in globals() or "half_kelly_units" not in globals():
                print("⚠️ ev_1u()/half_kelly_units() not found. Can't recompute EV/units. Keeping current board.")
            else:
                board2["EV_SAFE_1U_INJ"] = board2.apply(lambda r: ev_1u(r["SAFE_PROB_INJ"], r["DEC_ODDS"]), axis=1)
                board2["UNITS_INJ"] = board2.apply(lambda r: half_kelly_units(r["SAFE_PROB_INJ"], r["DEC_ODDS"]), axis=1)

                # Rank
                board2["SAFE_SCORE_INJ"] = (board2["EV_SAFE_1U_INJ"]*100) * (board2["SAFE_PROB_INJ"]**0.5)
                board2 = board2.sort_values(["SAFE_SCORE_INJ","EV_SAFE_1U_INJ","SAFE_PROB_INJ"], ascending=False)

                display(board2[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SAFE_PROB_INJ","EV_SAFE_1U_INJ","UNITS_INJ"]].head(15))
                print("✅ Injury-adjusted board built (CSV-driven).")

Injury/status columns detected: ['OUTCOME', 'IM_PROB_%', 'BOOK_PROB', 'IM_PROB_%_P', 'HIST_PROB', 'PROJ_PROB_CTX', 'BLEND_PROB', 'SAFE_PROB']
OUT players found in CSV: 0


TypeError: prob_from_mu() takes 5 positional arguments but 6 were given

In [30]:
# =========================
# FIXED INJURY LAYER (signature-safe + better injury column detection)
# Drop-in replacement for the failing cell
# =========================
import re, inspect
import numpy as np
import pandas as pd

if "df" not in globals():
    raise RuntimeError("df not found. Run the main pipeline cell first.")

# 1) Detect actual injury/status columns (avoid prob columns)
def is_injury_col(name: str) -> bool:
    n = str(name).strip().lower()
    # must contain injury/status keywords AND must NOT be a probability column
    if re.search(r"(injur|status|designation|availability|report|question|doubt|probable|out\b|ruled\s*out|did\s+not\s+play|dnp)", n):
        if re.search(r"(prob|im_prob|book_prob|hist_prob|proj_prob|blend_prob|safe_prob)", n):
            return False
        return True
    return False

inj_cols = [c for c in df.columns if is_injury_col(c)]
print("Injury/status columns detected:", inj_cols)

# 2) Build INJ_STATUS if columns exist; else mark UNKNOWN
def normalize_status(x):
    s = str(x).strip().lower()
    if s in ("", "nan", "none"): return None
    if "out" in s or "ruled out" in s or "dnp" in s: return "OUT"
    if "doubt" in s: return "DOUBTFUL"
    if "quest" in s or s == "q": return "QUESTIONABLE"
    if "prob" in s: return "PROBABLE"
    if "active" in s: return "ACTIVE"
    return None

df = df.copy()
if len(inj_cols) == 0:
    df["INJ_STATUS"] = "UNKNOWN"
else:
    status = pd.Series([None]*len(df), index=df.index)
    for c in inj_cols:
        status = status.fillna(df[c].map(normalize_status))
    df["INJ_STATUS"] = status.fillna("UNKNOWN")

out_players = set(df.loc[df["INJ_STATUS"].eq("OUT"), "PLAYER"].astype(str))
print("OUT players found in CSV:", len(out_players))

# 3) If we don't have projection columns, stop safely
required_mu_cols = {"PROJ_MU_CTX","MKT","LINE","DIST_USED","SIDE"}
missing_mu = [c for c in required_mu_cols if c not in df.columns]
if missing_mu:
    print("⚠️ Missing projection columns; cannot apply injury μ adjustments:", missing_mu)
    print("Keeping existing board (no injury adjustments applied).")
else:
    # 4) Create PROJ_MU_CTX_INJ (small teammate bump ONLY if any OUT exists in game)
    bump_by_mkt = {
        "assists": 0.25, "points": 0.35, "rebounds": 0.20,
        "pa": 0.30, "pr": 0.30, "ra": 0.25, "pra": 0.40,
        "three": 0.10, "blocks": 0.05, "steals": 0.05,
    }

    df["PROJ_MU_CTX_INJ"] = df["PROJ_MU_CTX"].astype(float)

    if len(out_players) > 0:
        games_with_out = set(df.loc[df["PLAYER"].astype(str).isin(out_players), "GAME"].astype(str).unique())
        for i, r in df.iterrows():
            if str(r["GAME"]) in games_with_out and str(r["PLAYER"]) not in out_players:
                m = str(r["MKT"])
                df.at[i, "PROJ_MU_CTX_INJ"] = float(r["PROJ_MU_CTX"]) + bump_by_mkt.get(m, 0.0)

    # 5) Recompute PROJ_PROB_CTX_INJ using prob_from_mu() with signature detection
    if "prob_from_mu" not in globals():
        raise RuntimeError("prob_from_mu() not found in notebook. Run the base distribution/prob cell first.")

    n_params = len(inspect.signature(prob_from_mu).parameters)
    print(f"prob_from_mu() param count detected: {n_params}")

    def prob_row(r):
        line = float(r["LINE"])
        mu   = float(r["PROJ_MU_CTX_INJ"])
        dist = r["DIST_USED"]
        side = r["SIDE"]

        # Your notebook likely has 5-arg version: (line, mu, dist, side, aux)
        # Some versions have 4-arg: (line, mu, dist, side)
        # Some versions have 6-arg: adds market or other
        aux = r.get("AUX_PARAM", None)

        if n_params == 4:
            return float(prob_from_mu(line, mu, dist, side))
        elif n_params == 5:
            return float(prob_from_mu(line, mu, dist, side, aux))
        elif n_params == 6:
            return float(prob_from_mu(line, mu, dist, side, aux, r["MKT"]))
        else:
            raise RuntimeError(f"Unexpected prob_from_mu() signature with {n_params} params.")

    df["PROJ_PROB_CTX_INJ"] = df.apply(prob_row, axis=1).clip(0, 1)

    # 6) Reblend / Safe / EV / Units (reuse notebook helpers if present)
    if "HIST_PROB" not in df.columns:
        raise RuntimeError("HIST_PROB missing. Run hit-rate build first.")

    w_hist = globals().get("W_HIST", 0.6)
    w_proj = globals().get("W_PROJ", 0.4)

    df["BLEND_PROB_INJ"] = (w_hist*df["HIST_PROB"] + w_proj*df["PROJ_PROB_CTX_INJ"]).clip(0, 1)

    if "safe_prob" in globals():
        df["SAFE_PROB_INJ"] = df["BLEND_PROB_INJ"].map(safe_prob)
    else:
        df["SAFE_PROB_INJ"] = (0.92*df["BLEND_PROB_INJ"]).clip(0, 1)

    if "ev_1u" not in globals() or "half_kelly_units" not in globals():
        print("⚠️ ev_1u()/half_kelly_units() missing — cannot recompute EV/units. Keeping existing.")
    else:
        df["EV_SAFE_1U_INJ"] = df.apply(lambda r: ev_1u(r["SAFE_PROB_INJ"], r["DEC_ODDS"]), axis=1)
        df["UNITS_INJ"] = df.apply(lambda r: half_kelly_units(r["SAFE_PROB_INJ"], r["DEC_ODDS"]), axis=1)

        # replace board columns in-place so downstream ranking uses injury-adjusted versions
        df["SAFE_PROB"] = df["SAFE_PROB_INJ"]
        df["EV_SAFE_1U"] = df["EV_SAFE_1U_INJ"]
        df["UNITS"] = df["UNITS_INJ"]

        print("✅ Injury layer recompute complete (if OUT players exist in CSV, μ bumps applied).")
        display(df[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SAFE_PROB","EV_SAFE_1U","UNITS","INJ_STATUS"]].head(15))

Injury/status columns detected: ['INJ_STATUS']
OUT players found in CSV: 0
prob_from_mu() param count detected: 5
⚠️ ev_1u()/half_kelly_units() missing — cannot recompute EV/units. Keeping existing.


In [31]:
# ==========================================
# BOTTOM CELL — PROJECTED STARTERS GATE (STRICT)
# Adds projected starters and reruns final layer outputs
# ==========================================
import re, time, math
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime

assert "df" in globals(), "df is not defined. Run your CSV load/clean cell first."
req_cols = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","SIDE","LINE","MKT"]
missing = [c for c in req_cols if c not in df.columns]
assert not missing, f"df missing required columns for this bottom cell: {missing}"

# --- helpers ---
def norm_name(s: str) -> str:
    if s is None: return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("’","'").replace("–","-").replace("—","-")
    return s

def norm_game(s: str) -> str:
    # normalize variants into "AAA at BBB"
    s = str(s).strip()
    s = s.replace("@", " at ").replace(" vs ", " at ").replace("VS", "at").replace("vs.", "at")
    s = re.sub(r"\s+", " ", s)
    # if "AAA at BBB" already, ok
    return s

def fetch_projected_starters():
    """
    Returns dict: game_str ("AAA at BBB") -> set(player_names) (10 players, both teams)
    Tries NBA.com todays-lineups first; falls back to other lineup pages if needed.
    """
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nba.com/",
    }
    sess = requests.Session()

    # 1) Try NBA.com (often JS; may return shell)
    nba_url = "https://www.nba.com/players/todays-lineups"
    try:
        r = sess.get(nba_url, headers=headers, timeout=20)
        html = r.text or ""
        # If meaningful player names appear, parse them
        if r.status_code == 200 and len(html) > 20000 and ("Projected Lineup" in html or "Starting Lineups" in html):
            soup = BeautifulSoup(html, "html.parser")
            text = soup.get_text(" ", strip=True)
            # crude parse: look for repeated player-like tokens is unreliable; so only accept if we can extract enough names
            # If not enough, fall through to fallback sources.
    except Exception:
        pass

    # 2) Fallback: BasketballMonster lineups (HTML, easy to parse)
    # Source page example  [oai_citation:2‡Basketball Monster](https://basketballmonster.com/nbalineups.aspx?utm_source=chatgpt.com)
    bm_url = "https://basketballmonster.com/nbalineups.aspx"
    starters_by_game = {}
    try:
        r = sess.get(bm_url, headers=headers, timeout=25)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        page_text = soup.get_text("\n", strip=True)

        # BasketballMonster blocks look like: "WAS @ ORL ... PG <name> <name> SG <name> <name> ..."
        # We'll regex game headers then capture the next ~30 lines for starters.
        # NOTE: This is intentionally robust rather than fragile on exact DOM.
        lines = [ln.strip() for ln in page_text.splitlines() if ln.strip()]
        # detect game header lines like "WAS @ ORL"
        game_idx = [i for i,ln in enumerate(lines) if re.match(r"^[A-Z]{2,3}\s*@\s*[A-Z]{2,3}$", ln)]
        for gi in game_idx:
            g = lines[gi].replace("@", " at ")
            # scan forward for position rows; capture next 50 lines
            window = lines[gi:gi+80]
            # positions
            pos = {"PG","SG","SF","PF","C"}
            names = []
            for j in range(len(window)-2):
                if window[j] in pos:
                    # format tends to be: POS, away_player, home_player
                    away_p = window[j+1] if j+1 < len(window) else ""
                    home_p = window[j+2] if j+2 < len(window) else ""
                    if away_p and away_p not in pos and len(away_p) < 40: names.append(away_p)
                    if home_p and home_p not in pos and len(home_p) < 40: names.append(home_p)
            # keep only if we got something close to 10
            names = [norm_name(n) for n in names if re.search(r"[A-Za-z]", n)]
            if len(names) >= 8:
                starters_by_game[norm_game(g)] = set(names)
    except Exception:
        starters_by_game = {}

    # 3) Fallback 2: FantasyPros lineups (HTML, sometimes sparse)  [oai_citation:3‡FantasyPros](https://www.fantasypros.com/nba/lineups/?utm_source=chatgpt.com)
    if len(starters_by_game) < 3:
        fp_url = "https://www.fantasypros.com/nba/lineups/"
        try:
            r = sess.get(fp_url, headers=headers, timeout=25)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            text = soup.get_text("\n", strip=True)
            lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
            # Look for "XXX at YYY" patterns and then next players; this page can be empty in some sections,
            # so only use if we can detect 8+ names.
            for i,ln in enumerate(lines):
                m = re.match(r"^([A-Z]{2,3})\s+at\s+([A-Z]{2,3})$", ln)
                if not m:
                    continue
                g = norm_game(ln)
                window = lines[i:i+120]
                # player names on FantasyPros are often full names; just collect capitalized tokens with spaces
                cand = []
                for w in window:
                    if re.match(r"^[A-Z][a-zA-Z\.\'\-]+\s+[A-Z][a-zA-Z\.\'\-]+", w):
                        cand.append(norm_name(w))
                cand = list(dict.fromkeys(cand))[:20]
                if len(cand) >= 8:
                    starters_by_game[g] = set(cand)
        except Exception:
            pass

    return starters_by_game

# --- pull starters ---
starters_by_game = fetch_projected_starters()
print("Projected starters games found:", len(starters_by_game), " / unique games in df:", df["GAME"].nunique())

# normalize df names/games
df["PLAYER_N"] = df["PLAYER"].map(norm_name)
df["GAME_N"]   = df["GAME"].map(norm_game)

# mark starter if player appears in that game's starter set (either team)
def is_starter(row):
    s = starters_by_game.get(row["GAME_N"])
    if not s:
        return np.nan  # unknown coverage (don’t force)
    return row["PLAYER_N"] in s

df["IS_PROJECTED_STARTER"] = df.apply(is_starter, axis=1)
known = df["IS_PROJECTED_STARTER"].notna().sum()
print("Starter coverage:", known, "/", len(df))

# --- apply μ adjustment (bench -> lower μ) ---
# If starter unknown, do NOT change μ.
# Bench players get μ * 0.90 (this boosts unders naturally, reduces overs naturally).
bench_mult = 0.90

if "PROJ_MU_CTX" not in df.columns:
    raise RuntimeError("Missing PROJ_MU_CTX. Run your projection+context μ cells first (same as 3/1).")

df["PROJ_MU_CTX_START"] = df["PROJ_MU_CTX"]
mask_known = df["IS_PROJECTED_STARTER"].notna()
mask_bench = mask_known & (df["IS_PROJECTED_STARTER"] == False)
df.loc[mask_bench, "PROJ_MU_CTX_START"] = df.loc[mask_bench, "PROJ_MU_CTX"] * bench_mult

# --- probability function wrapper (handles 5-arg vs 6-arg versions) ---
def prob_from_mu_wrapper(line, mu, dist_used, side, aux_param, mkt):
    # Try common signatures without you having to edit older defs
    if "prob_from_mu" not in globals():
        raise RuntimeError("prob_from_mu() not found. Run the distribution/probability function cells first.")
    fn = globals()["prob_from_mu"]
    try:
        return float(fn(line, mu, dist_used, side, mkt))          # 5 args
    except TypeError:
        return float(fn(line, mu, dist_used, side, aux_param, mkt))  # 6 args

# recompute projection probability with starter-adjusted μ
if "DIST_USED" not in df.columns:
    # if you didn’t set DIST_USED, default to normal
    df["DIST_USED"] = "normal"
if "AUX_PARAM" not in df.columns:
    df["AUX_PARAM"] = np.nan

df["PROJ_PROB_CTX_START"] = df.apply(
    lambda r: np.clip(
        prob_from_mu_wrapper(r["LINE"], r["PROJ_MU_CTX_START"], r["DIST_USED"], r["SIDE"], r["AUX_PARAM"], r["MKT"]),
        0, 1
    ),
    axis=1
)

# --- blend / safe ---
# If your notebook already has weights, keep them; else default to 50/50.
w_hist = globals().get("W_HIST", 0.50)
w_proj = globals().get("W_PROJ", 0.50)
w_hist = float(w_hist); w_proj = float(w_proj)
if "HIST_PROB" not in df.columns:
    raise RuntimeError("Missing HIST_PROB. Run the hit-rate build cell first.")

df["BLEND_PROB_START"] = np.clip(w_hist*df["HIST_PROB"] + w_proj*df["PROJ_PROB_CTX_START"], 0, 1)

# Safe prob: conservative blend between blend and book, same shape as prior runs
if "BOOK_PROB" not in df.columns:
    raise RuntimeError("Missing BOOK_PROB. Run the odds->book prob cell first.")
df["SAFE_PROB_START"] = np.clip(0.65*df["BLEND_PROB_START"] + 0.35*df["BOOK_PROB"], 0, 1)

# --- EV per 1u at offered odds ---
def ev_1u(p, american):
    p = float(p)
    a = float(american)
    if a == 0: return np.nan
    if a > 0:
        b = a/100.0
    else:
        b = 100.0/abs(a)
    # EV = p*b - (1-p)*1
    return p*b - (1-p)

df["EV_SAFE_1U_START"] = df.apply(lambda r: ev_1u(r["SAFE_PROB_START"], r["AMERICAN_ODDS"]), axis=1)

# --- Half-Kelly units (clipped like your workflow) ---
def half_kelly_units(p, american, unit_cap=1.0, unit_floor=0.0, scale=1.0):
    p = float(p); a = float(american)
    if a > 0: b = a/100.0
    else:     b = 100.0/abs(a)
    q = 1.0 - p
    f = (b*p - q)/b
    f = max(0.0, f) * 0.5  # half Kelly
    u = f * scale
    return float(np.clip(u, unit_floor, unit_cap))

unit_cap = float(globals().get("UNIT_CAP", 1.0))
df["UNITS_START"] = df.apply(lambda r: half_kelly_units(r["SAFE_PROB_START"], r["AMERICAN_ODDS"], unit_cap=unit_cap), axis=1)

# --- STRICT gate (same spirit as your 3/1 runs) ---
edge_pts = (df["SAFE_PROB_START"] - df["BOOK_PROB"]) * 100.0
gate = (df["EV_SAFE_1U_START"] > 0) & (edge_pts >= 3.0) & (df["UNITS_START"] > 0)

topN = int(globals().get("TOP_N_CARD", 5))
card = df.loc[gate].copy().sort_values(["UNITS_START","EV_SAFE_1U_START"], ascending=False).head(topN)

# rename to your standard output column names
card_out = card.copy()
card_out["RANK"] = range(1, len(card_out)+1)
card_out = card_out[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "UNITS_START","SAFE_PROB_START","EV_SAFE_1U_START","IS_PROJECTED_STARTER"
]].rename(columns={
    "UNITS_START":"UNITS",
    "SAFE_PROB_START":"SAFE_PROB",
    "EV_SAFE_1U_START":"EV_SAFE_1U"
})

print("\nGate used: EV>0 & edge≥3pts & units>0 (strict) + starters μ-adjust")
display(card_out)

# --- Save outputs ---
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
full_path = f"prop_engine_full_nba_{ts}.csv"
card_path = f"hit_sheet_top{topN}_nba_{ts}.csv"
df.to_csv(full_path, index=False)
card_out.to_csv(card_path, index=False)
print("\nSaved:", full_path)
print("Saved:", card_path)

# --- Discord print (no model-lingo; keep your card style) ---
lines = []
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for i, r in card_out.iterrows():
    u = float(r["UNITS"])
    lines.append(f"{int(r['RANK'])}) {r['PLAYER']} — {r['GAME']}")
    lines.append(f"{r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS'])}) — {u:.2f}u")
print("\n" + "\n".join(lines))

Projected starters games found: 10  / unique games in df: 10
Starter coverage: 72 / 99

Gate used: EV>0 & edge≥3pts & units>0 (strict) + starters μ-adjust


,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,IS_PROJECTED_STARTER
8,1,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.325,0.833333,0.590909,True
9,2,Jaylin Williams,OKC at CHI,Rebounds + Assists,u12.5,-110,0.325,0.833333,0.590909,True
12,3,Jaylin Williams,OKC at CHI,Points + Assists,u15.5,-114,0.325,0.836449,0.570175,True
4,4,Jaylin Williams,OKC at CHI,Points + Assists,u16.5,-120,0.325,0.840909,0.541667,True
5,5,Jaylin Williams,OKC at CHI,Points + Rebounds + Assists,u23.5,-120,0.325,0.840909,0.541667,True



Saved: prop_engine_full_nba_20260303_182146.csv
Saved: hit_sheet_top5_nba_20260303_182146.csv

NBA ELITE PROP CARD
—
1) Jaylin Williams — OKC at CHI
Points + Rebounds u18.5 (-110) — 0.33u
2) Jaylin Williams — OKC at CHI
Rebounds + Assists u12.5 (-110) — 0.33u
3) Jaylin Williams — OKC at CHI
Points + Assists u15.5 (-114) — 0.33u
4) Jaylin Williams — OKC at CHI
Points + Assists u16.5 (-120) — 0.33u
5) Jaylin Williams — OKC at CHI
Points + Rebounds + Assists u23.5 (-120) — 0.33u


In [32]:
# ==========================================
# NEXT CELL — DIVERSIFY CARD (1 per player)
# Keeps the same scoring you just computed, only changes selection
# ==========================================
import numpy as np
import pandas as pd

assert "df" in globals(), "df not found"
need = ["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","UNITS_START","SAFE_PROB_START","EV_SAFE_1U_START"]
miss = [c for c in need if c not in df.columns]
assert not miss, f"Missing required columns from prior cell: {miss}"

topN = int(globals().get("TOP_N_CARD", 5))

# same strict gate as prior cell
edge_pts = (df["SAFE_PROB_START"] - df["BOOK_PROB"]) * 100.0
gate = (df["EV_SAFE_1U_START"] > 0) & (edge_pts >= 3.0) & (df["UNITS_START"] > 0)

pool = df.loc[gate].copy()

# Rank key: units first, then EV
pool["RANK_KEY_1"] = pool["UNITS_START"].astype(float)
pool["RANK_KEY_2"] = pool["EV_SAFE_1U_START"].astype(float)

# 1) keep only best play per PLAYER (you can switch to ["PLAYER","GAME"] if you prefer)
pool = pool.sort_values(["RANK_KEY_1","RANK_KEY_2"], ascending=False)
pool = pool.drop_duplicates(subset=["PLAYER"], keep="first")

# OPTIONAL: max 2 plays per game (uncomment if you want)
# pool["GAME_RANK"] = pool.groupby("GAME").cumcount()
# pool = pool[pool["GAME_RANK"] < 2]

card = pool.head(topN).copy()
card["RANK"] = range(1, len(card)+1)

card_out = card[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "UNITS_START","SAFE_PROB_START","EV_SAFE_1U_START","IS_PROJECTED_STARTER"
]].rename(columns={
    "UNITS_START":"UNITS",
    "SAFE_PROB_START":"SAFE_PROB",
    "EV_SAFE_1U_START":"EV_SAFE_1U"
})

display(card_out)

# Discord print (same style)
lines = []
lines.append("NBA ELITE PROP CARD")
lines.append("—")
for _, r in card_out.iterrows():
    lines.append(f"{int(r['RANK'])}) {r['PLAYER']} — {r['GAME']}")
    lines.append(f"{r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS'])}) — {float(r['UNITS']):.2f}u")
print("\n" + "\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,IS_PROJECTED_STARTER
8,1,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.325000,0.833333,0.590909,True
19,2,Jalen Green,PHX at SAC,Rebounds + Assists,u6.5,-123,0.325000,0.843049,0.528455,NaN
32,3,Jett Howard,WAS at ORL,Points + Rebounds + Assists,u10.5,-130,0.318013,0.841750,0.489251,False
21,4,Isaiah Joe,OKC at CHI,Rebounds + Assists,u6.5,-175,0.311813,0.863137,0.356358,True
14,5,Will Riley,WAS at ORL,Rebounds + Assists,u7.5,-120,0.309923,0.827203,0.516539,False



NBA ELITE PROP CARD
—
1) Jaylin Williams — OKC at CHI
Points + Rebounds u18.5 (-110) — 0.33u
2) Jalen Green — PHX at SAC
Rebounds + Assists u6.5 (-123) — 0.33u
3) Jett Howard — WAS at ORL
Points + Rebounds + Assists u10.5 (-130) — 0.32u
4) Isaiah Joe — OKC at CHI
Rebounds + Assists u6.5 (-175) — 0.31u
5) Will Riley — WAS at ORL
Rebounds + Assists u7.5 (-120) — 0.31u


In [33]:
# ==========================================
# STRICT STARTER-GATED CARD (drop NaN starter status)
# ==========================================
import numpy as np
import pandas as pd

topN = int(globals().get("TOP_N_CARD", 5))

edge_pts = (df["SAFE_PROB_START"] - df["BOOK_PROB"]) * 100.0
gate = (
    (df["EV_SAFE_1U_START"] > 0) &
    (edge_pts >= 3.0) &
    (df["UNITS_START"] > 0) &
    (df["IS_PROJECTED_STARTER"].notna())   # <— STRICT: only games we have starters for
)

pool = df.loc[gate].copy()
pool["RANK_KEY_1"] = pool["UNITS_START"].astype(float)
pool["RANK_KEY_2"] = pool["EV_SAFE_1U_START"].astype(float)

pool = pool.sort_values(["RANK_KEY_1","RANK_KEY_2"], ascending=False)
pool = pool.drop_duplicates(subset=["PLAYER"], keep="first")
card = pool.head(topN).copy()
card["RANK"] = range(1, len(card)+1)

card_out = card[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "UNITS_START","SAFE_PROB_START","EV_SAFE_1U_START","IS_PROJECTED_STARTER"
]].rename(columns={
    "UNITS_START":"UNITS",
    "SAFE_PROB_START":"SAFE_PROB",
    "EV_SAFE_1U_START":"EV_SAFE_1U"
})

display(card_out)

lines = ["NBA ELITE PROP CARD","—"]
for _, r in card_out.iterrows():
    lines.append(f"{int(r['RANK'])}) {r['PLAYER']} — {r['GAME']}")
    lines.append(f"{r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS'])}) — {float(r['UNITS']):.2f}u")
print("\n" + "\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,IS_PROJECTED_STARTER
8,1,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.325000,0.833333,0.590909,True
32,2,Jett Howard,WAS at ORL,Points + Rebounds + Assists,u10.5,-130,0.318013,0.841750,0.489251,False
21,3,Isaiah Joe,OKC at CHI,Rebounds + Assists,u6.5,-175,0.311813,0.863137,0.356358,True
14,4,Will Riley,WAS at ORL,Rebounds + Assists,u7.5,-120,0.309923,0.827203,0.516539,False
33,5,Nolan Traore,BKN at MIA,Rebounds,u2.5,-168,0.308370,0.856992,0.367107,True



NBA ELITE PROP CARD
—
1) Jaylin Williams — OKC at CHI
Points + Rebounds u18.5 (-110) — 0.33u
2) Jett Howard — WAS at ORL
Points + Rebounds + Assists u10.5 (-130) — 0.32u
3) Isaiah Joe — OKC at CHI
Rebounds + Assists u6.5 (-175) — 0.31u
4) Will Riley — WAS at ORL
Rebounds + Assists u7.5 (-120) — 0.31u
5) Nolan Traore — BKN at MIA
Rebounds u2.5 (-168) — 0.31u


In [34]:
# ==========================================
# STRICT STARTER-GATED CARD (drop NaN starter status)
# ==========================================
import numpy as np
import pandas as pd

topN = int(globals().get("TOP_N_CARD", 5))

edge_pts = (df["SAFE_PROB_START"] - df["BOOK_PROB"]) * 100.0
gate = (
    (df["EV_SAFE_1U_START"] > 0) &
    (edge_pts >= 3.0) &
    (df["UNITS_START"] > 0) &
    (df["IS_PROJECTED_STARTER"].notna())   # <— STRICT: only games we have starters for
)

pool = df.loc[gate].copy()
pool["RANK_KEY_1"] = pool["UNITS_START"].astype(float)
pool["RANK_KEY_2"] = pool["EV_SAFE_1U_START"].astype(float)

pool = pool.sort_values(["RANK_KEY_1","RANK_KEY_2"], ascending=False)
pool = pool.drop_duplicates(subset=["PLAYER"], keep="first")
card = pool.head(topN).copy()
card["RANK"] = range(1, len(card)+1)

card_out = card[[
    "RANK","PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "UNITS_START","SAFE_PROB_START","EV_SAFE_1U_START","IS_PROJECTED_STARTER"
]].rename(columns={
    "UNITS_START":"UNITS",
    "SAFE_PROB_START":"SAFE_PROB",
    "EV_SAFE_1U_START":"EV_SAFE_1U"
})

display(card_out)

lines = ["NBA ELITE PROP CARD","—"]
for _, r in card_out.iterrows():
    lines.append(f"{int(r['RANK'])}) {r['PLAYER']} — {r['GAME']}")
    lines.append(f"{r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS'])}) — {float(r['UNITS']):.2f}u")
print("\n" + "\n".join(lines))

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,IS_PROJECTED_STARTER
8,1,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.325000,0.833333,0.590909,True
32,2,Jett Howard,WAS at ORL,Points + Rebounds + Assists,u10.5,-130,0.318013,0.841750,0.489251,False
21,3,Isaiah Joe,OKC at CHI,Rebounds + Assists,u6.5,-175,0.311813,0.863137,0.356358,True
14,4,Will Riley,WAS at ORL,Rebounds + Assists,u7.5,-120,0.309923,0.827203,0.516539,False
33,5,Nolan Traore,BKN at MIA,Rebounds,u2.5,-168,0.308370,0.856992,0.367107,True



NBA ELITE PROP CARD
—
1) Jaylin Williams — OKC at CHI
Points + Rebounds u18.5 (-110) — 0.33u
2) Jett Howard — WAS at ORL
Points + Rebounds + Assists u10.5 (-130) — 0.32u
3) Isaiah Joe — OKC at CHI
Rebounds + Assists u6.5 (-175) — 0.31u
4) Will Riley — WAS at ORL
Rebounds + Assists u7.5 (-120) — 0.31u
5) Nolan Traore — BKN at MIA
Rebounds u2.5 (-168) — 0.31u


In [35]:
# ==========================================
# EXPAND TO TOP 10 (same strict logic)
# ==========================================
import numpy as np
import pandas as pd

TOP_N = 10  # <-- expand to 10

edge_pts = (df["SAFE_PROB_START"] - df["BOOK_PROB"]) * 100.0
gate = (
    (df["EV_SAFE_1U_START"] > 0) &
    (edge_pts >= 3.0) &
    (df["UNITS_START"] > 0)
)

pool = df.loc[gate].copy()

# Rank priority
pool = pool.sort_values(
    ["UNITS_START", "EV_SAFE_1U_START"],
    ascending=False
)

# Keep best per player (prevents 5 Jaylin props again)
pool = pool.drop_duplicates(subset=["PLAYER"], keep="first")

card10 = pool.head(TOP_N).copy()
card10["RANK"] = range(1, len(card10)+1)

card10_out = card10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS","UNITS_START",
    "SAFE_PROB_START","EV_SAFE_1U_START",
    "IS_PROJECTED_STARTER"
]].rename(columns={
    "UNITS_START":"UNITS",
    "SAFE_PROB_START":"SAFE_PROB",
    "EV_SAFE_1U_START":"EV_SAFE_1U"
})

display(card10_out)

print("\nNBA ELITE PROP CARD — TOP 10")
print("—")
for _, r in card10_out.iterrows():
    print(f"{int(r['RANK'])}) {r['PLAYER']} — {r['GAME']}")
    print(f"{r['PROP']} {r['OUTCOME']} ({int(r['AMERICAN_ODDS'])}) — {float(r['UNITS']):.2f}u")

,RANK,PLAYER,GAME,PROP,OUTCOME,AMERICAN_ODDS,UNITS,SAFE_PROB,EV_SAFE_1U,IS_PROJECTED_STARTER
8,1,Jaylin Williams,OKC at CHI,Points + Rebounds,u18.5,-110,0.325000,0.833333,0.590909,True
19,2,Jalen Green,PHX at SAC,Rebounds + Assists,u6.5,-123,0.325000,0.843049,0.528455,NaN
32,3,Jett Howard,WAS at ORL,Points + Rebounds + Assists,u10.5,-130,0.318013,0.841750,0.489251,False
21,4,Isaiah Joe,OKC at CHI,Rebounds + Assists,u6.5,-175,0.311813,0.863137,0.356358,True
14,5,Will Riley,WAS at ORL,Rebounds + Assists,u7.5,-120,0.309923,0.827203,0.516539,False
33,6,Nolan Traore,BKN at MIA,Rebounds,u2.5,-168,0.308370,0.856992,0.367107,True
45,7,Terance Mann,BKN at MIA,Points + Rebounds,o6.5,100,0.307121,0.807121,0.614242,True
42,8,Dejounte Murray,NOP at LAL,Rebounds + Assists,o9.5,110,0.302974,0.793592,0.666542,NaN
10,9,Chet Holmgren,OKC at CHI,Points + Assists,u21.5,-110,0.298999,0.808570,0.543634,True
20,10,Nique Clifford,PHX at SAC,Points + Rebounds,u18.5,-103,0.291507,0.794589,0.566034,NaN



NBA ELITE PROP CARD — TOP 10
—
1) Jaylin Williams — OKC at CHI
Points + Rebounds u18.5 (-110) — 0.33u
2) Jalen Green — PHX at SAC
Rebounds + Assists u6.5 (-123) — 0.33u
3) Jett Howard — WAS at ORL
Points + Rebounds + Assists u10.5 (-130) — 0.32u
4) Isaiah Joe — OKC at CHI
Rebounds + Assists u6.5 (-175) — 0.31u
5) Will Riley — WAS at ORL
Rebounds + Assists u7.5 (-120) — 0.31u
6) Nolan Traore — BKN at MIA
Rebounds u2.5 (-168) — 0.31u
7) Terance Mann — BKN at MIA
Points + Rebounds o6.5 (100) — 0.31u
8) Dejounte Murray — NOP at LAL
Rebounds + Assists o9.5 (110) — 0.30u
9) Chet Holmgren — OKC at CHI
Points + Assists u21.5 (-110) — 0.30u
10) Nique Clifford — PHX at SAC
Points + Rebounds u18.5 (-103) — 0.29u
